# 作者公开数据：校准来源，不冒充自行重建

来源：[Stambaugh 的 Wharton 数据页](https://fnce.wharton.upenn.edu/profile/stambaug/)。保存 2026-10-13 读取的 **更新至2021年** CH-4 月度因子和 CH-3 日度因子快照，原始 CSV 文本直接嵌入本 notebook；不写额外 `.csv`，后续离线可运行。

作者月文件明示 `rf_mon` 为一年期银行存款利率转月收益，`mktrf` 为市场超额收益。把 `rf_mon` 用作本地重建的利率输入，可以排除 RESSET 无风险率字段释义的不确定性；这属于引用公开基础数据。CH-4 的 MKT/VMG 与 CH-3 相同，SMB经过修改，不能当作 CH-3 的 SMB。更新数据可能与2019论文最终样本版本不同。

日度多空因子不能靠简单逐日复利精确还原月度多空因子。因此这里不以日频 SMB/VMG 的复利结果替代原论文月因子。下游自己的因子计算值与作者数据始终保留独立名称。

原文件单位是百分数（Unit: %）；解析函数全部收益列除以100转小数。公开RF按0.01个百分点精度四舍五入，未把该精度等同于作者底层全精度数据。

In [ ]:
from IPython.display import display
import pandas as pd
import numpy as np
from io import StringIO

author_source_urls = {'ch4_monthly': 'https://faculty.wharton.upenn.edu/wp-content/uploads/2016/11/CH_4_fac_update_20211231.csv', 'ch3_daily': 'https://faculty.wharton.upenn.edu/wp-content/uploads/2016/11/CH_3_fac_daily_update_20211231.csv'}
author_csv_snapshots = {'ch4_monthly': 'Monthly CH-4 factors,,,,,\r\nmnthdt: Month-end date,,,,,\r\nrf_mon: 1-year bank deposit rate converted to monthly rate,,,,,\r\nmktrf: the excess return on the market,,,,,\r\nSMB (Small-minus-big): Size factor,,,,,\r\nVMG(Value-minus-growth) Value factor,,,,,\r\nPMO (Pessimistic-minus-Optimistic) Turnover factor,,,,,\r\nUnit: %,,,,,\r\n,,,,,\r\nmnthdt,rf_mon,mktrf,VMG,SMB,PMO\r\n20000131,0.19,14.79,-0.84,-1.88,-10.9\r\n20000229,0.18,11.97,-7.28,1.65,-1.27\r\n20000331,0.19,4.32,-0.5,7.85,7.93\r\n20000430,0.18,1.68,2.43,-0.56,2.78\r\n20000531,0.19,2.2,2.06,1.85,-0.93\r\n20000630,0.18,2.34,5.55,0.89,0.08\r\n20000731,0.19,4.37,1.93,1.02,-3.93\r\n20000831,0.19,-1.75,-1.47,2.07,2.59\r\n20000930,0.18,-5.51,-0.6,1.88,1.45\r\n20001031,0.19,1.57,0.2,2.73,-0.96\r\n20001130,0.18,5.57,1.93,1.26,1.18\r\n20001231,0.19,-0.89,1.69,1.82,0.08\r\n20010131,0.19,-1.2,2.24,0.31,3.27\r\n20010228,0.17,-5.78,3.59,-0.9,-1.15\r\n20010331,0.19,6.7,-0.82,1.96,-0.76\r\n20010430,0.18,-1.37,-1.88,1.06,1.39\r\n20010531,0.19,2.55,-1.8,2.78,-0.37\r\n20010630,0.18,0.15,2.01,1.52,4.14\r\n20010731,0.19,-13.45,-0.26,-0.15,0.2\r\n20010831,0.19,-4.06,-0.98,0.75,0.65\r\n20010930,0.18,-4.51,3.25,-1.53,0.78\r\n20011031,0.19,-4.77,-0.46,-1.8,-0.38\r\n20011130,0.18,3.46,1.41,2.1,2.18\r\n20011231,0.19,-6.08,5.34,-1.45,3.96\r\n20020131,0.19,-9.93,4.92,-3.78,-0.4\r\n20020228,0.17,2.14,0.39,2.15,-0.42\r\n20020331,0.17,5.53,-2.39,2.27,1.49\r\n20020430,0.16,3.84,-1.78,1.33,0.76\r\n20020531,0.17,-8.64,-1.66,-1.13,-0.09\r\n20020630,0.16,14.23,1.09,-3.3,3.11\r\n20020731,0.17,-4.6,1.44,1.16,2.33\r\n20020831,0.17,0.73,0.65,-0.33,0.34\r\n20020930,0.16,-5.44,0.65,-0.93,-0.37\r\n20021031,0.17,-5,0.39,-0.96,-0.51\r\n20021130,0.16,-5.46,2.42,-3.51,-0.65\r\n20021231,0.17,-5.35,0.64,-0.26,0.79\r\n20030131,0.17,9.36,1.74,1.25,4.48\r\n20030228,0.15,0.63,0.1,0.94,1.26\r\n20030331,0.17,-0.4,4.15,-3.03,-0.49\r\n20030430,0.16,0.67,7.01,-5.89,-2.88\r\n20030531,0.17,4.14,0.46,-0.56,0.08\r\n20030630,0.16,-6.2,2.95,-1.22,3.1\r\n20030731,0.17,-0.61,2.6,-3.41,-0.46\r\n20030831,0.17,-3.81,-0.86,0.95,1.12\r\n20030930,0.16,-4.41,0.49,0.23,0.66\r\n20031031,0.17,-1.67,9.41,-4.87,5.63\r\n20031130,0.16,2.78,2.55,0.31,-1.98\r\n20031231,0.17,6.16,10.36,-7.22,4.06\r\n20040131,0.17,5.96,0.28,3.66,3.93\r\n20040229,0.16,6.05,-3.37,3.81,1.45\r\n20040331,0.17,3.61,2.96,2,2.15\r\n20040430,0.16,-9.36,3.9,0.34,0.84\r\n20040531,0.17,-2.66,-1.06,1.33,0.95\r\n20040630,0.16,-10.2,5.07,-2.95,0.5\r\n20040731,0.17,-0.34,3.15,-1.05,0.35\r\n20040831,0.17,-3.56,4.86,-1.01,3.2\r\n20040930,0.16,4.57,3.83,0.5,-1.48\r\n20041031,0.17,-5.41,4.12,-0.81,1.84\r\n20041130,0.18,1.18,-2.9,3.98,2.81\r\n20041231,0.19,-6.01,5.58,-2.03,3.39\r\n20050131,0.19,-6.16,2.38,-1.42,-1.09\r\n20050228,0.17,9.34,1.54,1.72,-0.87\r\n20050331,0.19,-9.55,4.71,-3.9,3\r\n20050430,0.18,-2.25,7.21,-5.36,-2.42\r\n20050531,0.19,-8.42,-4.68,5.65,2.91\r\n20050630,0.18,2,4.93,-2.09,0.26\r\n20050731,0.19,0.1,8.1,-7.7,-3.56\r\n20050831,0.19,8.36,-4.38,8.09,1.74\r\n20050930,0.18,-0.49,2.92,3.05,1.36\r\n20051031,0.19,-5.75,1.81,-0.21,3.78\r\n20051130,0.18,0.23,-0.62,1.48,2.63\r\n20051231,0.19,6.07,4.37,-4.69,0.65\r\n20060131,0.19,8.62,-0.42,-0.77,2.1\r\n20060228,0.17,3.61,-2.62,-1.67,2.65\r\n20060331,0.19,0.65,1.72,-1.02,-2.85\r\n20060430,0.18,11.28,4.69,-1.07,-3.19\r\n20060531,0.19,16.03,-8.13,9.7,7.51\r\n20060630,0.18,2.59,3.75,4.76,-1.08\r\n20060731,0.19,-6.88,-1.24,4.23,1.69\r\n20060831,0.2,3.94,-0.16,-0.72,0.04\r\n20060930,0.2,4.64,-0.98,-0.63,3.44\r\n20061031,0.21,2.65,8.28,-3.59,1.71\r\n20061130,0.2,14.63,3.99,-12.64,-1.11\r\n20061231,0.21,13.93,6.17,-9.55,-1.84\r\n20070131,0.21,13.68,1.46,12.3,7.74\r\n20070228,0.19,6.51,-5.9,11.81,-8.69\r\n20070331,0.22,11.09,-1.15,6.39,-2.22\r\n20070430,0.23,24.58,-9.3,6.99,-12.46\r\n20070531,0.24,9.41,0.08,-0.91,3.2\r\n20070630,0.25,-9.22,13.35,-6.53,11.85\r\n20070731,0.27,17.61,-3.15,6.26,0.93\r\n20070831,0.29,15.16,1.92,-4.49,1.11\r\n20070930,0.3,6.17,3.51,-2.35,1.07\r\n20071031,0.32,4.83,13.28,-12.72,11.76\r\n20071130,0.31,-15.26,3.42,10.47,0.56\r\n20071231,0.33,11.86,-0.92,8.79,2.68\r\n20080131,0.34,-15.7,-1.59,9.14,0.86\r\n20080229,0.32,2.29,0.8,8.41,0.41\r\n20080331,0.34,-19.92,-0.04,0.17,1.55\r\n20080430,0.33,4.6,7.63,-10.48,-4.45\r\n20080531,0.34,-6.93,-5.64,3.13,4.11\r\n20080630,0.33,-20.76,4.09,-5.27,2.27\r\n20080731,0.34,1.65,-1.66,8.06,0.07\r\n20080831,0.34,-15.16,4.16,-8,6.63\r\n20080930,0.33,-4.69,-4.16,-5.08,1.85\r\n20081031,0.33,-25.47,4.42,-0.6,1.52\r\n20081130,0.28,9.37,-3.91,10.8,6.31\r\n20081231,0.2,-2.15,3.37,9.09,2.43\r\n20090131,0.19,9.65,0.79,6.23,-6.87\r\n20090228,0.17,4.91,-4.33,3.24,3.2\r\n20090331,0.19,14.63,-0.92,7.26,-3.88\r\n20090430,0.18,4.17,3.93,3.3,1.04\r\n20090531,0.19,6.09,-2.62,1.08,3.15\r\n20090630,0.18,12.65,2.59,-8.13,1.4\r\n20090731,0.19,15.55,-8.08,-4.1,4.41\r\n20090831,0.19,-21.82,5.21,8.61,-1.59\r\n20090930,0.18,4.57,5.66,-0.69,3.09\r\n20091031,0.19,8.29,0.42,4.41,0.86\r\n20091130,0.18,7.31,-0.6,9.4,2.2\r\n20091231,0.19,2.25,3.93,2.69,3.88\r\n20100131,0.19,-8.95,-1.24,6.82,1.66\r\n20100228,0.17,2.3,-2.5,3.82,-0.07\r\n20100331,0.19,1.49,1.16,2.28,0.21\r\n20100430,0.18,-7.91,0.09,1.73,5.8\r\n20100531,0.19,-8.7,0,-0.33,-1.66\r\n20100630,0.18,-7.4,2.99,-1.73,1.73\r\n20100731,0.19,11.27,1.17,4.96,-0.83\r\n20100831,0.19,1.78,-3.9,6.97,-1.73\r\n20100930,0.18,0.64,-1.93,-1.17,-1.54\r\n20101031,0.2,12.39,1.57,-4.73,-5.34\r\n20101130,0.2,-4.72,0.4,3.52,10.41\r\n20101231,0.21,-1.14,3.99,-1.68,-0.62\r\n20110131,0.23,-2.75,4.26,-1.85,-1.02\r\n20110228,0.22,4.84,-3.36,5.25,0.26\r\n20110331,0.25,-0.59,1.63,-0.31,1.25\r\n20110430,0.26,-1.51,2.1,-2.48,-0.41\r\n20110531,0.27,-6.44,1.06,-0.96,0.29\r\n20110630,0.26,2.27,0.36,1.23,0.59\r\n20110731,0.29,-1.62,-0.68,2.54,2.9\r\n20110831,0.29,-4.88,0.75,1.78,0.97\r\n20110930,0.28,-9.22,3.68,-2.6,4.77\r\n20111031,0.29,4.03,1.04,0,-2.33\r\n20111130,0.28,-5.47,1.4,1.89,1.44\r\n20111231,0.29,-7.33,7.68,-8.56,3.35\r\n20120131,0.29,2.48,2.01,-4.26,2.17\r\n20120229,0.27,6.97,-3.09,5.79,0.92\r\n20120331,0.29,-7.03,1.23,-0.59,2.83\r\n20120430,0.28,5.46,-1.68,1.58,2.84\r\n20120531,0.29,0.06,-0.69,1,-1.63\r\n20120630,0.27,-5.21,4.75,-1.08,0.97\r\n20120731,0.25,-5.76,4.92,-3.51,-0.94\r\n20120831,0.25,-2.85,-0.84,6.32,2.34\r\n20120930,0.24,2.06,0.15,-2.36,-0.35\r\n20121031,0.25,-1.22,0.59,1.25,2.44\r\n20121130,0.24,-6.35,3.66,-4.89,-1.58\r\n20121231,0.25,15.02,-0.9,1,0.6\r\n20130131,0.25,4.93,0.73,1.65,3\r\n20130228,0.23,0,0.06,3.19,2.75\r\n20130331,0.25,-5.49,0.38,1.48,4.02\r\n20130430,0.24,-2.5,1.33,-0.67,0.5\r\n20130531,0.25,7.79,-3.83,6.1,-1.3\r\n20130630,0.24,-13.46,3.55,-2.27,-0.31\r\n20130731,0.25,2.65,-2.96,3.53,-3.65\r\n20130831,0.25,4.99,0.97,3.78,2.8\r\n20130930,0.24,4.25,-2.89,-0.42,-3.41\r\n20131031,0.25,-2.65,5.47,0.89,2.49\r\n20131130,0.24,4.18,-2.79,4.28,-0.45\r\n20131231,0.25,-4.58,3.4,1.31,1.12\r\n20140131,0.25,-2.3,-4.83,4.22,-1.32\r\n20140228,0.23,0.53,-1.15,4.3,2.89\r\n20140331,0.25,-2.69,4.92,0.86,6.31\r\n20140430,0.24,-1.09,1.91,-1.64,0.29\r\n20140531,0.25,1.06,-3.27,1.7,2.15\r\n20140630,0.24,2.16,-2.19,2.89,0.86\r\n20140731,0.25,7.71,1.88,0.27,2.15\r\n20140831,0.25,1.19,-2.98,3.05,3.13\r\n20140930,0.24,6.96,-3.32,5.73,-2.45\r\n20141031,0.25,1.54,2.64,0.03,-0.89\r\n20141130,0.24,8.83,4.76,-2.89,-3.7\r\n20141231,0.23,15.25,15.17,-17.2,-20.19\r\n20150131,0.23,0.45,-2.47,5.15,11.34\r\n20150228,0.21,4.31,-3.06,1.76,3.07\r\n20150331,0.21,14.92,-4.46,8.78,1.06\r\n20150430,0.2,16.01,-0.91,-0.28,1.48\r\n20150531,0.2,6.99,-10.28,18.41,12.75\r\n20150630,0.18,-8.12,8.76,-3.72,0.75\r\n20150731,0.17,-14.61,6.71,-4.88,4.02\r\n20150831,0.16,-13.81,1.98,0.15,0.8\r\n20150930,0.14,-4.31,2.25,3.54,0.25\r\n20151031,0.14,14.39,-5.95,7.35,-8.57\r\n20151130,0.12,3.24,-0.04,7.9,-3.5\r\n20151231,0.13,3.17,4.21,3.82,3.71\r\n20160131,0.13,-25.09,5.99,-4.81,7.92\r\n20160229,0.12,-2.2,0.83,-0.04,1.71\r\n20160331,0.13,14.52,-0.03,5.65,-7.05\r\n20160430,0.12,-2.71,1.02,1.81,3.3\r\n20160531,0.13,-0.11,1.58,-1.02,-0.41\r\n20160630,0.12,2.21,-0.83,4.32,-4.37\r\n20160731,0.13,1.15,5.96,-1.59,8.61\r\n20160831,0.13,3.71,-0.87,0.89,2.12\r\n20160930,0.12,-2.54,2.08,1.06,0.11\r\n20161031,0.13,2.48,-0.76,1.53,0.21\r\n20161130,0.12,3.64,1.28,-1.79,-0.52\r\n20161231,0.13,-5.65,0.33,-0.14,-0.95\r\n20170131,0.13,0.7,2.85,-3.34,-2.78\r\n20170228,0.11,2.82,0.34,1,1.85\r\n20170331,0.13,-0.84,3.36,-1.01,-0.93\r\n20170430,0.12,-2.27,4.69,-4.12,-1.92\r\n20170531,0.13,-1.91,5.68,-5.79,-3.06\r\n20170630,0.12,4.14,2.66,-0.78,2.37\r\n20170731,0.13,2.45,4.48,-2.03,0.84\r\n20170831,0.13,2.48,-2.28,-0.15,1.82\r\n20170930,0.12,0.62,-0.78,1.1,0.06\r\n20171031,0.13,1.35,3.82,-4.4,4.63\r\n20171130,0.12,-2.93,3.78,-3.88,-0.48\r\n20171231,0.13,-0.12,2.29,-1.78,-1.26\r\n20180131,0.13,3.66,6.67,-6.76,-1.69\r\n20180228,0.11,-5.19,0.19,0.03,1.27\r\n20180331,0.13,-1.41,-6.66,4.8,2.69\r\n20180430,0.12,-3.28,-0.04,-1.21,0.46\r\n20180531,0.13,0.27,4.49,-1.77,2.03\r\n20180630,0.12,-7.89,3.71,-3.46,-0.4\r\n20180731,0.13,0.9,3.66,-0.9,4.08\r\n20180831,0.13,-5.86,0.71,-2.53,2.43\r\n20180930,0.12,2.29,3.09,-3.24,-0.82\r\n20181031,0.13,-8.94,2.55,-1.47,1.8\r\n20181130,0.12,0.86,-3.16,5.06,2.2\r\n20181231,0.13,-4.16,1.94,-0.25,2.44\r\n20190131,0.13,2.88,6.35,-5.11,3.26\r\n20190228,0.11,15.74,-9.55,5.69,-7.22\r\n20190331,0.13,6.44,-2.81,4.66,0.51\r\n20190430,0.12,-1.33,3.06,-2.45,2.6\r\n20190531,0.13,-6.36,0.35,1.69,3.71\r\n20190630,0.12,3.15,3.98,-3.8,0.86\r\n20190731,0.13,-0.17,0.71,-3.08,1.18\r\n20190831,0.13,-0.77,-3.63,-0.74,-3.34\r\n20190930,0.12,0.61,1.13,0.53,0.49\r\n20191031,0.13,1.11,1.35,-2.12,0.69\r\n20191130,0.12,-1.8,1.83,-1.06,-0.3\r\n20191231,0.13,6.86,0.21,1,-0.41\r\n20200131,0.13,-0.7,-3.06,0.81,-5.27\r\n20200229,0.12,-0.89,-3.72,1.12,-6.38\r\n20200331,0.13,-6.27,5.08,2.39,7.63\r\n20200430,0.12,5.07,0.38,-3.77,-0.57\r\n20200531,0.13,0.3,0.12,1.67,0.06\r\n20200630,0.12,8.01,1.03,-1.7,-2.03\r\n20200731,0.13,12.6,0.76,0.1,-2.07\r\n20200831,0.13,1.99,1.7,2.14,1.28\r\n20200930,0.12,-5.79,1.1,-2.14,-1.08\r\n20201031,0.13,0.86,1.24,-1.78,-3.52\r\n20201130,0.12,3.65,4.63,-1.8,1.99\r\n20201231,0.13,2.99,-4.72,-7.99,-2.76\r\n20210131,0.13,0.34,3.36,-9.07,-2.17\r\n20210228,0.13,-0.44,6.43,4.55,1.67\r\n20210331,0.13,-2.93,4.87,5.35,1.15\r\n20210430,0.13,1.77,-4.17,-2.74,2.93\r\n20210531,0.13,5.05,-3.35,0.73,-1.4\r\n20210630,0.13,0.79,-4.6,2.24,1.39\r\n20210731,0.13,-4.1,-3.54,4.96,-3.96\r\n20210831,0.13,2.85,6.8,3.92,-0.44\r\n20210930,0.13,-0.36,4.94,-1.7,5.03\r\n20211031,0.13,-0.27,-4.33,-2.57,6.3\r\n20211130,0.13,1.96,-3.8,11.49,4.85\r\n20211231,0.13,1.01,5.07,3.11,2.24\r\n', 'ch3_daily': 'Daily CH-3 factors,,,,\r\ndate: daily date,,,,\r\nrf_dly:  1-year bank deposit rate converted to daily rate,,,,\r\nmktrf: the excess return on the market,,,,\r\nSMB (Small-minus-big): Size factor,,,,\r\nVMG(Value-minus-growth) Value factor,,,,\r\nUnit: %,,,,\r\n,,,,\r\ndate,rf_dly,mktrf,SMB,VMG\r\n20000104,0.01,3.04,-0.58,-0.59\r\n20000105,0.01,0.3,0.58,-0.49\r\n20000106,0.01,4.11,-0.9,0.76\r\n20000107,0.01,3.74,0.22,1.22\r\n20000110,0.01,1.98,0.57,-0.44\r\n20000111,0.01,-4.22,-0.04,-1.12\r\n20000112,0.01,-2.82,0.3,0.5\r\n20000113,0.01,-0.89,0.01,-0.1\r\n20000114,0.01,-1.04,-0.18,-0.3\r\n20000117,0.01,1.75,0.37,0.81\r\n20000118,0.01,-0.19,0.15,1.05\r\n20000119,0.01,1.23,0.04,0.06\r\n20000120,0.01,1.78,-0.14,-0.01\r\n20000121,0.01,0.32,-1.01,-1.49\r\n20000124,0.01,1.07,0.33,-0.01\r\n20000125,0.01,-0.2,0.68,0.91\r\n20000126,0.01,0.36,-0.43,-0.1\r\n20000127,0.01,1.79,-1.12,-0.51\r\n20000128,0.01,2.08,-0.26,-0.83\r\n20000214,0.0148,9.09,-0.23,-0.02\r\n20000215,0.0148,0.44,-0.76,-3.59\r\n20000216,0.0148,1.7,0.65,-3.33\r\n20000217,0.0148,-3.29,0.21,1.44\r\n20000218,0.0148,1.69,0.1,-1.48\r\n20000221,0.0148,0.36,2.39,2.26\r\n20000222,0.0148,-2.8,-0.72,-0.23\r\n20000223,0.0148,-2.69,0.4,0.45\r\n20000224,0.0148,2.74,-0.07,-0.42\r\n20000225,0.0148,-0.11,0.34,-0.92\r\n20000228,0.0148,4.6,-0.66,-1.41\r\n20000229,0.0148,0.26,-0.5,0.6\r\n20000301,0.0082,-1.02,1.6,0.96\r\n20000302,0.0082,0.72,0.09,-1.02\r\n20000303,0.0082,1.29,0.41,0.93\r\n20000306,0.0082,-3.73,-0.05,1.46\r\n20000307,0.0082,1.17,-0.44,0.97\r\n20000308,0.0082,1.81,-0.13,-1.27\r\n20000309,0.0082,0.17,-0.5,0.26\r\n20000310,0.0082,-1.41,-0.31,-1.83\r\n20000313,0.0082,1.52,0.28,-0.64\r\n20000314,0.0082,-2.64,1.06,0.18\r\n20000315,0.0082,-0.09,1.29,0.11\r\n20000316,0.0082,-4.51,-0.24,0.36\r\n20000317,0.0082,3.36,-0.21,-1.2\r\n20000320,0.0082,2.51,0.91,-0.78\r\n20000321,0.0082,0.59,0.86,0.44\r\n20000322,0.0082,0.82,-0.34,-0.68\r\n20000323,0.0082,1.48,0.8,-0.21\r\n20000324,0.0082,-0.95,0.28,-0.28\r\n20000327,0.0082,2.54,1.64,1.31\r\n20000328,0.0082,0.9,0.16,0.83\r\n20000329,0.0082,-0.23,-0.35,-0.66\r\n20000330,0.0082,1.11,0.57,0.5\r\n20000331,0.0082,-0.74,-0.04,-0.22\r\n20000403,0.0092,-0.14,-0.19,-0.34\r\n20000404,0.0092,-2.22,-0.34,0.07\r\n20000405,0.0092,0.58,0.75,-0.02\r\n20000406,0.0092,2.13,0.55,-0.41\r\n20000407,0.0092,0.69,0.07,0.53\r\n20000410,0.0092,0.45,0.72,1.19\r\n20000411,0.0092,-0.17,-0.79,-0.71\r\n20000412,0.0092,-0.87,0.34,0.99\r\n20000413,0.0092,1.27,-0.42,0.26\r\n20000414,0.0092,0.05,-0.24,0\r\n20000417,0.0092,-2.18,0.15,0.17\r\n20000418,0.0092,0.98,-0.04,0.4\r\n20000419,0.0092,0.71,0.16,-0.13\r\n20000420,0.0092,0.89,-0.14,-0.36\r\n20000421,0.0092,-0.44,0.27,0.13\r\n20000424,0.0092,-0.38,0.53,0.57\r\n20000425,0.0092,-0.08,-0.39,0.59\r\n20000426,0.0092,0,-0.31,0.08\r\n20000427,0.0092,-1.29,-0.49,-0.11\r\n20000428,0.0092,1.81,-0.65,-0.55\r\n20000508,0.0105,-0.18,0.41,0.85\r\n20000509,0.0105,-1.64,0.4,0.06\r\n20000510,0.0105,-3.14,-0.07,-0.42\r\n20000511,0.0105,-1.77,-0.17,-0.45\r\n20000512,0.0105,-0.21,0.06,0.33\r\n20000515,0.0105,-1.13,0.55,0.55\r\n20000516,0.0105,1.14,0.33,-0.15\r\n20000517,0.0105,0.11,-0.37,-0.76\r\n20000518,0.0105,1.99,0.11,0.2\r\n20000519,0.0105,1.24,0.55,0.31\r\n20000522,0.0105,2.59,0.05,0.03\r\n20000523,0.0105,0.16,0.26,-0.03\r\n20000524,0.0105,1.2,-0.04,0.21\r\n20000525,0.0105,1.3,0.4,0.21\r\n20000526,0.0105,0.16,-0.35,0.21\r\n20000529,0.0105,1.02,0.03,0.78\r\n20000530,0.0105,-1.13,-0.43,-0.56\r\n20000531,0.0105,0.64,0.23,0.73\r\n20000601,0.0083,0.49,-0.18,0.2\r\n20000602,0.0083,0.73,-0.56,0.61\r\n20000605,0.0083,-0.84,-0.16,0.03\r\n20000606,0.0083,1.43,0.27,0.18\r\n20000607,0.0083,0.37,0.11,0.23\r\n20000608,0.0083,-0.66,0.27,0.06\r\n20000609,0.0083,-1.22,-0.15,-0.77\r\n20000612,0.0083,0.35,0.2,-0.13\r\n20000613,0.0083,1.26,0.29,0.63\r\n20000614,0.0083,-0.3,0.17,0.72\r\n20000615,0.0083,0.33,0.24,-0.02\r\n20000616,0.0083,-0.64,0.27,-0.32\r\n20000619,0.0083,-0.36,0.03,0.66\r\n20000620,0.0083,-0.88,0.03,-0.09\r\n20000621,0.0083,1.01,0.06,0.2\r\n20000622,0.0083,0.15,-0.02,0.17\r\n20000623,0.0083,1.23,-0.12,-0.14\r\n20000626,0.0083,0.67,0.41,1.27\r\n20000627,0.0083,-0.14,0.11,0.78\r\n20000628,0.0083,0.16,0.21,0.22\r\n20000629,0.0083,-0.28,-0.29,0.47\r\n20000630,0.0083,-0.46,-0.11,0.5\r\n20000703,0.009,-1.67,0.38,0.52\r\n20000704,0.009,0.74,0.65,0.42\r\n20000705,0.009,0.04,-0.01,0.06\r\n20000706,0.009,-0.31,0.41,0.11\r\n20000707,0.009,1.51,-0.13,0.6\r\n20000710,0.009,0.81,0.33,1.46\r\n20000711,0.009,0.65,-0.81,-0.45\r\n20000712,0.009,0.53,-0.03,0.66\r\n20000713,0.009,0.15,-0.79,-0.79\r\n20000714,0.009,0.37,-0.11,0.49\r\n20000717,0.009,-0.49,0.04,-0.13\r\n20000718,0.009,0.64,0.17,0.16\r\n20000719,0.009,0.18,0.26,-0.31\r\n20000720,0.009,-0.3,0.3,0.13\r\n20000721,0.009,0.18,0.43,0\r\n20000724,0.009,-0.4,0.12,0.3\r\n20000725,0.009,0.3,-0.01,-0.36\r\n20000726,0.009,0.73,0.11,-0.05\r\n20000727,0.009,0.01,-0.2,-0.52\r\n20000728,0.009,0.09,0.12,0.05\r\n20000731,0.009,0.55,-0.17,-0.51\r\n20000801,0.0082,0.2,0.14,0.4\r\n20000802,0.0082,0.11,-0.12,0.01\r\n20000803,0.0082,0.36,-0.36,-0.21\r\n20000804,0.0082,-0.63,-0.53,-0.54\r\n20000807,0.0082,0.46,0.06,-0.42\r\n20000808,0.0082,0.57,0.27,-0.47\r\n20000809,0.0082,0.38,0.15,-0.11\r\n20000810,0.0082,0.09,-0.1,-0.29\r\n20000811,0.0082,-0.14,0,0.27\r\n20000814,0.0082,0.46,0.39,0.46\r\n20000815,0.0082,-0.17,0.26,0.04\r\n20000816,0.0082,0.29,0.3,0.02\r\n20000817,0.0082,0.19,-0.11,-0.47\r\n20000818,0.0082,-0.1,0.39,-0.22\r\n20000821,0.0082,0.62,0.46,0.1\r\n20000822,0.0082,-1.57,-0.37,0.12\r\n20000823,0.0082,-0.98,0.05,-0.15\r\n20000824,0.0082,0.56,0.18,-0.2\r\n20000825,0.0082,0.76,0.12,-0.21\r\n20000828,0.0082,0.51,0.97,-0.08\r\n20000829,0.0082,0.22,0.15,0.24\r\n20000830,0.0082,-1.27,-0.05,0.34\r\n20000831,0.0082,-2.62,-0.12,-0.12\r\n20000901,0.0087,-1.01,0.17,0.19\r\n20000904,0.0087,0.5,0.54,0.05\r\n20000905,0.0087,-2.11,0.08,-0.17\r\n20000906,0.0087,-2,0.37,-0.35\r\n20000907,0.0087,1.26,-0.09,0.1\r\n20000908,0.0087,0.37,0.15,-0.41\r\n20000911,0.0087,0.5,0.56,0.08\r\n20000912,0.0087,0.61,0.3,0.54\r\n20000913,0.0087,0.66,-0.49,-0.06\r\n20000914,0.0087,-0.5,0.24,-0.5\r\n20000915,0.0087,-0.09,0.27,-0.17\r\n20000918,0.0087,-1.58,0,-0.04\r\n20000919,0.0087,-2.25,0.06,0.13\r\n20000920,0.0087,0.97,0.26,0.14\r\n20000921,0.0087,-0.78,-0.08,-0.01\r\n20000922,0.0087,-0.88,-0.1,-0.07\r\n20000925,0.0087,-0.93,-0.19,-0.18\r\n20000926,0.0087,0.68,0.01,0.1\r\n20000927,0.0087,1.34,-0.2,0.3\r\n20000928,0.0087,-1.35,0.12,-0.19\r\n20000929,0.0087,1.09,-0.03,-0.07\r\n20001009,0.0111,0.1,0.9,-0.33\r\n20001010,0.0111,1.25,0.44,0.23\r\n20001011,0.0111,-0.45,-0.14,-0.15\r\n20001012,0.0111,-0.57,0.47,-0.37\r\n20001013,0.0111,-0.31,-0.01,0.09\r\n20001016,0.0111,-0.79,-0.05,-0.3\r\n20001017,0.0111,0.26,0.24,0.14\r\n20001018,0.0111,0.85,-0.13,0.18\r\n20001019,0.0111,-0.57,0.13,0.16\r\n20001020,0.0111,1.71,-0.26,0.42\r\n20001023,0.0111,0.43,0.57,0.07\r\n20001024,0.0111,-0.18,0.23,-0.38\r\n20001025,0.0111,0.71,-0.07,0.19\r\n20001026,0.0111,-0.68,0.11,-0.12\r\n20001027,0.0111,0.13,0.31,0.12\r\n20001030,0.0111,0.21,0.26,0.27\r\n20001031,0.0111,-0.51,-0.23,-0.01\r\n20001101,0.0083,0.79,-0.03,0.13\r\n20001102,0.0083,0.43,-0.07,0.53\r\n20001103,0.0083,-0.21,0.12,0.12\r\n20001106,0.0083,0.4,0.27,0.33\r\n20001107,0.0083,0.18,0.2,0.14\r\n20001108,0.0083,0.68,-0.08,0.3\r\n20001109,0.0083,1.21,-0.32,-0.17\r\n20001110,0.0083,0.82,-0.46,0.13\r\n20001113,0.0083,1.25,-0.09,0.31\r\n20001114,0.0083,0.68,0.22,0.66\r\n20001115,0.0083,-0.14,0.33,-0.31\r\n20001116,0.0083,0.64,0.23,-0.11\r\n20001117,0.0083,-0.15,-0.25,-0.15\r\n20001120,0.0083,0.32,0.34,-0.03\r\n20001121,0.0083,-0.19,0.07,0.19\r\n20001122,0.0083,0.91,0.19,-0.2\r\n20001123,0.0083,0.29,0.4,0.34\r\n20001124,0.0083,-3.11,-0.55,-0.33\r\n20001127,0.0083,-0.2,0.49,0.13\r\n20001128,0.0083,1.39,0.23,-0.26\r\n20001129,0.0083,-0.6,0.23,-0.08\r\n20001130,0.0083,0.1,-0.14,0.17\r\n20001201,0.009,0.46,0.1,0.2\r\n20001204,0.009,0.47,0.46,-0.26\r\n20001205,0.009,-0.16,-0.29,-0.16\r\n20001206,0.009,-0.87,-0.24,-0.17\r\n20001207,0.009,-0.27,-0.06,-0.79\r\n20001208,0.009,0.05,0.09,0.32\r\n20001211,0.009,-1.44,0.08,-0.17\r\n20001212,0.009,0.62,0.33,0.09\r\n20001213,0.009,-0.1,0.37,0.22\r\n20001214,0.009,-0.18,0.02,-0.05\r\n20001215,0.009,-0.52,-0.07,0.44\r\n20001218,0.009,0.16,0.2,0.24\r\n20001219,0.009,0.12,0.06,-0.23\r\n20001220,0.009,1.03,0.19,0.07\r\n20001221,0.009,0.17,0.05,0.05\r\n20001222,0.009,-0.42,0.25,-0.19\r\n20001225,0.009,-0.29,0.61,0.34\r\n20001226,0.009,0.3,0.12,0.53\r\n20001227,0.009,-0.86,-0.11,0.35\r\n20001228,0.009,-0.03,0.07,0.76\r\n20001229,0.009,0.89,-0.33,0.09\r\n20010102,0.0135,1.38,0.29,0.32\r\n20010103,0.0135,0.91,0.1,-0.19\r\n20010104,0.0135,-0.37,0.37,-0.35\r\n20010105,0.0135,0.29,0.46,0.48\r\n20010108,0.0135,-1.14,0.11,1.08\r\n20010109,0.0135,-0.13,-0.6,-0.12\r\n20010110,0.0135,1.02,0.19,-0.13\r\n20010111,0.0135,-0.35,-0.11,0.24\r\n20010112,0.0135,-0.75,-0.1,0.03\r\n20010115,0.0135,-3.33,-0.29,-0.14\r\n20010116,0.0135,0.56,-0.04,0.71\r\n20010117,0.0135,-0.52,0.07,0.31\r\n20010118,0.0135,0.23,-0.13,0.16\r\n20010119,0.0135,1.08,0.05,-0.15\r\n20010205,0.0095,-2.62,0.09,0.16\r\n20010206,0.0095,-0.76,0.01,0.45\r\n20010207,0.0095,-0.73,0.04,0.66\r\n20010208,0.0095,-2.58,0.01,0.27\r\n20010209,0.0095,1.36,0.03,-0.15\r\n20010212,0.0095,0.1,-0.08,0.31\r\n20010213,0.0095,0.17,0.23,-0.11\r\n20010214,0.0095,0.03,-0.03,0.31\r\n20010215,0.0095,-0.94,-0.12,0.06\r\n20010216,0.0095,-0.29,-0.09,0.11\r\n20010219,0.0095,1.08,0.03,-0.27\r\n20010220,0.0095,-0.67,-0.2,0.07\r\n20010221,0.0095,-2.43,-0.24,1.09\r\n20010222,0.0095,-0.28,-0.24,0.27\r\n20010223,0.0095,1.42,-0.3,0.18\r\n20010226,0.0095,1.21,0.07,0.55\r\n20010227,0.0095,0.45,0.12,-0.31\r\n20010228,0.0095,-0.32,0.15,0\r\n20010301,0.0086,0.4,-0.07,0.1\r\n20010302,0.0086,0.82,0.18,0.09\r\n20010305,0.0086,-0.22,0.28,-0.13\r\n20010306,0.0086,0.1,-0.14,0.34\r\n20010307,0.0086,0.2,0.08,0.06\r\n20010308,0.0086,0.58,-0.26,-0.39\r\n20010309,0.0086,0.58,-0.12,-0.49\r\n20010312,0.0086,-0.1,0.64,-0.03\r\n20010313,0.0086,-0.78,0.11,-0.18\r\n20010314,0.0086,0.91,-0.12,-0.17\r\n20010315,0.0086,0.19,0.37,0.14\r\n20010316,0.0086,-0.11,0.23,0.22\r\n20010319,0.0086,0.69,0.3,-0.21\r\n20010320,0.0086,0.53,0.04,-0.28\r\n20010321,0.0086,0.94,-0.52,0.11\r\n20010322,0.0086,0.78,-0.18,-0.11\r\n20010323,0.0086,-1.53,-0.01,-0.34\r\n20010326,0.0086,1.58,0.3,-0.01\r\n20010327,0.0086,0.93,0.11,0.13\r\n20010328,0.0086,0,0.14,0.15\r\n20010329,0.0086,-0.37,0.24,0.06\r\n20010330,0.0086,0.41,0.13,0.18\r\n20010402,0.0087,0.38,-0.06,0.16\r\n20010403,0.0087,-0.23,-0.13,-0.23\r\n20010404,0.0087,-1.53,-0.18,-0.54\r\n20010405,0.0087,0.94,0.08,0.02\r\n20010406,0.0087,-0.22,0.07,-0.12\r\n20010409,0.0087,0.27,0.45,0.17\r\n20010410,0.0087,0.46,-0.06,-0.24\r\n20010411,0.0087,0.12,-0.16,0.13\r\n20010412,0.0087,0.23,-0.26,0.19\r\n20010413,0.0087,0.4,0.17,0.14\r\n20010416,0.0087,0.92,0,0.26\r\n20010417,0.0087,0.18,0.08,-0.4\r\n20010418,0.0087,-0.39,-0.03,-0.28\r\n20010419,0.0087,-0.13,0.19,-0.25\r\n20010420,0.0087,-0.96,0.22,-0.85\r\n20010423,0.0087,-0.28,0.21,-0.27\r\n20010424,0.0087,0.53,0.24,-0.27\r\n20010425,0.0087,-1.92,-0.02,0.12\r\n20010426,0.0087,-0.64,-0.01,0.14\r\n20010427,0.0087,0.47,0,0.18\r\n20010430,0.0087,0.07,0.18,0.06\r\n20010508,0.0105,0.73,0.3,-0.05\r\n20010509,0.0105,-0.18,0.19,-0.32\r\n20010510,0.0105,0.49,0.15,-0.66\r\n20010511,0.0105,0.18,0.15,0.27\r\n20010514,0.0105,0.87,-0.15,0.16\r\n20010515,0.0105,0.5,0.06,-0.1\r\n20010516,0.0105,0.4,0.06,0.03\r\n20010517,0.0105,-0.3,-0.04,-0.02\r\n20010518,0.0105,0.17,0.19,-0.18\r\n20010521,0.0105,0.12,0.45,-0.13\r\n20010522,0.0105,-0.16,-0.01,-0.26\r\n20010523,0.0105,-0.54,-0.33,-0.38\r\n20010524,0.0105,-0.12,0.49,0.09\r\n20010525,0.0105,-0.23,0.1,0.14\r\n20010528,0.0105,-0.83,0.26,-0.1\r\n20010529,0.0105,0.17,0.05,0.05\r\n20010530,0.0105,1.28,0.2,0.07\r\n20010531,0.0105,-0.01,0.45,-0.36\r\n20010601,0.0087,0.27,-0.16,-0.11\r\n20010604,0.0087,0.22,0.59,-0.18\r\n20010605,0.0087,0.33,0.15,-0.33\r\n20010606,0.0087,0.16,0.2,0.32\r\n20010607,0.0087,-0.42,-0.02,0.35\r\n20010608,0.0087,-0.19,-0.05,0.04\r\n20010611,0.0087,-0.18,0.36,0.21\r\n20010612,0.0087,0.31,0.41,0.22\r\n20010613,0.0087,0.66,0.22,-0.08\r\n20010614,0.0087,-1.52,-0.55,0.54\r\n20010615,0.0087,0.26,0.34,0.39\r\n20010618,0.0087,-1.9,-0.3,0.51\r\n20010619,0.0087,0.24,-0.03,-0.09\r\n20010620,0.0087,-0.36,-0.04,-0.15\r\n20010621,0.0087,1.06,0.18,0.06\r\n20010622,0.0087,0.82,-0.35,0.14\r\n20010625,0.0087,0.82,0.45,0.24\r\n20010626,0.0087,0.15,0.23,-0.07\r\n20010627,0.0087,-0.15,0.26,0.14\r\n20010628,0.0087,-0.42,-0.27,-0.12\r\n20010629,0.0087,0.04,-0.12,-0.04\r\n20010702,0.0086,-0.48,-0.06,0.21\r\n20010703,0.0086,0.2,0.04,0.09\r\n20010704,0.0086,-0.36,0.03,-0.02\r\n20010705,0.0086,-0.9,-0.05,0.09\r\n20010706,0.0086,-0.45,-0.1,0.02\r\n20010709,0.0086,0.02,0.18,0.13\r\n20010710,0.0086,0.82,0.36,-0.15\r\n20010711,0.0086,-0.87,-0.16,-0.16\r\n20010712,0.0086,-0.16,-0.1,0.01\r\n20010713,0.0086,-0.17,0.05,0.1\r\n20010716,0.0086,-0.67,-0.14,0.09\r\n20010717,0.0086,-0.32,0.04,0.09\r\n20010718,0.0086,0.3,-0.1,-0.08\r\n20010719,0.0086,0.23,-0.12,0.07\r\n20010720,0.0086,1.21,0.35,0.05\r\n20010723,0.0086,-0.35,0.28,-0.25\r\n20010724,0.0086,-1.45,0.09,-0.39\r\n20010725,0.0086,-1.06,-0.01,-0.49\r\n20010726,0.0086,-1.02,-0.19,-0.04\r\n20010727,0.0086,-1.38,0.19,0.14\r\n20010730,0.0086,-5.38,-0.54,0.12\r\n20010731,0.0086,-1.92,-0.2,0.09\r\n20010801,0.0082,3.5,0.49,0.28\r\n20010802,0.0082,-1.44,-0.12,0.03\r\n20010803,0.0082,0.09,0.21,-0.1\r\n20010806,0.0082,-3.76,-0.11,0.12\r\n20010807,0.0082,1.17,0.08,-0.09\r\n20010808,0.0082,-0.56,0.5,-0.39\r\n20010809,0.0082,1.71,0.22,-0.09\r\n20010810,0.0082,1.44,0.05,0.24\r\n20010813,0.0082,0.15,-0.04,-0.05\r\n20010814,0.0082,-0.95,-0.06,-0.32\r\n20010815,0.0082,0.44,0.04,-0.02\r\n20010816,0.0082,-1.35,-0.01,-0.04\r\n20010817,0.0082,0.09,-0.21,0.23\r\n20010820,0.0082,0.56,-0.01,-0.1\r\n20010821,0.0082,0.38,0.26,-0.33\r\n20010822,0.0082,-1.65,0.05,-0.12\r\n20010823,0.0082,0.03,0.1,-0.27\r\n20010824,0.0082,-0.88,-0.33,0.02\r\n20010827,0.0082,-3.19,-0.38,-0.04\r\n20010828,0.0082,0.84,0.18,0.06\r\n20010829,0.0082,0.37,-0.1,0.02\r\n20010830,0.0082,-0.7,-0.1,-0.06\r\n20010831,0.0082,-0.14,0.1,0.02\r\n20010903,0.0092,-0.92,0.13,0.47\r\n20010904,0.0092,2.5,0.13,-0.02\r\n20010905,0.0092,0.3,0.17,0.13\r\n20010906,0.0092,-0.36,0.03,-0.02\r\n20010907,0.0092,-2.94,-0.29,-0.09\r\n20010910,0.0092,2.6,0.07,-0.22\r\n20010911,0.0092,0.23,0.28,0.3\r\n20010912,0.0092,-1.06,-0.23,0.22\r\n20010913,0.0092,-0.42,0.1,0.12\r\n20010914,0.0092,-1.16,-0.19,0.48\r\n20010917,0.0092,-1.93,-0.17,0.69\r\n20010918,0.0092,0.72,0.05,0\r\n20010919,0.0092,1.39,-0.06,0.46\r\n20010920,0.0092,-0.18,-0.12,0.22\r\n20010921,0.0092,-0.64,-0.28,0.21\r\n20010924,0.0092,-0.72,0.01,-0.04\r\n20010925,0.0092,-0.18,-0.18,-0.31\r\n20010926,0.0092,-1.43,-0.31,0.34\r\n20010927,0.0092,-0.27,-0.03,0.06\r\n20010928,0.0092,0.05,-0.33,0.38\r\n20011008,0.0105,-2.28,-0.63,0.31\r\n20011009,0.0105,0.97,-0.14,-0.15\r\n20011010,0.0105,-3.41,-0.98,-0.01\r\n20011011,0.0105,-2.93,-0.32,-0.42\r\n20011012,0.0105,3.1,0.18,0.11\r\n20011015,0.0105,-0.1,-0.07,-0.6\r\n20011016,0.0105,-1.73,-0.07,0.26\r\n20011017,0.0105,-0.83,0.04,0.08\r\n20011018,0.0105,-1.66,-0.26,0.02\r\n20011019,0.0105,-2.89,-0.46,0.1\r\n20011022,0.0105,-3.23,-0.3,-0.34\r\n20011023,0.0105,9.81,-0.04,0.43\r\n20011024,0.0105,3.08,1.64,0.23\r\n20011025,0.0105,-2.52,-0.49,-0.02\r\n20011026,0.0105,-0.1,-0.17,-0.29\r\n20011029,0.0105,1.39,0.2,-0.13\r\n20011030,0.0105,-1.03,-0.01,0.1\r\n20011031,0.0105,0.33,0.19,-0.1\r\n20011101,0.0083,1.07,-0.02,0.13\r\n20011102,0.0083,-0.88,0.16,0.16\r\n20011105,0.0083,-0.87,-0.02,-0.19\r\n20011106,0.0083,-0.44,-0.17,0.15\r\n20011107,0.0083,-4.86,-0.67,-0.11\r\n20011108,0.0083,0.95,-0.12,-0.23\r\n20011109,0.0083,1.72,-0.07,-0.04\r\n20011112,0.0083,-0.69,-0.04,-0.08\r\n20011113,0.0083,-0.37,-0.47,0.11\r\n20011114,0.0083,0.06,0.3,-0.04\r\n20011115,0.0083,0.14,0.19,0.2\r\n20011116,0.0083,1.52,0.44,0.23\r\n20011119,0.0083,1.21,0.55,0.3\r\n20011120,0.0083,1.9,0.6,0.64\r\n20011121,0.0083,0.53,0.02,0.23\r\n20011122,0.0083,0.74,0.31,0\r\n20011123,0.0083,-0.23,-0.05,0.22\r\n20011126,0.0083,-0.42,0.1,-0.17\r\n20011127,0.0083,1.03,0.28,-0.05\r\n20011128,0.0083,0.79,0.31,0.17\r\n20011129,0.0083,-0.16,0.39,-0.14\r\n20011130,0.0083,0.87,0.31,-0.12\r\n20011203,0.009,0.83,0.18,0.31\r\n20011204,0.009,0.48,-0.05,0.21\r\n20011205,0.009,-1.48,-0.5,0.9\r\n20011206,0.009,0.52,0.29,0.16\r\n20011207,0.009,-0.16,0.1,0.06\r\n20011210,0.009,0.06,0.03,-0.23\r\n20011211,0.009,-1.17,0.02,0.12\r\n20011212,0.009,-0.93,-0.14,0.2\r\n20011213,0.009,-1.57,-0.14,0.15\r\n20011214,0.009,-0.8,-0.2,0.12\r\n20011217,0.009,-0.65,0.09,0.78\r\n20011218,0.009,1.46,0.25,0.39\r\n20011219,0.009,-0.97,0,-0.02\r\n20011220,0.009,-2.43,-0.26,0.04\r\n20011221,0.009,0.53,0.11,0.04\r\n20011224,0.009,-1.94,-0.28,0.28\r\n20011225,0.009,0.27,-0.06,0.42\r\n20011226,0.009,1.28,0.12,0.37\r\n20011227,0.009,-0.46,-0.04,0.32\r\n20011228,0.009,0.59,-0.36,0.64\r\n20011231,0.009,0.4,-0.09,0.45\r\n20020104,0.0095,-2.02,-0.09,-0.59\r\n20020107,0.0095,-1.11,-0.19,0.01\r\n20020108,0.0095,-0.68,-0.17,-0.01\r\n20020109,0.0095,-1.69,-0.54,0.48\r\n20020110,0.0095,0.85,0.2,0.08\r\n20020111,0.0095,-2.59,-0.38,0.25\r\n20020114,0.0095,-3.61,-0.55,0.24\r\n20020115,0.0095,-1.85,-0.5,0.55\r\n20020116,0.0095,1.22,0.03,0.95\r\n20020117,0.0095,-4.47,-1.31,0.61\r\n20020118,0.0095,-0.5,-0.43,1.8\r\n20020121,0.0095,-3.78,-1.43,1.16\r\n20020122,0.0095,-0.26,-0.96,1.63\r\n20020123,0.0095,6.71,1.14,0.59\r\n20020124,0.0095,0.74,0.32,-1.17\r\n20020125,0.0095,-0.04,0.19,-0.33\r\n20020128,0.0095,-6.36,-1.02,-0.95\r\n20020129,0.0095,2.6,0.71,0.43\r\n20020130,0.0095,0.54,0.72,-0.65\r\n20020131,0.0095,6.92,0.91,0.4\r\n20020201,0.0165,-0.4,0.41,0.1\r\n20020204,0.0165,1.71,0.82,0.26\r\n20020205,0.0165,0.53,-0.13,0.16\r\n20020206,0.0165,-2.3,-0.88,-0.09\r\n20020207,0.0165,2.05,0.9,0.66\r\n20020208,0.0165,-0.47,-0.09,0.22\r\n20020225,0.0165,1.72,0.63,-0.19\r\n20020226,0.0165,0.06,0.3,-0.28\r\n20020227,0.0165,0.26,0.31,-0.3\r\n20020228,0.0165,-0.96,-0.07,-0.14\r\n20020301,0.008,-1.43,-0.52,0.05\r\n20020304,0.008,1.6,0.51,-0.28\r\n20020305,0.008,2.26,0.43,0.1\r\n20020306,0.008,0.84,0.06,-0.77\r\n20020307,0.008,2.87,0.55,-0.53\r\n20020308,0.008,1.84,0.28,0.08\r\n20020311,0.008,1.59,0.91,-0.12\r\n20020312,0.008,-1.28,-0.47,-0.55\r\n20020313,0.008,-1.35,-1,0.25\r\n20020314,0.008,2.62,0.91,-0.28\r\n20020315,0.008,-2.6,-0.55,-0.17\r\n20020318,0.008,0.19,-0.1,-0.13\r\n20020319,0.008,2.25,0.55,0.13\r\n20020320,0.008,0.98,0.33,0\r\n20020321,0.008,0.23,-0.02,0.17\r\n20020322,0.008,-1.01,0.04,-0.09\r\n20020325,0.008,0.31,0.35,0\r\n20020326,0.008,-1.57,0,-0.07\r\n20020327,0.008,0.36,0.22,0.04\r\n20020328,0.008,-0.17,0.12,0.01\r\n20020329,0.008,-2.85,-0.52,-0.12\r\n20020401,0.0074,0.4,0.23,-0.16\r\n20020402,0.0074,-1.57,-0.43,-0.13\r\n20020403,0.0074,0.66,-0.02,0\r\n20020404,0.0074,2.88,0.4,-0.09\r\n20020405,0.0074,-0.38,0.11,-0.17\r\n20020408,0.0074,0.33,0.12,-0.1\r\n20020409,0.0074,1.63,0.46,0.12\r\n20020410,0.0074,0.6,0.11,-0.16\r\n20020411,0.0074,-1.66,-0.39,-0.14\r\n20020412,0.0074,0.61,0.05,-0.07\r\n20020415,0.0074,-0.47,0.19,-0.2\r\n20020416,0.0074,-0.41,0.01,-0.1\r\n20020417,0.0074,0.09,0.1,0.15\r\n20020418,0.0074,-0.99,-0.31,0.27\r\n20020419,0.0074,0.3,0.25,0.03\r\n20020422,0.0074,-0.44,-0.17,-0.09\r\n20020423,0.0074,-0.28,0.09,0.11\r\n20020424,0.0074,-0.79,-0.28,-0.05\r\n20020425,0.0074,0.22,-0.04,-0.24\r\n20020426,0.0074,1.51,0.52,-0.2\r\n20020429,0.0074,1.56,0.38,-0.74\r\n20020430,0.0074,0.13,-0.14,0.25\r\n20020508,0.0093,-0.86,-0.14,0.09\r\n20020509,0.0093,0.09,0.01,-0.38\r\n20020510,0.0093,-0.73,-0.05,0.01\r\n20020513,0.0093,-0.97,0.12,-0.41\r\n20020514,0.0093,-0.45,0.06,-0.29\r\n20020515,0.0093,-1.03,0.04,0.16\r\n20020516,0.0093,-3.21,-0.82,0.31\r\n20020517,0.0093,1.19,-0.12,-0.23\r\n20020520,0.0093,-1.53,-0.11,-0.42\r\n20020521,0.0093,3.14,0.84,-0.22\r\n20020522,0.0093,-1.39,-0.21,-0.01\r\n20020523,0.0093,0.46,0.18,-0.12\r\n20020524,0.0093,-1.4,-0.19,-0.43\r\n20020527,0.0093,0.28,0.23,0.07\r\n20020528,0.0093,0.22,-0.2,0.03\r\n20020529,0.0093,-2.05,-0.68,0.13\r\n20020530,0.0093,-0.09,-0.11,-0.07\r\n20020531,0.0093,-0.51,-0.18,-0.03\r\n20020603,0.0081,-2.11,-0.65,0.4\r\n20020604,0.0081,0.6,-0.2,-0.11\r\n20020605,0.0081,-2.05,-1,0.78\r\n20020606,0.0081,4.06,1.06,-0.24\r\n20020607,0.0081,0.67,0.5,0.03\r\n20020610,0.0081,-0.74,-0.28,0.14\r\n20020611,0.0081,0.88,0.22,-0.33\r\n20020612,0.0081,-0.45,-0.13,0\r\n20020613,0.0081,-0.83,-0.38,0.04\r\n20020614,0.0081,-1.14,-0.41,0.58\r\n20020617,0.0081,0.21,-0.04,-0.08\r\n20020618,0.0081,2.02,0.68,-0.04\r\n20020619,0.0081,-1.02,-0.25,0.14\r\n20020620,0.0081,0.75,0,-0.34\r\n20020621,0.0081,3.29,0.92,0.16\r\n20020624,0.0081,9.29,0.09,0.43\r\n20020625,0.0081,-0.45,-1.41,-0.74\r\n20020626,0.0081,0.42,0.41,-0.33\r\n20020627,0.0081,0.36,-0.75,-0.14\r\n20020628,0.0081,0.16,-1.11,0.59\r\n20020701,0.0073,-1.09,0.43,0.15\r\n20020702,0.0073,0.74,0.65,-0.19\r\n20020703,0.0073,0.23,0.08,0.11\r\n20020704,0.0073,-0.98,0.14,0.41\r\n20020705,0.0073,0.41,0.15,-0.15\r\n20020708,0.0073,0.73,0.26,0.07\r\n20020709,0.0073,-0.21,-0.55,-0.1\r\n20020710,0.0073,-1.68,0.12,0.03\r\n20020711,0.0073,-0.16,0.65,0.47\r\n20020712,0.0073,0.11,-0.05,0.03\r\n20020715,0.0073,-0.66,0.05,-0.04\r\n20020716,0.0073,0.22,-0.18,-0.24\r\n20020717,0.0073,0.56,0.02,-0.13\r\n20020718,0.0073,0.21,-0.09,0.23\r\n20020719,0.0073,0.39,0.27,-0.06\r\n20020722,0.0073,-1.61,0.06,0.16\r\n20020723,0.0073,-0.47,-0.02,0.3\r\n20020724,0.0073,-0.2,-0.19,0.02\r\n20020725,0.0073,-0.46,0.09,0.06\r\n20020726,0.0073,-0.42,-0.01,0.36\r\n20020729,0.0073,0.48,-0.23,-0.17\r\n20020730,0.0073,0.22,-0.18,0.02\r\n20020731,0.0073,-1,0.04,0.15\r\n20020801,0.0076,0.05,-0.04,-0.16\r\n20020802,0.0076,0.48,-0.12,-0.1\r\n20020805,0.0076,-0.43,0.22,0.02\r\n20020806,0.0076,1.05,-0.25,-0.41\r\n20020807,0.0076,-0.21,0.03,0.3\r\n20020808,0.0076,-0.54,-0.06,0.1\r\n20020809,0.0076,-0.73,-0.03,0.13\r\n20020812,0.0076,-0.7,-0.14,-0.03\r\n20020813,0.0076,0.13,0,-0.06\r\n20020814,0.0076,0.65,0.07,0.03\r\n20020815,0.0076,-0.75,-0.01,0.22\r\n20020816,0.0076,0.73,-0.01,-0.1\r\n20020819,0.0076,-0.18,-0.21,0.24\r\n20020820,0.0076,1.61,-0.45,-0.13\r\n20020821,0.0076,0.14,-0.08,-0.19\r\n20020822,0.0076,0.78,0.36,0.03\r\n20020823,0.0076,-0.18,-0.19,0.31\r\n20020826,0.0076,-0.82,0.36,0.44\r\n20020827,0.0076,0.46,0.53,-0.14\r\n20020828,0.0076,-0.18,-0.03,0.16\r\n20020829,0.0076,-0.56,-0.3,0\r\n20020830,0.0076,-0.02,-0.01,-0.02\r\n20020902,0.0081,0.15,0.1,-0.2\r\n20020903,0.0081,0.29,-0.01,-0.08\r\n20020904,0.0081,-0.56,0.03,-0.09\r\n20020905,0.0081,-1.3,-0.06,0\r\n20020906,0.0081,-0.56,-0.05,-0.05\r\n20020909,0.0081,-0.68,-0.02,0.33\r\n20020910,0.0081,0.18,0.19,0.15\r\n20020911,0.0081,-0.37,-0.06,-0.18\r\n20020912,0.0081,0.23,0.14,-0.23\r\n20020913,0.0081,-0.04,-0.13,0.2\r\n20020916,0.0081,-1.35,-0.19,0.11\r\n20020917,0.0081,-0.02,-0.26,0.22\r\n20020918,0.0081,-0.7,-0.01,-0.07\r\n20020919,0.0081,1.69,-0.04,-0.36\r\n20020920,0.0081,-0.98,-0.16,0.1\r\n20020923,0.0081,-0.08,-0.17,-0.1\r\n20020924,0.0081,-0.42,0.01,0.12\r\n20020925,0.0081,0.01,-0.04,0.29\r\n20020926,0.0081,-0.69,-0.05,0.29\r\n20020927,0.0081,-0.36,-0.17,0.24\r\n20021008,0.0093,-1.12,0.03,-0.6\r\n20021009,0.0093,-2.16,0.25,-0.17\r\n20021010,0.0093,-0.36,-0.11,-0.03\r\n20021011,0.0093,-0.35,-0.19,-0.19\r\n20021014,0.0093,0.46,-0.15,-0.3\r\n20021015,0.0093,-0.38,-0.04,0.21\r\n20021016,0.0093,0.86,-0.02,-0.08\r\n20021017,0.0093,-1.58,-0.25,-0.06\r\n20021018,0.0093,0.39,-0.24,0.14\r\n20021021,0.0093,0.32,-0.11,-0.25\r\n20021022,0.0093,1.46,0.28,0.16\r\n20021023,0.0093,0.06,0.17,-0.11\r\n20021024,0.0093,-0.88,0,0.21\r\n20021025,0.0093,-0.95,-0.3,0.57\r\n20021028,0.0093,-1.15,-0.18,0.41\r\n20021029,0.0093,0.49,0.08,0.02\r\n20021030,0.0093,-0.04,-0.22,0.44\r\n20021031,0.0093,-0.11,-0.03,0.04\r\n20021101,0.0077,0.24,-0.14,0.63\r\n20021104,0.0077,1.38,0.12,-0.46\r\n20021105,0.0077,1.56,0.3,-0.11\r\n20021106,0.0077,-0.44,0.09,-0.05\r\n20021107,0.0077,0.06,-0.32,0.42\r\n20021108,0.0077,-2.14,-0.77,0.57\r\n20021111,0.0077,-1.16,0.03,0.12\r\n20021112,0.0077,-1.4,-1.34,1.31\r\n20021113,0.0077,0.38,-0.27,-0.04\r\n20021114,0.0077,-1.99,-0.49,0.62\r\n20021115,0.0077,0.2,0.14,-0.11\r\n20021118,0.0077,-0.97,-0.03,0.08\r\n20021119,0.0077,0.68,0.38,-0.45\r\n20021120,0.0077,-2.44,-0.72,0.67\r\n20021121,0.0077,-2.66,-0.52,0.79\r\n20021122,0.0077,0.27,0.14,-0.05\r\n20021125,0.0077,0.29,0.08,-0.27\r\n20021126,0.0077,-1.92,-0.54,0.31\r\n20021127,0.0077,0.69,-0.2,-0.23\r\n20021128,0.0077,2.73,0.4,-0.66\r\n20021129,0.0077,1.26,0.36,-0.65\r\n20021202,0.0076,-2.75,-0.92,0.3\r\n20021203,0.0076,0.9,0.39,-0.51\r\n20021204,0.0076,0.32,0.44,0.32\r\n20021205,0.0076,-0.67,0.09,0.03\r\n20021206,0.0076,0.02,-0.04,0.06\r\n20021209,0.0076,-0.33,-0.12,0.05\r\n20021210,0.0076,-1.21,-0.47,0.26\r\n20021211,0.0076,-0.41,-0.16,-0.24\r\n20021212,0.0076,0.4,0.29,-0.54\r\n20021213,0.0076,0.88,0.77,-0.19\r\n20021216,0.0076,0.86,0.35,0.04\r\n20021217,0.0076,0.6,0.35,0\r\n20021218,0.0076,-0.38,0.04,0.45\r\n20021219,0.0076,0.19,0.16,-0.21\r\n20021220,0.0076,1.03,0.14,-0.35\r\n20021223,0.0076,0.15,-0.09,0.01\r\n20021224,0.0076,0.02,0.18,-0.03\r\n20021225,0.0076,-0.64,-0.34,0.02\r\n20021226,0.0076,-2.67,-0.77,0.66\r\n20021227,0.0076,-0.03,0.07,0.02\r\n20021230,0.0076,-1.1,-0.22,0.12\r\n20021231,0.0076,-0.56,-0.25,0.36\r\n20030102,0.0084,-2.63,0.22,-0.86\r\n20030103,0.0084,-0.02,0.21,-0.52\r\n20030106,0.0084,1.13,0.55,-0.22\r\n20030107,0.0084,-0.28,0.31,0.4\r\n20030108,0.0084,2.73,0.22,0.47\r\n20030109,0.0084,1.79,0.19,-0.18\r\n20030110,0.0084,-0.91,-0.11,0.5\r\n20030113,0.0084,0.2,0.33,-0.18\r\n20030114,0.0084,5.36,-0.71,1.37\r\n20030115,0.0084,-0.76,0.08,-0.69\r\n20030116,0.0084,1.6,0.59,0.02\r\n20030117,0.0084,-0.35,-0.15,0.02\r\n20030120,0.0084,0.31,0.11,-0.08\r\n20030121,0.0084,-1.83,-0.23,-0.17\r\n20030122,0.0084,0.38,0.35,-0.16\r\n20030123,0.0084,-0.72,-0.4,0.38\r\n20030124,0.0084,1.78,-0.3,0.8\r\n20030127,0.0084,1.2,0.21,0.33\r\n20030128,0.0084,0.23,-0.02,-0.03\r\n20030129,0.0084,0.02,-0.22,0.52\r\n20030210,0.0101,-1.19,-0.19,-0.08\r\n20030211,0.0101,0.84,0.39,-0.25\r\n20030212,0.0101,1.13,0.5,-0.25\r\n20030213,0.0101,-0.44,0.16,-0.13\r\n20030214,0.0101,0.41,0.13,0.09\r\n20030217,0.0101,-0.9,0.32,-0.4\r\n20030218,0.0101,-0.02,0.05,0.01\r\n20030219,0.0101,0.92,-0.03,0.15\r\n20030220,0.0101,-0.1,-0.1,0.09\r\n20030221,0.0101,-1.99,-0.3,0.09\r\n20030224,0.0101,0.17,-0.08,0.06\r\n20030225,0.0101,1.9,0.37,-0.11\r\n20030226,0.0101,0.04,0.02,0.31\r\n20030227,0.0101,0.04,-0.14,0.41\r\n20030228,0.0101,-0.11,-0.09,0.11\r\n20030303,0.008,0.82,-0.07,0.12\r\n20030304,0.008,-0.15,0.22,0.04\r\n20030305,0.008,-0.42,-0.13,0.31\r\n20030306,0.008,-1.22,-0.28,0.19\r\n20030307,0.008,-0.28,-0.07,0.04\r\n20030310,0.008,-1.69,-0.25,0.25\r\n20030311,0.008,-0.03,0.09,0.02\r\n20030312,0.008,0.45,0.03,0.34\r\n20030313,0.008,-0.66,-0.05,0.03\r\n20030314,0.008,0.14,-0.07,0.15\r\n20030317,0.008,0.2,-0.38,0.75\r\n20030318,0.008,-0.69,-0.22,0.01\r\n20030319,0.008,0.59,-0.49,0.25\r\n20030320,0.008,-0.45,0.07,-0.01\r\n20030321,0.008,0.72,0.35,-0.13\r\n20030324,0.008,0.18,-0.08,0.05\r\n20030325,0.008,-0.97,-0.33,0.22\r\n20030326,0.008,-0.5,-0.18,0.44\r\n20030327,0.008,2.71,0.04,0.12\r\n20030328,0.008,-0.35,-0.13,0.29\r\n20030331,0.008,1.29,-0.46,0.72\r\n20030401,0.0074,0.67,0.04,-0.27\r\n20030402,0.0074,-0.08,-0.01,0.43\r\n20030403,0.0074,-0.6,0.2,-0.25\r\n20030404,0.0074,0.56,-0.29,0.5\r\n20030407,0.0074,-0.43,-0.89,0.33\r\n20030408,0.0074,0.25,-0.92,0.74\r\n20030409,0.0074,1.46,-1.08,1.34\r\n20030410,0.0074,0.66,-0.08,0.24\r\n20030411,0.0074,1.65,-0.42,0.91\r\n20030414,0.0074,2.43,0.66,0.45\r\n20030415,0.0074,0.88,0.19,0.49\r\n20030416,0.0074,-1.41,-0.72,0.2\r\n20030417,0.0074,0.4,0.17,-0.1\r\n20030418,0.0074,-0.58,0.94,-0.46\r\n20030421,0.0074,-1.89,0.6,-0.52\r\n20030422,0.0074,-0.04,-0.15,0.09\r\n20030423,0.0074,-1.56,0.05,-0.81\r\n20030424,0.0074,-2.62,-0.1,-0.15\r\n20030425,0.0074,-1.17,-0.37,0.56\r\n20030428,0.0074,3.3,-1.76,2.45\r\n20030429,0.0074,-1.69,-0.46,-0.28\r\n20030430,0.0074,0.7,-1.2,1.32\r\n20030512,0.0112,0.85,-1.65,2.38\r\n20030513,0.0112,-2.85,-1.32,-0.32\r\n20030514,0.0112,2.49,-0.03,0.48\r\n20030515,0.0112,2.22,0.59,-0.57\r\n20030516,0.0112,-0.03,0.05,0.51\r\n20030519,0.0112,0.1,0.78,-0.51\r\n20030520,0.0112,-1.16,-0.09,0.03\r\n20030521,0.0112,-0.16,0.32,-0.28\r\n20030522,0.0112,0.83,-0.32,0.36\r\n20030523,0.0112,1.06,-0.1,0.35\r\n20030526,0.0112,0.55,-0.14,-0.13\r\n20030527,0.0112,-0.09,0.43,-0.67\r\n20030528,0.0112,-1.13,0.69,-0.79\r\n20030529,0.0112,1.11,-0.11,-0.2\r\n20030530,0.0112,0.4,0.49,-0.18\r\n20030602,0.0077,-0.03,0.38,0\r\n20030603,0.0077,-1,0.04,0.26\r\n20030604,0.0077,0.75,-0.99,0.89\r\n20030605,0.0077,-1.2,0.5,-0.11\r\n20030606,0.0077,-1.22,0.02,0.14\r\n20030609,0.0077,-0.62,-0.22,0.68\r\n20030610,0.0077,0.56,0.17,0.03\r\n20030611,0.0077,1.7,-0.01,0.34\r\n20030612,0.0077,-0.3,0.07,-0.1\r\n20030613,0.0077,0.25,0.36,0.18\r\n20030616,0.0077,-0.45,0.07,-0.52\r\n20030617,0.0077,-0.2,-0.09,-0.05\r\n20030618,0.0077,-0.11,-0.15,0.11\r\n20030619,0.0077,-1.06,-0.06,-0.14\r\n20030620,0.0077,-0.53,0.26,0.18\r\n20030623,0.0077,-1.08,-0.12,0.49\r\n20030624,0.0077,0.08,0.23,0.06\r\n20030625,0.0077,-0.73,-0.01,-0.07\r\n20030626,0.0077,0.85,-0.49,0.3\r\n20030627,0.0077,-1.29,-0.2,0.13\r\n20030630,0.0077,-0.69,-0.2,0.37\r\n20030701,0.0073,-0.07,-0.26,0.21\r\n20030702,0.0073,1.02,-0.33,0.88\r\n20030703,0.0073,0.27,0.04,-0.11\r\n20030704,0.0073,-0.11,-0.18,0.3\r\n20030707,0.0073,0.05,-0.23,0.6\r\n20030708,0.0073,0.69,0.42,-0.41\r\n20030709,0.0073,-0.61,0.06,-0.23\r\n20030710,0.0073,1.88,-0.35,0.8\r\n20030711,0.0073,-0.25,0.02,0.05\r\n20030714,0.0073,-0.44,-0.09,0.24\r\n20030715,0.0073,0.28,-0.32,0.53\r\n20030716,0.0073,0.28,-0.14,-0.05\r\n20030717,0.0073,0.6,-0.42,0.73\r\n20030718,0.0073,-2.15,0.02,-0.17\r\n20030721,0.0073,-0.77,-0.47,0.67\r\n20030722,0.0073,0.22,0.27,-0.68\r\n20030723,0.0073,0.27,0.21,-0.15\r\n20030724,0.0073,-0.64,-0.37,0.07\r\n20030725,0.0073,-1.12,-0.14,-0.71\r\n20030728,0.0073,-0.43,-0.36,0.07\r\n20030729,0.0073,0.46,0.12,0.04\r\n20030730,0.0073,0.14,-0.34,0.33\r\n20030731,0.0073,-0.12,-0.17,-0.34\r\n20030801,0.008,0.07,-0.02,-0.52\r\n20030804,0.008,1.17,0.24,-0.06\r\n20030805,0.008,-0.02,0.2,0.17\r\n20030806,0.008,-0.47,0.32,-0.5\r\n20030807,0.008,0.03,-0.2,0.11\r\n20030808,0.008,-1.18,0.15,-0.18\r\n20030811,0.008,0.26,0.07,-0.04\r\n20030812,0.008,0.34,0.18,-0.11\r\n20030813,0.008,-1.03,0.19,0.14\r\n20030814,0.008,-0.7,0.11,-0.49\r\n20030815,0.008,-0.29,0.02,-0.28\r\n20030818,0.008,-0.33,-0.03,-0.13\r\n20030819,0.008,0.3,0.03,-0.05\r\n20030820,0.008,-0.55,0.08,-0.13\r\n20030821,0.008,-0.08,0,0.09\r\n20030822,0.008,-0.16,0.15,0.01\r\n20030825,0.008,-1.13,-0.3,0.26\r\n20030826,0.008,0.18,0.12,0.15\r\n20030827,0.008,0,0.18,-0.19\r\n20030828,0.008,-0.72,-0.09,0.48\r\n20030829,0.008,0.48,-0.56,0.36\r\n20030901,0.0074,1.82,0.35,0.1\r\n20030902,0.0074,-0.25,0.17,-0.44\r\n20030903,0.0074,0.09,0.45,-0.58\r\n20030904,0.0074,-0.07,-0.01,-0.07\r\n20030905,0.0074,-1.12,-0.05,-0.31\r\n20030908,0.0074,-0.45,0.15,-0.19\r\n20030909,0.0074,-0.43,0.1,-0.08\r\n20030910,0.0074,0.23,-0.18,0.39\r\n20030911,0.0074,0.08,-0.12,0.38\r\n20030912,0.0074,-1.14,-0.38,0.08\r\n20030915,0.0074,-0.97,-0.23,0.05\r\n20030916,0.0074,0.65,0.47,-0.29\r\n20030917,0.0074,-1.12,-0.19,0.05\r\n20030918,0.0074,-0.02,0.05,-0.18\r\n20030919,0.0074,0.08,0.02,-0.18\r\n20030922,0.0074,-0.65,0.43,-0.31\r\n20030923,0.0074,0.58,0.25,0.11\r\n20030924,0.0074,0.26,0.31,0.14\r\n20030925,0.0074,-1.26,-0.34,0.36\r\n20030926,0.0074,-0.44,-0.25,0.4\r\n20030929,0.0074,-1.24,-0.38,0.67\r\n20030930,0.0074,0.92,-0.26,0.39\r\n20031008,0.0093,0.37,-0.1,0.17\r\n20031009,0.0093,-0.18,0.06,0.19\r\n20031010,0.0093,2.44,-0.09,0.41\r\n20031013,0.0093,-0.3,-0.14,-0.16\r\n20031014,0.0093,-0.88,-0.06,0.05\r\n20031015,0.0093,-0.31,0.14,0.02\r\n20031016,0.0093,-1.29,0.06,0.47\r\n20031017,0.0093,0.34,-0.11,0.43\r\n20031020,0.0093,-0.74,-0.17,0.6\r\n20031021,0.0093,0.38,-0.27,0.39\r\n20031022,0.0093,2.43,-0.3,0.84\r\n20031023,0.0093,-0.78,-0.11,0.16\r\n20031024,0.0093,-0.34,-0.32,1.42\r\n20031027,0.0093,-0.9,-0.81,1.25\r\n20031028,0.0093,-0.07,-0.64,1.91\r\n20031029,0.0093,-0.77,-1.28,1.48\r\n20031030,0.0093,-1.24,-0.3,0.5\r\n20031031,0.0093,0.24,0.55,-0.44\r\n20031103,0.0081,1.14,-0.39,1.42\r\n20031104,0.0081,1.95,-0.3,0.83\r\n20031105,0.0081,-0.43,-0.57,1.67\r\n20031106,0.0081,-1.94,-1.3,1.85\r\n20031107,0.0081,-1.54,0.15,-1.1\r\n20031110,0.0081,0.4,-0.33,1.42\r\n20031111,0.0081,0.24,0.5,-0.32\r\n20031112,0.0081,-1.85,0.13,-0.22\r\n20031113,0.0081,0.13,0.13,-0.97\r\n20031114,0.0081,0.78,0.35,0.02\r\n20031117,0.0081,-0.29,0.19,-0.06\r\n20031118,0.0081,-1.41,0.2,-0.26\r\n20031119,0.0081,1.59,0.51,-0.76\r\n20031120,0.0081,2.77,1.4,-1.41\r\n20031121,0.0081,-1.08,-0.25,0.77\r\n20031124,0.0081,2.95,0.27,-0.38\r\n20031125,0.0081,0.36,0.48,-1.47\r\n20031126,0.0081,0.28,0.07,0.21\r\n20031127,0.0081,-1.68,-0.36,0.58\r\n20031128,0.0081,0.57,-0.01,0.54\r\n20031201,0.0073,2.41,0.01,0.33\r\n20031202,0.0073,0.1,0.01,-0.07\r\n20031203,0.0073,0.55,-0.12,0.98\r\n20031204,0.0073,0.44,0.14,0.2\r\n20031205,0.0073,-0.13,0.56,-0.15\r\n20031208,0.0073,-0.92,-0.23,0.36\r\n20031209,0.0073,0.46,-0.19,0.06\r\n20031210,0.0073,1.59,-0.25,0.56\r\n20031211,0.0073,0.06,0.25,0.23\r\n20031212,0.0073,-0.07,-0.05,0.03\r\n20031215,0.0073,0.03,-0.13,0.15\r\n20031216,0.0073,-0.38,-0.66,0.4\r\n20031217,0.0073,-0.74,-0.05,0.04\r\n20031218,0.0073,0.29,-1.48,2.01\r\n20031219,0.0073,-0.74,-2.19,3.26\r\n20031222,0.0073,3.03,-0.46,0.5\r\n20031223,0.0073,0.5,0.05,0.02\r\n20031224,0.0073,0.81,-0.43,0.71\r\n20031225,0.0073,-0.1,0.03,-0.25\r\n20031226,0.0073,0.11,0.28,-0.08\r\n20031229,0.0073,-1.12,0.21,-0.23\r\n20031230,0.0073,0.88,-0.72,0.92\r\n20031231,0.0073,-0.98,0.19,0.02\r\n20040102,0.0129,1.08,-0.28,1.02\r\n20040105,0.0129,3.05,-2.14,3.08\r\n20040106,0.0129,0.36,-0.47,0.91\r\n20040107,0.0129,0.84,0.15,0.4\r\n20040108,0.0129,1.29,0.72,-0.27\r\n20040109,0.0129,-1.59,0.62,-1.66\r\n20040112,0.0129,2.16,-0.72,1.88\r\n20040113,0.0129,-0.21,1.48,-1.24\r\n20040114,0.0129,-1.15,1.26,-0.8\r\n20040115,0.0129,-0.32,-0.09,0.27\r\n20040116,0.0129,0.81,0.51,0\r\n20040129,0.0129,1.9,1.39,-1.3\r\n20040130,0.0129,-2.29,1.22,-1.91\r\n20040202,0.0078,2.04,0.33,-0.81\r\n20040203,0.0078,1.42,0.25,0.25\r\n20040204,0.0078,2.04,-0.01,-0.38\r\n20040205,0.0078,0.5,0.88,-1.39\r\n20040206,0.0078,-0.77,-1.07,0.29\r\n20040209,0.0078,1.6,0.34,-0.38\r\n20040210,0.0078,-0.46,0.74,-1.28\r\n20040211,0.0078,-0.7,0.79,-1.09\r\n20040212,0.0078,-0.74,1.57,-0.88\r\n20040213,0.0078,-0.43,-0.94,1.25\r\n20040216,0.0078,2.29,0.11,-0.03\r\n20040217,0.0078,0.6,0.79,-0.77\r\n20040218,0.0078,0.89,-0.19,0.31\r\n20040219,0.0078,-0.13,0.69,-0.18\r\n20040220,0.0078,0.32,0.2,-0.27\r\n20040223,0.0078,-2.25,0.44,-0.61\r\n20040224,0.0078,-0.01,-0.95,1.07\r\n20040225,0.0078,-2.03,-0.09,0.23\r\n20040226,0.0078,0.57,-1.26,1.97\r\n20040227,0.0078,1.26,0.46,-0.47\r\n20040301,0.0073,1.13,0.6,-0.33\r\n20040302,0.0073,-0.35,0.39,-0.11\r\n20040303,0.0073,-1.49,-0.15,0.57\r\n20040304,0.0073,0.52,0.43,-0.19\r\n20040305,0.0073,-0.51,0.19,0.42\r\n20040308,0.0073,-1.69,-0.72,1.35\r\n20040309,0.0073,-0.15,-0.89,0.72\r\n20040310,0.0073,2.36,-0.1,-0.09\r\n20040311,0.0073,0.96,0.18,-0.23\r\n20040312,0.0073,0.17,-0.4,0.39\r\n20040315,0.0073,2.39,-0.54,0.32\r\n20040316,0.0073,0.5,0.18,0.08\r\n20040317,0.0073,0,0.39,-0.2\r\n20040318,0.0073,-1.71,0.18,0.21\r\n20040319,0.0073,1.76,-0.06,0.16\r\n20040322,0.0073,0.43,0.39,-0.17\r\n20040323,0.0073,-0.8,0.68,-0.41\r\n20040324,0.0073,-0.09,0.16,0.29\r\n20040325,0.0073,-0.11,0.1,0.14\r\n20040326,0.0073,-0.05,0.06,-0.05\r\n20040329,0.0073,-0.9,0.52,0.26\r\n20040330,0.0073,0.24,0.21,-0.01\r\n20040331,0.0073,1.06,0.46,-0.28\r\n20040401,0.0074,0.84,-0.35,0.19\r\n20040402,0.0074,0.55,0.37,0.49\r\n20040405,0.0074,-0.11,1.54,0.21\r\n20040406,0.0074,0.64,0.41,0.2\r\n20040407,0.0074,-0.09,0.04,0.18\r\n20040408,0.0074,-0.33,0.11,0.06\r\n20040409,0.0074,-2.43,-0.09,-0.14\r\n20040412,0.0074,-0.22,1.01,-0.23\r\n20040413,0.0074,-0.51,0.28,-0.05\r\n20040414,0.0074,-1.38,-0.24,0.49\r\n20040415,0.0074,-1.04,-0.55,0.5\r\n20040416,0.0074,0.89,-0.98,1.21\r\n20040419,0.0074,-1.09,0.25,0.48\r\n20040420,0.0074,-0.66,0.24,-0.1\r\n20040421,0.0074,-1.17,0.46,-0.79\r\n20040422,0.0074,0.46,0.08,-0.21\r\n20040423,0.0074,-1.33,-0.41,0.61\r\n20040426,0.0074,0.07,-0.53,0.86\r\n20040427,0.0074,-1.33,-0.34,-0.1\r\n20040428,0.0074,-0.41,0.49,-0.5\r\n20040429,0.0074,-1.87,-0.04,-0.12\r\n20040430,0.0074,0.82,-0.76,1.07\r\n20040510,0.0105,-2.33,0.52,-0.05\r\n20040511,0.0105,0.53,-0.23,-0.41\r\n20040512,0.0105,2.32,0.37,-0.25\r\n20040513,0.0105,-0.41,0.13,-0.07\r\n20040514,0.0105,-2.29,0.26,-0.28\r\n20040517,0.0105,-1.34,0.72,-1.21\r\n20040518,0.0105,1.03,0.5,-0.76\r\n20040519,0.0105,0.14,0.33,0.08\r\n20040520,0.0105,-0.7,-0.11,0.41\r\n20040521,0.0105,0.64,-0.21,0.6\r\n20040524,0.0105,-0.56,-0.04,0.67\r\n20040525,0.0105,-2.12,-0.91,0.36\r\n20040526,0.0105,0.42,-0.23,0.09\r\n20040527,0.0105,2.22,-0.02,-0.28\r\n20040528,0.0105,-0.54,0.07,0.06\r\n20040531,0.0105,0.44,0,-0.06\r\n20040601,0.0074,1.48,0.38,-0.11\r\n20040602,0.0074,-0.89,0.59,-0.83\r\n20040603,0.0074,-1.56,-0.27,0.26\r\n20040604,0.0074,-0.08,-0.05,-0.57\r\n20040607,0.0074,-1.56,-0.14,0.11\r\n20040608,0.0074,-1.51,0.11,-0.1\r\n20040609,0.0074,-1.88,-0.63,2.23\r\n20040610,0.0074,0.17,-0.51,0.86\r\n20040611,0.0074,0.22,0.03,-0.35\r\n20040614,0.0074,-2.29,-0.76,0.77\r\n20040615,0.0074,1.75,0.1,-0.7\r\n20040616,0.0074,0.8,-0.21,0.72\r\n20040617,0.0074,-2.09,0.14,-0.24\r\n20040618,0.0074,-1.17,0.48,-0.71\r\n20040621,0.0074,0.5,0.82,-0.75\r\n20040622,0.0074,1.19,-0.03,-0.03\r\n20040623,0.0074,-0.81,-0.3,-0.08\r\n20040624,0.0074,-0.98,0.3,0.16\r\n20040625,0.0074,-1.73,-0.68,1.01\r\n20040628,0.0074,-1.12,-0.53,1.66\r\n20040629,0.0074,1.61,-0.66,1.58\r\n20040630,0.0074,-0.61,-0.64,0.7\r\n20040701,0.0076,3.06,0.21,-0.41\r\n20040702,0.0076,0.04,0.07,0.43\r\n20040705,0.0076,0.2,-0.28,-0.15\r\n20040706,0.0076,0.4,0.32,-0.07\r\n20040707,0.0076,-0.82,-0.22,0.31\r\n20040708,0.0076,0.1,0,0.19\r\n20040709,0.0076,-0.57,-0.06,0.28\r\n20040712,0.0076,-2.23,-0.45,1\r\n20040713,0.0076,0.2,0.02,0.65\r\n20040714,0.0076,0.32,-1.12,1.49\r\n20040715,0.0076,1.37,-0.25,0.74\r\n20040716,0.0076,2.46,0.5,-0.75\r\n20040719,0.0076,0.13,0.16,0.29\r\n20040720,0.0076,-0.65,0.21,0.09\r\n20040721,0.0076,0.05,0.01,-0.04\r\n20040722,0.0076,-2.75,0.08,-0.83\r\n20040723,0.0076,0.05,0.29,-0.17\r\n20040726,0.0076,-1.35,-0.06,0.44\r\n20040727,0.0076,-0.3,-0.3,0.2\r\n20040728,0.0076,0.35,0.29,-0.06\r\n20040729,0.0076,1.33,0.12,-0.49\r\n20040730,0.0076,-1.54,-0.31,-0.01\r\n20040802,0.0076,-0.96,-0.09,-0.06\r\n20040803,0.0076,-0.65,0.73,-0.62\r\n20040804,0.0076,2.76,-0.11,0.32\r\n20040805,0.0076,-1.17,0.18,0.04\r\n20040806,0.0076,0.01,-0.06,1.13\r\n20040809,0.0076,0.41,-0.43,0.44\r\n20040810,0.0076,0.34,0.07,-0.19\r\n20040811,0.0076,-1.1,-0.28,0.18\r\n20040812,0.0076,-1.2,-0.56,0.37\r\n20040813,0.0076,0.07,0.2,0.46\r\n20040816,0.0076,-0.96,-0.41,1.4\r\n20040817,0.0076,0.4,-0.34,0.29\r\n20040818,0.0076,-0.15,-0.19,0.58\r\n20040819,0.0076,-1.33,-0.55,1.14\r\n20040820,0.0076,0.04,0.44,0.62\r\n20040823,0.0076,-1.17,0.69,-0.45\r\n20040824,0.0076,1.41,0.87,-0.62\r\n20040825,0.0076,-0.42,-0.21,-0.34\r\n20040826,0.0076,-0.41,-0.4,0.95\r\n20040827,0.0076,-1.06,0.02,0.41\r\n20040830,0.0076,-0.1,0.32,-0.19\r\n20040831,0.0076,1.73,0.31,-0.87\r\n20040901,0.0074,-1.51,-0.15,-0.2\r\n20040902,0.0074,0.5,1.07,-0.81\r\n20040903,0.0074,-0.12,0.19,-0.07\r\n20040906,0.0074,-0.4,0.35,-0.26\r\n20040907,0.0074,0.2,-0.2,-0.23\r\n20040908,0.0074,-1.07,0.44,-0.24\r\n20040909,0.0074,-1.96,-0.05,0.11\r\n20040910,0.0074,0.2,-0.23,-0.12\r\n20040913,0.0074,-2.19,-0.12,0.49\r\n20040914,0.0074,3.14,0.39,-0.41\r\n20040915,0.0074,4.23,0.59,-0.72\r\n20040916,0.0074,1.11,0.06,0.11\r\n20040917,0.0074,3.22,-0.01,0.22\r\n20040920,0.0074,3.52,0.43,-0.82\r\n20040921,0.0074,-0.76,0.54,0.19\r\n20040922,0.0074,-1.95,-0.24,0.47\r\n20040923,0.0074,3.2,-0.42,0.73\r\n20040924,0.0074,-1.89,-0.73,1.15\r\n20040927,0.0074,-0.52,-0.34,1.73\r\n20040928,0.0074,0.67,0.02,0.63\r\n20040929,0.0074,-1.08,-0.08,0.76\r\n20040930,0.0074,-1.64,-0.45,0.8\r\n20041008,0.0106,2.05,0.76,-0.45\r\n20041011,0.0106,-0.54,1.04,-0.38\r\n20041012,0.0106,-2.03,0.53,0.1\r\n20041013,0.0106,0.17,0.12,-0.25\r\n20041014,0.0106,-4.02,-1.29,1.21\r\n20041015,0.0106,-0.18,-0.39,0.48\r\n20041018,0.0106,0.35,0.36,-0.08\r\n20041019,0.0106,0.17,-0.24,-0.01\r\n20041020,0.0106,-0.41,-0.63,1.09\r\n20041021,0.0106,-1.35,-0.56,1.31\r\n20041022,0.0106,1.27,-0.76,1.95\r\n20041025,0.0106,-1.45,0.6,0.05\r\n20041026,0.0106,1.04,0.68,0.48\r\n20041027,0.0106,1.27,1.12,-0.74\r\n20041028,0.0106,-0.04,-0.89,-0.69\r\n20041029,0.0106,-1.68,-0.59,0.21\r\n20041101,0.0083,-1.12,0.2,-0.14\r\n20041102,0.0083,-0.4,0.26,-0.28\r\n20041103,0.0083,1.86,0.47,-0.41\r\n20041104,0.0083,-1.83,-0.21,0.54\r\n20041105,0.0083,0.01,0.22,-0.12\r\n20041108,0.0083,0.01,0.13,-0.04\r\n20041109,0.0083,0.26,0.2,-0.17\r\n20041110,0.0083,3.64,0.08,-0.63\r\n20041111,0.0083,-0.59,0.4,-0.73\r\n20041112,0.0083,0.33,-0.02,0.03\r\n20041115,0.0083,1.32,0.17,0.27\r\n20041116,0.0083,-0.04,-0.08,0.39\r\n20041117,0.0083,-0.99,0.03,-0.06\r\n20041118,0.0083,0.95,0.48,-0.18\r\n20041119,0.0083,0.77,0.29,-0.27\r\n20041122,0.0083,0.23,0.89,-1.03\r\n20041123,0.0083,-0.9,1.05,-1.15\r\n20041124,0.0083,-0.87,-0.95,1.03\r\n20041125,0.0083,-0.11,-0.57,0.59\r\n20041126,0.0083,0,0.54,-0.19\r\n20041129,0.0083,-1.42,-0.33,0.29\r\n20041130,0.0083,0.2,0.32,-0.6\r\n20041201,0.0082,-0.49,0.33,-0.11\r\n20041202,0.0082,-0.09,1.12,-1.33\r\n20041203,0.0082,0.25,0.68,-0.31\r\n20041206,0.0082,0.1,-0.1,-0.22\r\n20041207,0.0082,-1.26,-0.56,1.02\r\n20041208,0.0082,0.16,0.25,0.25\r\n20041209,0.0082,0.97,-0.62,0.78\r\n20041210,0.0082,-1.62,-1.13,1.69\r\n20041213,0.0082,-0.56,-0.18,0.82\r\n20041214,0.0082,-0.2,-0.12,0.49\r\n20041215,0.0082,0.42,-0.77,1.35\r\n20041216,0.0082,-0.63,-0.29,0.3\r\n20041217,0.0082,-1.11,0.35,-0.1\r\n20041220,0.0082,-1.2,0.53,-1\r\n20041221,0.0082,-0.04,0.43,-0.26\r\n20041222,0.0082,2.57,0.28,-0.86\r\n20041223,0.0082,-1.93,-0.35,0.48\r\n20041224,0.0082,0.14,0,-0.01\r\n20041227,0.0082,-0.34,-0.05,0.13\r\n20041228,0.0082,-0.17,-0.26,0.41\r\n20041229,0.0082,-0.4,0.02,0.36\r\n20041230,0.0082,-0.05,-0.42,0.58\r\n20041231,0.0082,-0.6,-0.61,1.49\r\n20050104,0.0095,-1.94,0.5,-0.8\r\n20050105,0.0095,0.82,0.93,-1.22\r\n20050106,0.0095,-1.11,-0.06,-0.36\r\n20050107,0.0095,0.32,0.14,-0.06\r\n20050110,0.0095,0.71,0.15,-0.18\r\n20050111,0.0095,0.3,-0.17,-0.06\r\n20050112,0.0095,-0.04,0.1,0.15\r\n20050113,0.0095,-0.02,0.38,-0.61\r\n20050114,0.0095,-0.92,-0.37,0.55\r\n20050117,0.0095,-2.34,0.03,-0.3\r\n20050118,0.0095,0.74,0.73,-1.23\r\n20050119,0.0095,-0.64,0.25,-0.09\r\n20050120,0.0095,-1.15,-0.14,-0.28\r\n20050121,0.0095,2.46,-0.62,1.52\r\n20050124,0.0095,1.72,0.29,-0.37\r\n20050125,0.0095,-0.07,-0.82,0.53\r\n20050126,0.0095,-0.92,-0.06,0.91\r\n20050127,0.0095,-1.4,-0.01,1.02\r\n20050128,0.0095,-0.96,-1.15,0.97\r\n20050131,0.0095,-1.77,-1.28,2.36\r\n20050201,0.0131,-0.16,-0.81,1.67\r\n20050202,0.0131,5.35,0.06,-0.27\r\n20050203,0.0131,-1.28,-0.72,1.07\r\n20050204,0.0131,2.19,-0.35,0.62\r\n20050216,0.0131,0.83,0.33,-0.69\r\n20050217,0.0131,-0.24,0.64,-1.16\r\n20050218,0.0131,-1.36,0.25,0.19\r\n20050221,0.0131,1.95,-0.03,0.49\r\n20050222,0.0131,2.03,0.59,-0.4\r\n20050223,0.0131,0.07,1.22,-0.39\r\n20050224,0.0131,0.06,0.53,0.02\r\n20050225,0.0131,0.12,-0.17,0.17\r\n20050228,0.0131,-0.4,0.16,0.03\r\n20050301,0.0082,-0.21,0.39,-0.27\r\n20050302,0.0082,-1.3,-0.15,0.08\r\n20050303,0.0082,0.54,0.02,-0.02\r\n20050304,0.0082,-0.48,-0.38,0.16\r\n20050307,0.0082,0.52,-0.34,0.36\r\n20050308,0.0082,1.94,0.26,0.09\r\n20050309,0.0082,-0.08,0.54,0.06\r\n20050310,0.0082,-2.37,-0.3,0.5\r\n20050311,0.0082,0.35,0.14,0.08\r\n20050314,0.0082,0.29,-0.6,1.42\r\n20050315,0.0082,-1.82,0.19,-0.74\r\n20050316,0.0082,-1.07,0.05,-0.48\r\n20050317,0.0082,-1.01,-0.41,0.22\r\n20050318,0.0082,-1.46,-0.54,1.06\r\n20050321,0.0082,0.28,-0.21,0.62\r\n20050322,0.0082,-1.91,-1.37,1.63\r\n20050323,0.0082,-0.39,0.52,-0.15\r\n20050324,0.0082,0.52,0.64,-0.62\r\n20050325,0.0082,-0.16,0.03,0.18\r\n20050328,0.0082,-0.42,-0.69,0.51\r\n20050329,0.0082,-0.45,-0.08,0.08\r\n20050330,0.0082,-1.95,-1.23,0.52\r\n20050331,0.0082,0.75,0.05,-0.07\r\n20050401,0.0087,3.62,-0.13,0.5\r\n20050404,0.0087,-1.62,-0.29,0.43\r\n20050405,0.0087,-0.81,-0.18,0.08\r\n20050406,0.0087,1.88,-0.29,0.78\r\n20050407,0.0087,0.93,0.69,-1.05\r\n20050408,0.0087,1.89,0.15,0.17\r\n20050411,0.0087,-0.68,0.65,-0.89\r\n20050412,0.0087,-1.77,-0.69,0.49\r\n20050413,0.0087,2.32,0.27,0.04\r\n20050414,0.0087,-1.16,-0.22,1.1\r\n20050415,0.0087,-1.49,-1.08,0.94\r\n20050418,0.0087,-1.4,-0.26,0.32\r\n20050419,0.0087,0.12,-0.16,-0.29\r\n20050420,0.0087,-1.46,-0.78,1.37\r\n20050421,0.0087,-0.97,-1.09,1.85\r\n20050422,0.0087,-0.5,-1,1.06\r\n20050425,0.0087,-1.08,-1.1,1.22\r\n20050426,0.0087,0.74,1.02,-1.25\r\n20050427,0.0087,-1.38,-0.47,0.7\r\n20050428,0.0087,1.8,0.83,-0.76\r\n20050429,0.0087,-0.99,-0.51,0.61\r\n20050509,0.0111,-2.44,-1.35,0.81\r\n20050510,0.0111,0.35,-0.3,0.72\r\n20050511,0.0111,-1.05,0.69,-0.94\r\n20050512,0.0111,-1.7,1.54,-0.81\r\n20050513,0.0111,0.31,0.7,-0.64\r\n20050516,0.0111,-1.1,0.74,-0.38\r\n20050517,0.0111,0.44,0.74,-0.47\r\n20050518,0.0111,0.26,0.69,-0.67\r\n20050519,0.0111,0.04,0.16,0.09\r\n20050520,0.0111,-0.37,-0.58,0.46\r\n20050523,0.0111,-2.5,-1.07,0.15\r\n20050524,0.0111,0.53,1.68,-1.81\r\n20050525,0.0111,-0.14,0.61,-0.82\r\n20050526,0.0111,-1.33,0.53,-0.45\r\n20050527,0.0111,-0.74,0.17,-0.27\r\n20050530,0.0111,0.76,0.58,-0.43\r\n20050531,0.0111,0.01,0,0.43\r\n20050601,0.0083,-2.11,-0.04,-0.76\r\n20050602,0.0083,-2.09,1.44,-1.18\r\n20050603,0.0083,-0.31,-0.07,0.21\r\n20050606,0.0083,2.3,-0.98,0.7\r\n20050607,0.0083,-0.27,0.32,-1.02\r\n20050608,0.0083,8.23,-0.58,0.99\r\n20050609,0.0083,1.22,-0.23,-0.01\r\n20050610,0.0083,-2.17,0.07,-0.35\r\n20050613,0.0083,0.09,0.52,0.32\r\n20050614,0.0083,-1.28,-0.76,0.58\r\n20050615,0.0083,-2.08,-0.78,0.63\r\n20050616,0.0083,1.29,-0.18,0.79\r\n20050617,0.0083,-0.07,-0.38,0.57\r\n20050620,0.0083,2.94,-0.05,0.35\r\n20050621,0.0083,-1.24,0.47,0.02\r\n20050622,0.0083,0.21,0.11,0.02\r\n20050623,0.0083,-0.75,-0.31,0.75\r\n20050624,0.0083,0.75,-0.21,0.66\r\n20050627,0.0083,2.11,0.64,-0.25\r\n20050628,0.0083,-1.45,-0.1,0.66\r\n20050629,0.0083,-0.44,-0.05,0.49\r\n20050630,0.0083,-2.34,-0.36,0.8\r\n20050701,0.009,-2.65,-1.74,1.82\r\n20050704,0.009,-0.69,-1.08,1.35\r\n20050705,0.009,-0.76,0.03,0.08\r\n20050706,0.009,-0.45,0.53,0.6\r\n20050707,0.009,0.52,0.32,0.43\r\n20050708,0.009,-2.16,-1.5,1.43\r\n20050711,0.009,-0.7,-0.52,0.78\r\n20050712,0.009,3.5,-0.03,0.14\r\n20050713,0.009,-0.85,0.01,0.09\r\n20050714,0.009,0.25,-0.29,0.22\r\n20050715,0.009,-1.34,-1.34,1.5\r\n20050718,0.009,-1.45,-1.58,0.9\r\n20050719,0.009,0.3,0.74,-0.42\r\n20050720,0.009,0.66,0.05,0.08\r\n20050721,0.009,-0.15,-0.26,-0.27\r\n20050722,0.009,2.63,-0.17,-0.47\r\n20050725,0.009,-0.2,-0.01,0.25\r\n20050726,0.009,2.73,-0.32,-0.07\r\n20050727,0.009,1.88,0,-0.71\r\n20050728,0.009,-0.3,0.35,0.11\r\n20050729,0.009,-0.43,0.2,0.27\r\n20050801,0.0082,0.53,0.17,0.11\r\n20050802,0.0082,1.58,0.35,-0.31\r\n20050803,0.0082,0.48,0.5,-1.02\r\n20050804,0.0082,-0.61,-0.06,0.2\r\n20050805,0.0082,2.66,0.35,0.17\r\n20050808,0.0082,1.02,0.57,-0.09\r\n20050809,0.0082,1.25,0.9,-0.51\r\n20050810,0.0082,1.08,0.11,-0.26\r\n20050811,0.0082,1.77,-0.47,0.4\r\n20050812,0.0082,-1.64,-0.5,-0.87\r\n20050815,0.0082,1.92,1.18,0.38\r\n20050816,0.0082,-0.83,0.92,-0.9\r\n20050817,0.0082,1.38,0.98,-0.59\r\n20050818,0.0082,-3.8,-0.69,-0.38\r\n20050819,0.0082,0.18,-0.34,0.73\r\n20050822,0.0082,0.84,0.43,0.08\r\n20050823,0.0082,-0.91,-0.89,0.93\r\n20050824,0.0082,1.42,0.03,-0.12\r\n20050825,0.0082,0.49,0.96,-1.49\r\n20050826,0.0082,0.06,0.76,-0.35\r\n20050829,0.0082,-1.43,0.89,-0.64\r\n20050830,0.0082,-0.74,-0.23,0.52\r\n20050831,0.0082,1.54,0.86,0.05\r\n20050901,0.0083,2,0.61,-0.34\r\n20050902,0.0083,0.39,0.27,-0.24\r\n20050905,0.0083,0.68,0.66,-1.24\r\n20050906,0.0083,-1.91,-0.83,0.7\r\n20050907,0.0083,2,1.22,0.02\r\n20050908,0.0083,0.11,0.01,0.1\r\n20050909,0.0083,-0.55,-0.41,0.75\r\n20050912,0.0083,-0.11,0.48,-0.39\r\n20050913,0.0083,1.73,0.61,-0.63\r\n20050914,0.0083,0.87,0.36,-0.08\r\n20050915,0.0083,-0.04,0.51,-0.34\r\n20050916,0.0083,-0.33,0.38,-0.21\r\n20050919,0.0083,0.52,1.02,-0.81\r\n20050920,0.0083,-0.87,0.24,-0.42\r\n20050921,0.0083,-1.89,0.23,-0.26\r\n20050922,0.0083,-2.5,-1.34,2.08\r\n20050923,0.0083,-0.8,-0.49,1.42\r\n20050926,0.0083,0.34,0.32,0.46\r\n20050927,0.0083,-1.9,-0.22,1.21\r\n20050928,0.0083,-0.13,-0.25,0.33\r\n20050929,0.0083,2.08,0.29,-0.09\r\n20050930,0.0083,-0.01,-0.28,0.64\r\n20051010,0.0118,-1.47,-0.06,0.74\r\n20051011,0.0118,1.71,0.77,-0.68\r\n20051012,0.0118,0.43,0.85,0.07\r\n20051013,0.0118,-0.88,0.42,0.13\r\n20051014,0.0118,-1.21,0.08,0.21\r\n20051017,0.0118,-0.74,0.86,-0.45\r\n20051018,0.0118,0.92,0.94,-0.36\r\n20051019,0.0118,-0.62,-0.39,-0.58\r\n20051020,0.0118,0.11,-0.19,0.93\r\n20051021,0.0118,0.6,0.53,-0.32\r\n20051024,0.0118,0.18,0.24,-0.17\r\n20051025,0.0118,-1.77,-0.79,0.23\r\n20051026,0.0118,-2.39,-0.58,0.9\r\n20051027,0.0118,0.04,-0.72,1.62\r\n20051028,0.0118,-1.74,-2,-0.07\r\n20051031,0.0118,1.04,-0.68,-0.32\r\n20051101,0.0083,-0.26,-0.48,-0.18\r\n20051102,0.0083,1.47,0.96,-0.99\r\n20051103,0.0083,-0.82,-0.16,-0.15\r\n20051104,0.0083,0.42,0.49,-0.02\r\n20051107,0.0083,0.06,1.01,-0.68\r\n20051108,0.0083,0.83,0.24,0.01\r\n20051109,0.0083,-0.24,-0.56,0.65\r\n20051110,0.0083,-1.97,-1.31,0.66\r\n20051111,0.0083,0.24,0.5,-0.4\r\n20051114,0.0083,-0.2,0.87,-0.42\r\n20051115,0.0083,-0.14,-0.13,0.01\r\n20051116,0.0083,0.81,0.06,-0.23\r\n20051117,0.0083,-0.03,-0.08,0.09\r\n20051118,0.0083,2.14,0.1,0.01\r\n20051121,0.0083,0.24,0.32,-0.06\r\n20051122,0.0083,-2,-0.27,0.33\r\n20051123,0.0083,0.59,0.1,-0.12\r\n20051124,0.0083,0.6,-0.35,-0.16\r\n20051125,0.0083,0.07,-0.02,0.04\r\n20051128,0.0083,-0.3,-0.03,-0.03\r\n20051129,0.0083,-1.32,-0.21,0.45\r\n20051130,0.0083,0.13,0.08,0.53\r\n20051201,0.0086,-0.03,-0.18,0.19\r\n20051202,0.0086,-0.46,-1.02,0.87\r\n20051205,0.0086,-1.6,-1.24,0.76\r\n20051206,0.0086,0.84,-0.21,0.44\r\n20051207,0.0086,1.18,-0.05,-0.02\r\n20051208,0.0086,0.01,-0.32,0.2\r\n20051209,0.0086,1.46,0.27,0.32\r\n20051212,0.0086,0.23,0.21,-0.6\r\n20051213,0.0086,0.2,-0.38,-0.04\r\n20051214,0.0086,0.82,0.22,-0.23\r\n20051215,0.0086,-0.2,-0.28,0.37\r\n20051216,0.0086,0.53,0.29,-0.12\r\n20051219,0.0086,0.3,-0.12,0.35\r\n20051220,0.0086,0.4,-0.3,-0.25\r\n20051221,0.0086,-0.61,-0.39,0.26\r\n20051222,0.0086,0.46,0.1,0.35\r\n20051223,0.0086,0.9,0.3,0.11\r\n20051226,0.0086,1.06,-0.06,0.55\r\n20051227,0.0086,-0.28,0.07,-0.22\r\n20051228,0.0086,0.26,0.12,0.25\r\n20051229,0.0086,1.29,-0.28,0.65\r\n20051230,0.0086,-0.81,-0.3,0.02\r\n20060104,0.0118,1.65,-0.11,0.28\r\n20060105,0.0118,1.54,-0.37,-0.16\r\n20060106,0.0118,0.87,-0.25,-0.01\r\n20060109,0.0118,0.54,0.74,-0.49\r\n20060110,0.0118,0.3,0.17,-0.17\r\n20060111,0.0118,-0.86,-0.23,-0.45\r\n20060112,0.0118,1.46,0.77,-0.55\r\n20060113,0.0118,-0.4,0.73,-0.62\r\n20060116,0.0118,-1.56,-0.08,-0.36\r\n20060117,0.0118,0.5,-0.56,-0.07\r\n20060118,0.0118,2.17,-0.03,0.13\r\n20060119,0.0118,1.49,-0.08,0.78\r\n20060120,0.0118,0.41,-0.16,0.4\r\n20060123,0.0118,0.02,0.43,0.04\r\n20060124,0.0118,-0.28,-0.31,-0.18\r\n20060125,0.0118,0.51,-1.29,1.05\r\n20060206,0.0101,2.7,0.07,0.48\r\n20060207,0.0101,-0.59,0.79,-1.31\r\n20060208,0.0101,0.57,0.12,-0.53\r\n20060209,0.0101,-1.66,0.15,-0.1\r\n20060210,0.0101,1.29,-0.21,-0.55\r\n20060213,0.0101,-0.18,0.01,-0.39\r\n20060214,0.0101,0.62,0.17,-0.06\r\n20060215,0.0101,0.89,-0.7,0.46\r\n20060216,0.0101,-2.4,-0.87,-0.73\r\n20060217,0.0101,-0.24,-0.24,0.8\r\n20060220,0.0101,0.07,-0.41,0.38\r\n20060221,0.0101,1.86,-0.27,-0.19\r\n20060222,0.0101,-0.3,0.14,-0.63\r\n20060223,0.0101,0.32,-0.2,-0.26\r\n20060224,0.0101,0.63,-0.03,0.34\r\n20060227,0.0101,-0.07,0.01,0.35\r\n20060228,0.0101,0.16,-0.18,-0.6\r\n20060301,0.0082,0.86,0.39,0.29\r\n20060302,0.0082,-1.97,-0.57,0.22\r\n20060303,0.0082,0.71,-0.71,-0.03\r\n20060306,0.0082,-0.36,0.05,-0.27\r\n20060307,0.0082,-2.81,-0.26,-0.04\r\n20060308,0.0082,-0.95,0.66,-0.34\r\n20060309,0.0082,-0.19,-0.22,0.34\r\n20060310,0.0082,-0.01,-0.22,-0.54\r\n20060313,0.0082,1.33,-0.35,0.46\r\n20060314,0.0082,-0.01,0.1,-0.17\r\n20060315,0.0082,1.52,-0.44,0.74\r\n20060316,0.0082,-0.08,0.44,0.1\r\n20060317,0.0082,-0.4,-0.02,-0.5\r\n20060320,0.0082,1.72,-0.3,0.52\r\n20060321,0.0082,0.14,-0.26,0.06\r\n20060322,0.0082,0.6,0.51,0.23\r\n20060323,0.0082,0.58,0.23,-0.16\r\n20060324,0.0082,-0.65,-0.4,0.28\r\n20060327,0.0082,0.26,0.65,-0.35\r\n20060328,0.0082,0.32,0.42,0.32\r\n20060329,0.0082,0.63,-0.35,0.53\r\n20060330,0.0082,-0.97,-0.25,0.05\r\n20060331,0.0082,0.5,-0.27,0\r\n20060403,0.0092,2.08,0.26,1.05\r\n20060404,0.0092,0.88,0.42,-0.27\r\n20060405,0.0092,0.91,0.03,0.45\r\n20060406,0.0092,0.02,-0.21,-0.17\r\n20060407,0.0092,0.31,0.81,-1.08\r\n20060410,0.0092,1.44,0.84,-0.39\r\n20060411,0.0092,0.25,0.04,0.11\r\n20060412,0.0092,-0.37,0.73,-0.11\r\n20060413,0.0092,-2.34,-0.58,0.01\r\n20060414,0.0092,2.43,-0.12,-0.18\r\n20060417,0.0092,1.36,0.15,1.15\r\n20060418,0.0092,0.28,-0.28,0.15\r\n20060419,0.0092,0.81,-1.08,0.59\r\n20060420,0.0092,-0.83,-0.22,-0.06\r\n20060421,0.0092,2.19,-0.83,1.77\r\n20060424,0.0092,-0.97,-0.58,0.99\r\n20060425,0.0092,-0.64,0.04,-0.29\r\n20060426,0.0092,1.5,0.06,0.74\r\n20060427,0.0092,-0.07,0.35,-0.85\r\n20060428,0.0092,1.61,-0.76,0.74\r\n20060508,0.0105,4.7,-1.06,1.43\r\n20060509,0.0105,2.79,0.37,-0.39\r\n20060510,0.0105,1.04,0.33,-0.41\r\n20060511,0.0105,-0.93,-0.23,-1.54\r\n20060512,0.0105,4.67,-0.25,0.83\r\n20060515,0.0105,4.59,0.64,1\r\n20060516,0.0105,-3.61,0.06,-1.16\r\n20060517,0.0105,0.65,2.25,-0.62\r\n20060518,0.0105,-0.25,0.35,-1.65\r\n20060519,0.0105,3.18,-0.03,1.26\r\n20060522,0.0105,-0.09,3.07,-1.93\r\n20060523,0.0105,-3.73,-1.11,-0.83\r\n20060524,0.0105,-0.94,-1.38,-0.04\r\n20060525,0.0105,0.16,1.9,-0.62\r\n20060526,0.0105,1.58,0.63,0.11\r\n20060529,0.0105,2.17,1.04,0.3\r\n20060530,0.0105,0.4,0.5,-0.96\r\n20060531,0.0105,-0.95,0.64,-1.23\r\n20060601,0.0083,2.95,1.47,-1.4\r\n20060602,0.0083,-1.06,1.64,-0.5\r\n20060605,0.0083,1.03,1.57,-0.41\r\n20060606,0.0083,-0.43,-0.18,0.03\r\n20060607,0.0083,-5.98,-1.98,0.46\r\n20060608,0.0083,0.18,0.06,1.67\r\n20060609,0.0083,-2.53,0.05,-0.45\r\n20060612,0.0083,0.22,0.02,0.48\r\n20060613,0.0083,-0.22,0.67,-0.15\r\n20060614,0.0083,-1.33,1.16,-0.29\r\n20060615,0.0083,0.27,0.26,0.34\r\n20060616,0.0083,3,-0.57,0.46\r\n20060619,0.0083,1.1,-0.37,0.11\r\n20060620,0.0083,0.3,0.92,-0.09\r\n20060621,0.0083,0.34,-0.67,0.23\r\n20060622,0.0083,-0.24,0.44,0.11\r\n20060623,0.0083,0.66,0.86,0.31\r\n20060626,0.0083,2.06,0.55,0.89\r\n20060627,0.0083,0.38,-0.03,0.16\r\n20060628,0.0083,0,0.06,0.36\r\n20060629,0.0083,2.22,-0.09,0.84\r\n20060630,0.0083,0.02,-0.48,0.4\r\n20060703,0.009,1.69,0.25,-0.17\r\n20060704,0.009,-1.07,0.26,-0.4\r\n20060705,0.009,-1.48,0.58,-0.81\r\n20060706,0.009,2.1,0.62,-0.31\r\n20060707,0.009,-0.63,0.8,-0.5\r\n20060710,0.009,0.21,0.67,-0.03\r\n20060711,0.009,0.49,0.16,0.07\r\n20060712,0.009,-0.05,-0.32,0.01\r\n20060713,0.009,-5.55,-1.4,0.78\r\n20060714,0.009,0.77,0.15,0.22\r\n20060717,0.009,1.39,0.65,0.25\r\n20060718,0.009,0.22,0.85,-0.41\r\n20060719,0.009,-2.56,0.42,0.16\r\n20060720,0.009,0.72,0.55,-0.39\r\n20060721,0.009,0.78,-0.27,-0.18\r\n20060724,0.009,0.29,0.18,0.66\r\n20060725,0.009,1.36,0.12,0.11\r\n20060726,0.009,0.11,0.83,-0.34\r\n20060727,0.009,-1.21,-0.03,-0.27\r\n20060728,0.009,-0.91,0.02,0.01\r\n20060731,0.009,-3.45,-0.67,0.21\r\n20060801,0.0086,-0.88,-0.23,0.39\r\n20060802,0.0086,-0.36,0.08,0.65\r\n20060803,0.0086,-0.04,0.4,0.79\r\n20060804,0.0086,-2.34,-1.21,0.35\r\n20060807,0.0086,-1.62,-1.29,0.56\r\n20060808,0.0086,2.68,0.79,-0.74\r\n20060809,0.0086,0.02,0.12,0.22\r\n20060810,0.0086,1.99,-0.12,0.15\r\n20060811,0.0086,0.09,0.25,-0.67\r\n20060814,0.0086,-2.54,-0.37,0.56\r\n20060815,0.0086,1.84,0.29,0.15\r\n20060816,0.0086,1.4,0.44,-0.07\r\n20060817,0.0086,-0.77,-0.11,-0.17\r\n20060818,0.0086,-0.36,-0.46,-0.01\r\n20060821,0.0086,0.27,0.44,-0.12\r\n20060822,0.0086,1.03,-0.11,0.34\r\n20060823,0.0086,-0.04,-0.49,0.05\r\n20060824,0.0086,0.58,0.33,-0.37\r\n20060825,0.0086,0.17,-0.15,-0.39\r\n20060828,0.0086,2.02,0.26,-0.78\r\n20060829,0.0086,0.13,-0.23,-0.56\r\n20060830,0.0086,0.35,0.22,-0.72\r\n20060831,0.0086,0.43,0.42,0.17\r\n20060901,0.0097,-1.71,-0.32,0.44\r\n20060904,0.0097,1.72,0.39,-0.56\r\n20060905,0.0097,0.29,0.13,-0.26\r\n20060906,0.0097,0.53,0.02,0.27\r\n20060907,0.0097,-1.53,-0.18,0.47\r\n20060908,0.0097,0.21,-0.14,0.05\r\n20060911,0.0097,0.5,0,-0.34\r\n20060912,0.0097,0.48,0.01,0.19\r\n20060913,0.0097,-0.84,-0.45,0.75\r\n20060914,0.0097,0.02,-0.12,-0.73\r\n20060915,0.0097,1.66,-0.36,0.63\r\n20060918,0.0097,1.1,0.39,-0.52\r\n20060919,0.0097,0.25,-0.06,0\r\n20060920,0.0097,0.08,-0.1,-0.47\r\n20060921,0.0097,0.66,-0.27,0.3\r\n20060922,0.0097,-0.99,-0.31,-0.53\r\n20060925,0.0097,-0.11,0.22,-0.82\r\n20060926,0.0097,-0.81,0.58,-0.44\r\n20060927,0.0097,0.96,0.06,0.12\r\n20060928,0.0097,1.05,-0.36,0.6\r\n20060929,0.0097,1.1,-0.15,-0.12\r\n20061009,0.0124,2.38,-0.15,0.58\r\n20061010,0.0124,0.14,0.55,-0.11\r\n20061011,0.0124,0.01,0.57,0.27\r\n20061012,0.0124,-0.77,-0.14,1.24\r\n20061013,0.0124,0.4,-0.09,-0.28\r\n20061016,0.0124,-0.78,0.68,0.06\r\n20061017,0.0124,-0.3,0.22,0.51\r\n20061018,0.0124,1.54,-0.21,0.03\r\n20061019,0.0124,0.25,0.22,0.15\r\n20061020,0.0124,-0.23,-0.71,0.4\r\n20061023,0.0124,-2.61,-0.9,0.9\r\n20061024,0.0124,2.24,0.59,0.55\r\n20061025,0.0124,0.1,-0.75,0.82\r\n20061026,0.0124,0.18,-1.03,0.65\r\n20061027,0.0124,-1.31,-0.57,0.91\r\n20061030,0.0124,0.38,-0.5,1.52\r\n20061031,0.0124,1.13,-0.12,-0.11\r\n20061101,0.0093,0.94,-0.42,0.91\r\n20061102,0.0093,-0.19,-0.06,0.27\r\n20061103,0.0093,0.52,-0.1,-0.02\r\n20061106,0.0093,1.2,-1.18,2.2\r\n20061107,0.0093,-0.04,-1.29,0.66\r\n20061108,0.0093,-1.12,0.46,-1.21\r\n20061109,0.0093,1.78,-0.6,0.22\r\n20061110,0.0093,-1.55,-1.4,0.07\r\n20061113,0.0093,-2.09,-1.34,0.21\r\n20061114,0.0093,0.93,0.35,-0.55\r\n20061115,0.0093,2.48,-0.95,0.49\r\n20061116,0.0093,-0.71,-1.19,-0.12\r\n20061117,0.0093,2.25,-1.09,0.38\r\n20061120,0.0093,2.01,-2.3,1.38\r\n20061121,0.0093,1.37,-0.19,1.4\r\n20061122,0.0093,0.57,0.71,-0.46\r\n20061123,0.0093,0.58,0.06,0.11\r\n20061124,0.0093,0.01,0.91,-1.03\r\n20061127,0.0093,1.02,0.43,-0.87\r\n20061128,0.0093,-0.07,0.27,-0.84\r\n20061129,0.0093,1.09,-0.73,-0.5\r\n20061130,0.0093,2.85,-1.17,0.9\r\n20061201,0.01,0.48,-0.39,0.04\r\n20061204,0.01,2.41,-0.38,0.63\r\n20061205,0.01,0.72,-0.27,-1.56\r\n20061206,0.01,-1.13,-0.7,-0.87\r\n20061207,0.01,-0.3,0.05,1.57\r\n20061208,0.01,-3.94,0.14,0.2\r\n20061211,0.01,4.15,-1.36,0.39\r\n20061212,0.01,0.52,-0.3,0.36\r\n20061213,0.01,0.65,0.18,1.13\r\n20061214,0.01,1.63,-0.05,-0.02\r\n20061215,0.01,1.43,-0.54,-0.02\r\n20061218,0.01,2.17,-0.75,0.07\r\n20061219,0.01,0.33,0.27,0.1\r\n20061220,0.01,0.64,1.15,-0.47\r\n20061221,0.01,-1.55,0.79,-0.52\r\n20061222,0.01,-0.73,0.55,-0.4\r\n20061225,0.01,1.9,-1.17,0.89\r\n20061226,0.01,0.51,-1.06,2.2\r\n20061227,0.01,1.84,-1.3,0.59\r\n20061228,0.01,-0.76,-1.6,0.46\r\n20061229,0.01,2.39,-1.27,0.94\r\n20070104,0.0106,2.28,-1.93,3.03\r\n20070105,0.0106,-0.73,3.64,1.65\r\n20070108,0.0106,2.62,1.63,-0.72\r\n20070109,0.0106,3.65,-1.71,1.01\r\n20070110,0.0106,1.49,0.76,-1.18\r\n20070111,0.0106,-1.68,2.09,-1.72\r\n20070112,0.0106,-3.43,-0.51,-1.27\r\n20070115,0.0106,5.03,0.54,-0.11\r\n20070116,0.0106,1.85,0.53,-1.63\r\n20070117,0.0106,-1.57,1.29,1.28\r\n20070118,0.0106,0.04,2.75,-0.74\r\n20070119,0.0106,3.19,0.42,-0.02\r\n20070122,0.0106,3.7,1.12,0.06\r\n20070123,0.0106,1.11,-0.5,0.68\r\n20070124,0.0106,1.27,-0.37,0.49\r\n20070125,0.0106,-3.91,-1.04,-0.52\r\n20070126,0.0106,1.71,-1.65,-0.11\r\n20070129,0.0106,2.56,1.22,-0.15\r\n20070130,0.0106,0.1,0.48,1.87\r\n20070131,0.0106,-5.7,1.32,-0.46\r\n20070201,0.0127,-0.05,0.71,-0.62\r\n20070202,0.0127,-3.79,2.34,-1.28\r\n20070205,0.0127,-1.51,2.8,-1.17\r\n20070206,0.0127,1.86,0.95,0.03\r\n20070207,0.0127,2.03,-0.77,-0.27\r\n20070208,0.0127,1.36,0.52,0.32\r\n20070209,0.0127,-0.69,1,-0.42\r\n20070212,0.0127,3.32,-1.21,0.23\r\n20070213,0.0127,1.48,0.31,0.01\r\n20070214,0.0127,2.73,-0.5,0.34\r\n20070215,0.0127,3.17,-0.29,-0.9\r\n20070216,0.0127,0.47,0.99,-0.03\r\n20070226,0.0127,2.15,2.49,0.57\r\n20070227,0.0127,-9.1,0.44,-1.32\r\n20070228,0.0127,3.69,1.07,-0.5\r\n20070301,0.01,-2.91,0.33,-1.46\r\n20070302,0.01,1.46,0.46,0.18\r\n20070305,0.01,-1.16,1.23,-0.1\r\n20070306,0.01,1.44,-1.09,1.28\r\n20070307,0.01,2.6,-0.32,-0.05\r\n20070308,0.01,1.5,0.72,-0.33\r\n20070309,0.01,-0.09,0.39,0.43\r\n20070312,0.01,0.72,1.94,-0.71\r\n20070313,0.01,0.69,0.6,-0.93\r\n20070314,0.01,-1.63,0.78,-1.51\r\n20070315,0.01,1.83,0.46,0.2\r\n20070316,0.01,-1.39,-0.44,0.21\r\n20070319,0.01,2.32,-0.56,0.98\r\n20070320,0.01,0.98,1.67,-0.22\r\n20070321,0.01,1.23,0.21,-0.7\r\n20070322,0.01,0.49,-0.14,-0.04\r\n20070323,0.01,0.44,1.02,0.05\r\n20070326,0.01,1.79,0.99,-0.55\r\n20070327,0.01,0.68,0.79,-0.87\r\n20070328,0.01,0.13,-1.22,1.22\r\n20070329,0.01,-0.46,-2.95,2.29\r\n20070330,0.01,0.06,0.95,-0.37\r\n20070402,0.0107,2.38,0.08,0.36\r\n20070403,0.0107,1.4,-0.07,-0.06\r\n20070404,0.0107,0.48,0.47,-0.39\r\n20070405,0.0107,1.15,0.51,-0.55\r\n20070406,0.0107,0.72,0.93,-0.06\r\n20070409,0.0107,1.94,-0.09,0.16\r\n20070410,0.0107,1.56,-0.58,0.44\r\n20070411,0.0107,1.63,0.29,0.96\r\n20070412,0.0107,1.53,0.99,0\r\n20070413,0.0107,-0.14,-0.83,-0.68\r\n20070416,0.0107,2.46,0.48,-0.82\r\n20070417,0.0107,1.01,1.88,-1.62\r\n20070418,0.0107,0.7,2.16,-0.62\r\n20070419,0.0107,-4.96,-0.73,-0.51\r\n20070420,0.0107,4.26,0.57,-0.6\r\n20070423,0.0107,4.22,0.31,0.41\r\n20070424,0.0107,0.28,0.61,-0.38\r\n20070425,0.0107,0.23,-0.49,-0.06\r\n20070426,0.0107,1.12,0.08,-0.24\r\n20070427,0.0107,-1,0.64,-1.66\r\n20070430,0.0107,1.44,-0.61,-1.2\r\n20070508,0.0135,3.55,0.65,-1.17\r\n20070509,0.0135,1.44,-0.87,-0.08\r\n20070510,0.0135,1.14,0.25,0.35\r\n20070511,0.0135,-0.76,0.41,-0.33\r\n20070514,0.0135,0.68,0.28,0.45\r\n20070515,0.0135,-3.65,0.57,0.24\r\n20070516,0.0135,2.3,0.25,-0.19\r\n20070517,0.0135,1.78,0.74,-0.85\r\n20070518,0.0135,-0.31,0.16,-1.48\r\n20070521,0.0135,1.27,1.11,-1.18\r\n20070522,0.0135,1.1,0.8,-0.88\r\n20070523,0.0135,1.83,0.76,-0.87\r\n20070524,0.0135,-0.47,-0.05,-0.56\r\n20070525,0.0135,0.89,0.04,-0.76\r\n20070528,0.0135,2.22,0.1,0.69\r\n20070529,0.0135,1.96,-0.63,0.53\r\n20070530,0.0135,-6.44,-1.86,0.22\r\n20070531,0.0135,0.94,-3.71,5.73\r\n20070601,0.0119,-3.43,-2.06,4.05\r\n20070604,0.0119,-8.49,0.17,0.02\r\n20070605,0.0119,2.87,-1.48,5.22\r\n20070606,0.0119,0.79,0.5,0.52\r\n20070607,0.0119,3.37,0.77,-1.59\r\n20070608,0.0119,0.68,0.69,-1.23\r\n20070611,0.0119,2.23,-0.17,0.83\r\n20070612,0.0119,2.33,0.54,2.07\r\n20070613,0.0119,2.6,0.87,-1.76\r\n20070614,0.0119,-1.69,0.09,-0.26\r\n20070615,0.0119,0.23,-0.06,0.01\r\n20070618,0.0119,2.55,0.4,-0.4\r\n20070619,0.0119,0.6,0.85,-1.27\r\n20070620,0.0119,-2.47,-0.52,-0.61\r\n20070621,0.0119,0.7,-0.95,0.98\r\n20070622,0.0119,-3.87,-1.73,1.37\r\n20070625,0.0119,-4.59,-1.59,1.47\r\n20070626,0.0119,1.32,0.86,2.24\r\n20070627,0.0119,2.58,-0.21,0.2\r\n20070628,0.0119,-4.42,-1.82,0.89\r\n20070629,0.0119,-2.48,-1.24,2.56\r\n20070702,0.0121,0.68,1.89,1.07\r\n20070703,0.0121,1.91,-0.17,0.53\r\n20070704,0.0121,-2.05,-0.61,0\r\n20070705,0.0121,-5.55,-0.68,-0.03\r\n20070706,0.0121,4.64,1.37,0.11\r\n20070709,0.0121,2.76,0.32,-0.59\r\n20070710,0.0121,-1.35,-1.49,0.63\r\n20070711,0.0121,0.65,0.88,-0.26\r\n20070712,0.0121,0.81,0.2,0.33\r\n20070713,0.0121,-0.11,-0.26,-0.05\r\n20070716,0.0121,-2.98,-0.11,0.29\r\n20070717,0.0121,2.13,-0.43,-0.48\r\n20070718,0.0121,0.33,-0.28,-0.16\r\n20070719,0.0121,-0.01,0.52,-0.89\r\n20070720,0.0121,3.7,-0.51,-0.36\r\n20070723,0.0121,4.56,0.86,0.35\r\n20070724,0.0121,-0.02,-0.21,-0.77\r\n20070725,0.0121,2.46,1.04,0.46\r\n20070726,0.0121,0.86,0.74,-1.3\r\n20070727,0.0121,0.31,1.36,-1.62\r\n20070730,0.0121,2.27,-0.15,0.41\r\n20070731,0.0121,0.84,0.39,-0.36\r\n20070801,0.0125,-3.87,-2.01,0.56\r\n20070802,0.0125,2.83,0.61,-0.94\r\n20070803,0.0125,3.17,-1.74,0.58\r\n20070806,0.0125,1.6,0.66,0.35\r\n20070807,0.0125,0.44,-1.08,-0.08\r\n20070808,0.0125,-0.07,-0.59,1.42\r\n20070809,0.0125,1.84,-1.08,-0.33\r\n20070810,0.0125,-0.86,-0.84,0.57\r\n20070813,0.0125,1.21,-0.33,2.02\r\n20070814,0.0125,1.22,1.25,-0.58\r\n20070815,0.0125,-0.17,-0.69,-0.35\r\n20070816,0.0125,-1.94,2.31,-1.16\r\n20070817,0.0125,-2.21,1.27,-0.39\r\n20070820,0.0125,5.52,-1.13,1.94\r\n20070821,0.0125,1,-0.66,-0.67\r\n20070822,0.0125,0.65,0.11,-0.58\r\n20070823,0.0125,1.25,-0.27,-0.75\r\n20070824,0.0125,1.09,-0.74,0.27\r\n20070827,0.0125,0.33,-0.56,0.8\r\n20070828,0.0125,0.42,-0.29,0.51\r\n20070829,0.0125,-1.63,0.9,-0.64\r\n20070830,0.0125,1.57,1.83,-0.37\r\n20070831,0.0125,1.11,-0.17,-0.46\r\n20070903,0.0151,2.24,0.33,0.33\r\n20070904,0.0151,-0.68,-0.16,-0.01\r\n20070905,0.0151,0.11,0.68,0.37\r\n20070906,0.0151,1.37,-0.11,0.46\r\n20070907,0.0151,-2.28,0.12,-0.04\r\n20070910,0.0151,1.32,1,-0.12\r\n20070911,0.0151,-4.88,-1.42,0.13\r\n20070912,0.0151,1.15,-0.22,0.38\r\n20070913,0.0151,2.34,-0.9,-0.05\r\n20070914,0.0151,1.03,0.6,-0.4\r\n20070917,0.0151,2.01,0.71,0.47\r\n20070918,0.0151,-0.2,0.93,0.16\r\n20070919,0.0151,-0.4,-0.89,-0.12\r\n20070920,0.0151,1.36,-0.1,0.17\r\n20070921,0.0151,-0.22,-0.73,0.53\r\n20070924,0.0151,0.6,-0.2,0.42\r\n20070925,0.0151,-1.06,-0.21,-0.95\r\n20070926,0.0151,-1.7,0.37,0.5\r\n20070927,0.0151,1.42,-0.28,0.99\r\n20070928,0.0151,2.75,-0.68,0.2\r\n20071008,0.0179,2.22,-1.97,1.79\r\n20071009,0.0179,0.35,0.81,0.35\r\n20071010,0.0179,0.4,-1.08,0.71\r\n20071011,0.0179,1.77,-3,1.73\r\n20071012,0.0179,-0.32,-2.41,1.18\r\n20071015,0.0179,1.97,-0.93,2.86\r\n20071016,0.0179,1.2,0.61,0.21\r\n20071017,0.0179,-0.85,0.9,-0.21\r\n20071018,0.0179,-3.43,1.62,0.17\r\n20071019,0.0179,-0.22,-0.33,0.15\r\n20071022,0.0179,-2.63,-0.15,-0.17\r\n20071023,0.0179,1.64,-1.42,1.48\r\n20071024,0.0179,1.02,-3.09,1.03\r\n20071025,0.0179,-5.11,-2.58,0.25\r\n20071026,0.0179,0.63,-1.15,1.67\r\n20071029,0.0179,2.87,-0.86,1.72\r\n20071030,0.0179,2.29,0.24,-0.01\r\n20071031,0.0179,1.31,2.07,-1.5\r\n20071101,0.0142,-0.8,-2.11,0.12\r\n20071102,0.0142,-2.45,0.72,-0.7\r\n20071105,0.0142,-2.21,3.89,0.21\r\n20071106,0.0142,-1.32,2.19,0.55\r\n20071107,0.0142,0.97,-1.27,0.64\r\n20071108,0.0142,-4.8,0.91,-0.32\r\n20071109,0.0142,-0.43,-0.09,0.81\r\n20071112,0.0142,-1.97,0.5,0.26\r\n20071113,0.0142,-0.59,1.34,-0.35\r\n20071114,0.0142,4.44,-0.64,0.79\r\n20071115,0.0142,-0.85,-0.35,0.31\r\n20071116,0.0142,-1.01,1.47,-0.36\r\n20071119,0.0142,-0.18,3.01,-0.01\r\n20071120,0.0142,1.22,0.14,0.39\r\n20071121,0.0142,-1.52,0.26,-0.2\r\n20071122,0.0142,-4.37,0.27,0.8\r\n20071123,0.0142,1.8,0.39,0.69\r\n20071126,0.0142,-1.18,0.5,-0.09\r\n20071127,0.0142,-1.74,0.95,0.49\r\n20071128,0.0142,-1.6,1.32,-0.3\r\n20071129,0.0142,4.57,-2.1,0.74\r\n20071130,0.0142,-1.89,0.69,-0.36\r\n20071203,0.0165,0.95,0,-0.64\r\n20071204,0.0165,1.25,-0.72,0.32\r\n20071205,0.0165,2.68,-0.81,0.92\r\n20071206,0.0165,0.05,0.29,-0.57\r\n20071207,0.0165,1.33,0.55,0.2\r\n20071210,0.0165,1.78,0.91,-0.18\r\n20071211,0.0165,-0.08,1.25,-0.45\r\n20071212,0.0165,-1.13,1.8,0.05\r\n20071213,0.0165,-3.14,0.47,0.39\r\n20071214,0.0165,1.55,0.37,-0.79\r\n20071217,0.0165,-2.49,3.19,-0.23\r\n20071218,0.0165,-0.55,-0.16,0.27\r\n20071219,0.0165,2.45,-0.23,0.49\r\n20071220,0.0165,1.67,-0.52,0.06\r\n20071221,0.0165,1.42,0.1,0.31\r\n20071224,0.0165,1.87,-0.17,-0.05\r\n20071225,0.0165,-0.07,1.25,0.09\r\n20071226,0.0165,0.63,1.22,-0.53\r\n20071227,0.0165,1.96,-1.19,-0.07\r\n20071228,0.0165,-0.67,0.23,-0.45\r\n20080102,0.0157,0.78,2.37,-1.01\r\n20080103,0.0157,0.74,1.5,-0.33\r\n20080104,0.0157,0.72,-0.59,-0.19\r\n20080107,0.0157,0.83,1,-0.25\r\n20080108,0.0157,-0.17,-2.43,0.87\r\n20080109,0.0157,1.43,1.57,-0.06\r\n20080110,0.0157,0.58,-0.56,-0.43\r\n20080111,0.0157,0.54,-0.46,1.01\r\n20080114,0.0157,0.72,0.72,-0.13\r\n20080115,0.0157,-0.72,1.94,-0.3\r\n20080116,0.0157,-2.84,1.83,-0.02\r\n20080117,0.0157,-2.71,0.33,0.47\r\n20080118,0.0157,0.63,-0.01,-0.52\r\n20080121,0.0157,-4.94,1.15,-0.08\r\n20080122,0.0157,-7.67,-0.47,0.36\r\n20080123,0.0157,3.94,0.49,-0.14\r\n20080124,0.0157,0.57,2.23,-1.26\r\n20080125,0.0157,0.94,-0.88,0.57\r\n20080128,0.0157,-7.3,0.46,-0.05\r\n20080129,0.0157,0.74,0.71,-0.41\r\n20080130,0.0157,-1.1,1.17,-0.79\r\n20080131,0.0157,-1.79,-1.79,0.58\r\n20080201,0.0202,-1.3,-3.04,1.6\r\n20080204,0.0202,8.26,-0.03,0.35\r\n20080205,0.0202,-0.27,0.97,-0.31\r\n20080213,0.0202,-2.1,1.67,-0.68\r\n20080214,0.0202,1.5,0.77,0.27\r\n20080215,0.0202,-1.11,0.76,-0.1\r\n20080218,0.0202,1.91,1.11,-0.32\r\n20080219,0.0202,1.98,0.01,0.44\r\n20080220,0.0202,-1.91,0.96,-0.22\r\n20080221,0.0202,-0.83,2.23,-0.36\r\n20080222,0.0202,-3.36,0.96,-0.52\r\n20080225,0.0202,-3.71,0.41,-0.3\r\n20080226,0.0202,0.63,0.45,1.23\r\n20080227,0.0202,2.57,-0.45,-0.24\r\n20080228,0.0202,-0.56,1.18,0.25\r\n20080229,0.0202,1.17,0.29,-0.28\r\n20080303,0.0164,2.3,0.75,-1.04\r\n20080304,0.0164,-2.21,1.2,-1.27\r\n20080305,0.0164,-0.89,1.4,-0.94\r\n20080306,0.0164,1.2,-0.52,0.78\r\n20080307,0.0164,-1.6,-0.57,-0.2\r\n20080310,0.0164,-3.9,0.71,-0.42\r\n20080311,0.0164,0.13,1.3,-0.74\r\n20080312,0.0164,-2.73,0.26,-0.07\r\n20080313,0.0164,-3.16,-0.75,0.54\r\n20080314,0.0164,-1.1,0.28,-0.3\r\n20080317,0.0164,-5.12,-2.52,0.68\r\n20080318,0.0164,-5.3,-1.56,1.28\r\n20080319,0.0164,3.24,1.05,-0.63\r\n20080320,0.0164,2.65,1.24,-0.28\r\n20080321,0.0164,0.93,0.21,-0.23\r\n20080324,0.0164,-4.09,2.01,-1.14\r\n20080325,0.0164,1.25,0.35,0.09\r\n20080326,0.0164,-0.07,-0.32,0.45\r\n20080327,0.0164,-4.51,-1.08,0.68\r\n20080328,0.0164,4.63,-1.51,1.15\r\n20080331,0.0164,-2.82,-1.18,1.41\r\n20080401,0.0159,-5.5,-2.25,1.67\r\n20080402,0.0159,-0.47,-4.57,3.11\r\n20080403,0.0159,3.08,-1,0.52\r\n20080407,0.0159,4.93,0.85,-1.27\r\n20080408,0.0159,0.82,1.01,-1.27\r\n20080409,0.0159,-5.45,0.41,1.11\r\n20080410,0.0159,1.97,0.62,-0.34\r\n20080411,0.0159,0.57,0.72,-0.66\r\n20080414,0.0159,-6.47,-0.14,0.18\r\n20080415,0.0159,1.65,1.24,-0.47\r\n20080416,0.0159,-2.36,-0.07,0.62\r\n20080417,0.0159,-2.57,-0.94,0.72\r\n20080418,0.0159,-3.94,-1.4,0.74\r\n20080421,0.0159,0.41,-0.29,1.65\r\n20080422,0.0159,0.95,-2.13,1.6\r\n20080423,0.0159,4.67,-0.3,0.37\r\n20080424,0.0159,9.3,-0.2,0.33\r\n20080425,0.0159,0.1,-0.08,-1.64\r\n20080428,0.0159,-1.83,1.41,-0.52\r\n20080429,0.0159,1.34,-0.43,0.39\r\n20080430,0.0159,4.82,-1.04,0.47\r\n20080505,0.0172,2.04,1.11,-1.13\r\n20080506,0.0172,-0.8,-0.02,-1.32\r\n20080507,0.0172,-4.26,-0.1,-0.1\r\n20080508,0.0172,2.24,1.2,0.16\r\n20080509,0.0172,-1.07,1.77,0.8\r\n20080512,0.0172,0.58,2.05,0.59\r\n20080513,0.0172,-1.49,2.16,0.69\r\n20080514,0.0172,2.7,-0.42,-0.2\r\n20080515,0.0172,-0.57,-0.82,-1.4\r\n20080516,0.0172,-0.44,-0.7,0.37\r\n20080519,0.0172,-0.58,0.88,0.56\r\n20080520,0.0172,-4.74,-1.61,-0.25\r\n20080521,0.0172,2.9,-1.32,-1.59\r\n20080522,0.0172,-1.78,0.61,-0.8\r\n20080523,0.0172,-0.49,-0.89,-0.65\r\n20080526,0.0172,-3.09,-0.05,-0.71\r\n20080527,0.0172,0.35,0.62,1.38\r\n20080528,0.0172,2.54,-0.07,-0.39\r\n20080529,0.0172,-1.63,-1.32,-1.32\r\n20080530,0.0172,0.91,-0.81,-0.52\r\n20080602,0.0167,0.57,0.08,-0.55\r\n20080603,0.0167,-0.65,0.21,0.17\r\n20080604,0.0167,-1.74,-0.29,-0.59\r\n20080605,0.0167,-0.6,-0.53,-0.43\r\n20080606,0.0167,-0.63,0.18,0.77\r\n20080610,0.0167,-7.84,-1.27,-0.14\r\n20080611,0.0167,-1.79,-1.41,0.43\r\n20080612,0.0167,-2.07,1.46,0.36\r\n20080613,0.0167,-3.19,-1.08,0.84\r\n20080616,0.0167,0.02,-2.4,-0.3\r\n20080617,0.0167,-3.07,-2.59,0.71\r\n20080618,0.0167,5.59,-0.82,-0.98\r\n20080619,0.0167,-6.76,-2.43,-0.87\r\n20080620,0.0167,2.95,-1.2,0.78\r\n20080623,0.0167,-2.5,1.21,1.89\r\n20080624,0.0167,1.69,2.11,1.32\r\n20080625,0.0167,3.92,1.67,-0.34\r\n20080626,0.0167,-0.01,1.56,-0.01\r\n20080627,0.0167,-5.48,-0.92,1.58\r\n20080630,0.0167,-0.41,0.91,0.39\r\n20080701,0.015,-3.1,1.26,0.29\r\n20080702,0.015,0.14,1.56,-0.45\r\n20080703,0.015,2.2,2.25,-0.07\r\n20080704,0.015,-1.13,1.12,0.3\r\n20080707,0.015,4.78,0.83,0.69\r\n20080708,0.015,0.86,0.87,-0.63\r\n20080709,0.015,3.79,-0.74,0.56\r\n20080710,0.015,-1.62,-0.22,0.17\r\n20080711,0.015,-0.83,-0.06,-0.01\r\n20080714,0.015,0.78,1.47,-0.01\r\n20080715,0.015,-3.51,0.4,-1.12\r\n20080716,0.015,-2.87,-2.63,-1.34\r\n20080717,0.015,-0.81,-0.03,0.94\r\n20080718,0.015,3.54,-0.55,-0.5\r\n20080721,0.015,3.1,1.32,0.37\r\n20080722,0.015,-0.46,1.24,0.27\r\n20080723,0.015,-0.42,-0.46,-0.58\r\n20080724,0.015,2.61,0.07,0.42\r\n20080725,0.015,-1.47,0.55,-0.59\r\n20080728,0.015,1.28,0.08,-0.43\r\n20080729,0.015,-1.86,-0.24,-0.56\r\n20080730,0.015,-0.49,0.07,-0.42\r\n20080731,0.015,-2.28,-0.91,1.15\r\n20080801,0.0164,1.34,0.06,-0.19\r\n20080804,0.0164,-2.5,-0.09,-0.8\r\n20080805,0.0164,-2.26,-0.69,-0.25\r\n20080806,0.0164,0.88,-0.78,0.06\r\n20080807,0.0164,0.2,0.29,-0.76\r\n20080808,0.0164,-5.15,-2.46,0.32\r\n20080811,0.0164,-5.43,-2.42,0.92\r\n20080812,0.0164,-0.63,-0.64,1.75\r\n20080813,0.0164,-0.22,-0.3,-0.06\r\n20080814,0.0164,-0.18,0.74,-0.12\r\n20080815,0.0164,0.16,-0.05,0.14\r\n20080818,0.0164,-5.83,-1.47,0.88\r\n20080819,0.0164,1.2,0.7,0.42\r\n20080820,0.0164,8.07,0.5,0.59\r\n20080821,0.0164,-3.65,0.79,0.64\r\n20080822,0.0164,-1.15,-0.6,0.08\r\n20080825,0.0164,0.08,-0.39,-0.21\r\n20080826,0.0164,-3.03,-2.48,0.42\r\n20080827,0.0164,0,-0.09,-0.18\r\n20080828,0.0164,0.43,-0.23,1.1\r\n20080829,0.0164,2.29,0.89,0.14\r\n20080901,0.0176,-3.22,0.23,-0.5\r\n20080902,0.0176,-0.88,1.7,-1.04\r\n20080903,0.0176,-1.3,0.77,-1.04\r\n20080904,0.0176,0.12,0.8,0.23\r\n20080905,0.0176,-3.4,-1.72,-0.09\r\n20080908,0.0176,-2.93,-1.73,1.73\r\n20080909,0.0176,0.12,0.82,0.34\r\n20080910,0.0176,0.26,0.37,-0.81\r\n20080911,0.0176,-3.29,1.08,-0.32\r\n20080912,0.0176,0.15,-0.3,-0.46\r\n20080916,0.0176,-4.16,0.86,-2.78\r\n20080917,0.0176,-2.88,0.37,-2.35\r\n20080918,0.0176,-1.9,-1.24,1.98\r\n20080919,0.0176,9.86,-0.16,0.34\r\n20080922,0.0176,7.5,-3.12,0.72\r\n20080923,0.0176,-2.1,-4.28,1.15\r\n20080924,0.0176,0.71,1.31,-1.05\r\n20080925,0.0176,3.74,-1.6,-0.55\r\n20080926,0.0176,0.02,-0.33,0.35\r\n20081006,0.0164,-5.19,1.36,-1.81\r\n20081007,0.0164,-0.84,0.12,-0.29\r\n20081008,0.0164,-3.15,0.69,-0.87\r\n20081009,0.0164,-0.98,-0.28,-0.23\r\n20081010,0.0164,-3.94,-2.96,0.71\r\n20081013,0.0164,3.54,-1.79,1.67\r\n20081014,0.0164,-2.75,-0.28,-0.71\r\n20081015,0.0164,-1.17,0.93,1.42\r\n20081016,0.0164,-4.42,-1.23,-0.1\r\n20081017,0.0164,1.08,0.38,-0.1\r\n20081020,0.0164,2.42,0.83,0.19\r\n20081021,0.0164,-0.77,1.03,0.05\r\n20081022,0.0164,-3.11,1.08,0.8\r\n20081023,0.0164,-0.87,2.07,0.11\r\n20081024,0.0164,-1.97,0.94,0.27\r\n20081027,0.0164,-6.57,-0.64,1.77\r\n20081028,0.0164,2.85,-0.44,1.14\r\n20081029,0.0164,-2.93,-0.31,1.13\r\n20081030,0.0164,2.42,-1.63,0.41\r\n20081031,0.0164,-1.99,0.2,0.51\r\n20081103,0.014,-0.65,-0.2,0.59\r\n20081104,0.014,-1.01,-0.69,-0.13\r\n20081105,0.014,3.24,0.51,0.18\r\n20081106,0.014,-2.43,1.02,-0.06\r\n20081107,0.014,1.65,0.28,-0.25\r\n20081110,0.014,7.45,0.06,-0.13\r\n20081111,0.014,-1.53,1.08,0.4\r\n20081112,0.014,0.94,2.1,-0.25\r\n20081113,0.014,3.9,0.81,-0.28\r\n20081114,0.014,3.4,1.29,-0.31\r\n20081117,0.014,2.41,2.45,-1.46\r\n20081118,0.014,-6.64,-0.46,0.17\r\n20081119,0.014,6.31,0.64,-0.08\r\n20081120,0.014,-1.48,1.49,-1.31\r\n20081121,0.014,-0.73,-0.92,-0.64\r\n20081124,0.014,-3.78,-0.35,0.3\r\n20081125,0.014,-0.43,-0.26,0.12\r\n20081126,0.014,0.51,-0.07,-0.23\r\n20081127,0.014,1.2,0.09,0.02\r\n20081128,0.014,-2.3,1.13,-0.26\r\n20081201,0.0089,1.48,3.72,0.37\r\n20081202,0.0089,-0.06,2.55,0.37\r\n20081203,0.0089,4.09,0.33,0.53\r\n20081204,0.0089,1.73,-2.56,-0.99\r\n20081205,0.0089,1.03,2.15,0.26\r\n20081208,0.0089,3.65,1.01,-0.09\r\n20081209,0.0089,-2.56,0.18,0.49\r\n20081210,0.0089,2.09,1.32,0.23\r\n20081211,0.0089,-2.45,-0.91,-0.44\r\n20081212,0.0089,-3.99,-1.06,-0.42\r\n20081215,0.0089,0.63,1.23,-0.15\r\n20081216,0.0089,0.77,1.06,0.53\r\n20081217,0.0089,0.21,1,-0.34\r\n20081218,0.0089,1.98,0.15,0.48\r\n20081219,0.0089,0.24,0.53,-0.62\r\n20081222,0.0089,-1.43,2.34,0.98\r\n20081223,0.0089,-4.81,-1.77,0.38\r\n20081224,0.0089,-1.88,-0.12,0.46\r\n20081225,0.0089,-0.62,0.44,0.38\r\n20081226,0.0089,-0.05,-0.52,0.7\r\n20081229,0.0089,-0.16,-0.64,-0.8\r\n20081230,0.0089,-0.87,0.24,0.22\r\n20081231,0.0089,-0.65,-0.95,0.75\r\n20090105,0.0126,3.38,1.11,-0.34\r\n20090106,0.0126,2.99,0.82,0.36\r\n20090107,0.0126,-0.61,1.46,-0.06\r\n20090108,0.0126,-2.34,0,0.26\r\n20090109,0.0126,1.56,1.62,0.42\r\n20090112,0.0126,-0.14,1.69,-0.05\r\n20090113,0.0126,-2.13,-0.9,-0.46\r\n20090114,0.0126,3.65,0.66,0.3\r\n20090115,0.0126,-0.37,1.7,-0.15\r\n20090116,0.0126,1.68,-1.17,-0.14\r\n20090119,0.0126,1.43,-1.66,-0.16\r\n20090120,0.0126,0.48,0.48,0.1\r\n20090121,0.0126,-0.47,0.78,0.66\r\n20090122,0.0126,1.06,0.75,0.26\r\n20090123,0.0126,-0.7,-0.35,-0.35\r\n20090202,0.0085,1.19,2.15,-0.35\r\n20090203,0.0085,2.44,0.5,-0.43\r\n20090204,0.0085,2.31,-0.1,-0.22\r\n20090205,0.0085,-0.61,-0.84,-0.46\r\n20090206,0.0085,4.08,0.54,0.18\r\n20090209,0.0085,2.24,0.94,-0.14\r\n20090210,0.0085,1.94,0.98,-0.39\r\n20090211,0.0085,-0.15,-0.24,-0.47\r\n20090212,0.0085,-0.39,1.92,-1.17\r\n20090213,0.0085,3.36,-0.06,-0.63\r\n20090216,0.0085,2.81,-0.68,-0.1\r\n20090217,0.0085,-3.09,-1.09,-0.27\r\n20090218,0.0085,-4.72,0.94,0.15\r\n20090219,0.0085,1.04,0.86,-0.39\r\n20090220,0.0085,1.76,1.78,0.05\r\n20090223,0.0085,2.16,1.31,-0.37\r\n20090224,0.0085,-4.48,0.69,0.02\r\n20090225,0.0085,0.08,0.98,0.08\r\n20090226,0.0085,-4.31,-3.11,-0.27\r\n20090227,0.0085,-2.17,-4.57,1.08\r\n20090302,0.0086,0.79,1.87,0.2\r\n20090303,0.0086,-0.83,1.98,0.36\r\n20090304,0.0086,6.17,-0.24,-0.16\r\n20090305,0.0086,0.89,-0.03,-0.08\r\n20090306,0.0086,-1.23,0.56,-0.33\r\n20090309,0.0086,-3.55,-0.7,0.52\r\n20090310,0.0086,1.93,0.8,-0.18\r\n20090311,0.0086,-0.9,0.38,-0.07\r\n20090312,0.0086,-0.18,0.67,-0.56\r\n20090313,0.0086,-0.3,-0.66,0.1\r\n20090316,0.0086,1.2,0.15,0.05\r\n20090317,0.0086,3.22,0.45,0.15\r\n20090318,0.0086,0.41,0.87,0.12\r\n20090319,0.0086,1.92,0.4,-0.25\r\n20090320,0.0086,0.62,-0.98,0.56\r\n20090323,0.0086,1.99,-0.37,-0.37\r\n20090324,0.0086,0.6,0.39,-0.42\r\n20090325,0.0086,-2.11,-0.3,-0.15\r\n20090326,0.0086,2.88,-0.46,-0.07\r\n20090327,0.0086,0.53,0.33,-0.01\r\n20090330,0.0086,-0.69,1.32,-0.07\r\n20090331,0.0086,0.71,0,-0.25\r\n20090401,0.0087,1.52,0.48,0.29\r\n20090402,0.0087,0.75,-0.65,-0.07\r\n20090403,0.0087,-0.3,-0.09,0.98\r\n20090407,0.0087,0.75,0.71,0.17\r\n20090408,0.0087,-3.75,0.39,0.14\r\n20090409,0.0087,1.43,0.36,-0.73\r\n20090410,0.0087,2.87,0.16,-0.41\r\n20090413,0.0087,2.67,-1.25,0.38\r\n20090414,0.0087,0.73,1.21,0.21\r\n20090415,0.0087,0.48,1.4,0.14\r\n20090416,0.0087,-0.15,0.56,-0.32\r\n20090417,0.0087,-1.34,0.09,0.08\r\n20090420,0.0087,2.11,0.69,0.12\r\n20090421,0.0087,-0.95,0.99,-0.17\r\n20090422,0.0087,-3.19,-1.4,0.53\r\n20090423,0.0087,0.2,0.47,0.34\r\n20090424,0.0087,-0.62,-0.05,0.71\r\n20090427,0.0087,-2,-1.88,1.64\r\n20090428,0.0087,-0.04,1.34,0.24\r\n20090429,0.0087,2.74,-0.23,-0.3\r\n20090430,0.0087,0.51,0.73,-0.38\r\n20090504,0.0105,3.47,-0.29,-0.56\r\n20090505,0.0105,0.34,0.86,-0.42\r\n20090506,0.0105,1.11,0.25,-0.88\r\n20090507,0.0105,-0.02,-0.55,0.21\r\n20090508,0.0105,0.93,-0.3,0.06\r\n20090511,0.0105,-2.06,-1.62,0.78\r\n20090512,0.0105,1.69,0.19,-0.8\r\n20090513,0.0105,1.64,-0.38,0.52\r\n20090514,0.0105,-0.79,0.93,-0.51\r\n20090515,0.0105,0.25,0.81,0.29\r\n20090518,0.0105,0.26,0.6,-0.45\r\n20090519,0.0105,0.95,-0.43,-0.93\r\n20090520,0.0105,-0.82,0.22,-0.21\r\n20090521,0.0105,-1.68,-0.07,-0.15\r\n20090522,0.0105,-0.48,0.63,-0.31\r\n20090525,0.0105,0.52,-0.37,-0.24\r\n20090526,0.0105,-0.83,1.44,0.43\r\n20090527,0.0105,1.57,-1.22,0.62\r\n20090601,0.0083,3.32,-1.91,0.31\r\n20090602,0.0083,0.21,-0.2,-0.81\r\n20090603,0.0083,1.93,-1.1,0.12\r\n20090604,0.0083,-0.13,-0.63,-0.87\r\n20090605,0.0083,-0.5,0.02,0.23\r\n20090608,0.0083,0.35,0.03,0.55\r\n20090609,0.0083,0.7,-0.04,0.26\r\n20090610,0.0083,1.02,0.19,-0.18\r\n20090611,0.0083,-0.73,0.57,0.54\r\n20090612,0.0083,-1.89,-0.33,0\r\n20090615,0.0083,1.68,0.4,0.88\r\n20090616,0.0083,-0.38,0.11,-0.07\r\n20090617,0.0083,1.35,-0.33,-0.47\r\n20090618,0.0083,1.48,-0.62,0.75\r\n20090619,0.0083,0.84,-0.32,1.06\r\n20090622,0.0083,0.4,-1.1,0.14\r\n20090623,0.0083,-0.15,-0.29,-0.02\r\n20090624,0.0083,1.22,-0.72,-1.06\r\n20090625,0.0083,0.06,0,0.18\r\n20090626,0.0083,0.28,0.31,0\r\n20090629,0.0083,1.54,-0.83,0.47\r\n20090630,0.0083,-0.56,-0.27,0.28\r\n20090701,0.0082,1.63,-0.89,-0.37\r\n20090702,0.0082,1.59,-0.47,0.1\r\n20090703,0.0082,1.01,-0.06,-0.13\r\n20090706,0.0082,1.19,-1.47,-0.53\r\n20090707,0.0082,-1.01,1.18,-0.11\r\n20090708,0.0082,-0.03,0.86,-0.65\r\n20090709,0.0082,1.41,0.57,0.17\r\n20090710,0.0082,-0.16,0.52,-0.54\r\n20090713,0.0082,-0.84,1.61,-0.57\r\n20090714,0.0082,2.04,-0.65,-0.14\r\n20090715,0.0082,1.3,-0.81,-0.77\r\n20090716,0.0082,-0.16,-0.38,-0.63\r\n20090717,0.0082,0.29,-0.12,-1.27\r\n20090720,0.0082,2.4,-0.9,-0.43\r\n20090721,0.0082,-1.77,-0.39,0.62\r\n20090722,0.0082,2.49,-0.28,0.69\r\n20090723,0.0082,0.93,-0.05,0.1\r\n20090724,0.0082,1.15,-1.59,-0.01\r\n20090727,0.0082,1.81,-0.69,-1.54\r\n20090728,0.0082,0.28,0.31,-1.29\r\n20090729,0.0082,-5.19,-0.25,1.2\r\n20090730,0.0082,1.64,-0.33,0.6\r\n20090731,0.0082,2.8,-0.38,-1.37\r\n20090803,0.009,1.6,-0.2,-1.73\r\n20090804,0.009,0.34,0.93,0.17\r\n20090805,0.009,-1.18,1.58,0.03\r\n20090806,0.009,-2.07,0.88,0.91\r\n20090807,0.009,-3.02,0.8,1.28\r\n20090810,0.009,-0.19,1.27,0.12\r\n20090811,0.009,0.51,0.46,0.44\r\n20090812,0.009,-4.62,0.14,0.65\r\n20090813,0.009,0.93,-1.52,-0.48\r\n20090814,0.009,-3.02,-1.55,0.57\r\n20090817,0.009,-6.05,0.21,1.77\r\n20090818,0.009,1.4,0.1,-0.64\r\n20090819,0.009,-4.4,0.64,2.67\r\n20090820,0.009,4.43,-0.52,0.22\r\n20090821,0.009,1.81,1.23,-0.17\r\n20090824,0.009,1.27,1.13,0.14\r\n20090825,0.009,-2.59,2.09,1.03\r\n20090826,0.009,1.93,0.86,-0.88\r\n20090827,0.009,-0.63,1.67,-0.75\r\n20090828,0.009,-3,0.53,-0.22\r\n20090831,0.009,-7.01,0.7,0.69\r\n20090901,0.0083,0.49,-1.04,1.65\r\n20090902,0.0083,1.12,-0.27,0.83\r\n20090903,0.0083,4.96,-0.97,-1.06\r\n20090904,0.0083,0.71,0.91,-0.33\r\n20090907,0.0083,0.76,0.52,0\r\n20090908,0.0083,1.84,-0.55,-0.14\r\n20090909,0.0083,0.66,-0.74,-0.84\r\n20090910,0.0083,-0.81,-0.12,-0.02\r\n20090911,0.0083,2.11,-0.14,0.63\r\n20090914,0.0083,1.42,1.14,-0.45\r\n20090915,0.0083,0.37,0.36,-1.14\r\n20090916,0.0083,-1.02,1.52,-0.04\r\n20090917,0.0083,2.07,0.63,0.24\r\n20090918,0.0083,-3.26,0.36,0.29\r\n20090921,0.0083,0.28,1.45,0.9\r\n20090922,0.0083,-2.28,0.03,0.9\r\n20090923,0.0083,-2.15,-0.58,0.57\r\n20090924,0.0083,0.26,-0.82,0.71\r\n20090925,0.0083,-0.61,-0.02,0.6\r\n20090928,0.0083,-2.64,-0.05,0.74\r\n20090929,0.0083,-0.49,-1.31,1.21\r\n20090930,0.0083,1.02,-0.2,0.1\r\n20091009,0.0118,4.88,-0.23,-0.86\r\n20091012,0.0118,-0.49,1.24,0.29\r\n20091013,0.0118,1.44,0.27,-0.11\r\n20091014,0.0118,0.99,-0.06,-0.27\r\n20091015,0.0118,0.34,0.3,0.45\r\n20091016,0.0118,-0.06,0.3,-0.14\r\n20091019,0.0118,2.25,-0.38,-0.75\r\n20091020,0.0118,1.5,0.1,-0.04\r\n20091021,0.0118,-0.44,-0.28,-0.32\r\n20091022,0.0118,-0.56,1.08,-0.08\r\n20091023,0.0118,1.83,-0.44,0.05\r\n20091026,0.0118,0.07,-0.08,0.12\r\n20091027,0.0118,-2.86,0.53,0.82\r\n20091028,0.0118,0.45,0.67,0.05\r\n20091029,0.0118,-2.29,0.79,0.71\r\n20091030,0.0118,1.17,0.17,0.35\r\n20091102,0.0087,2.82,0.3,0.11\r\n20091103,0.0087,1.31,1.08,-0.48\r\n20091104,0.0087,0.53,0.24,-0.43\r\n20091105,0.0087,0.73,0.52,0.08\r\n20091106,0.0087,0.35,-0.01,-0.38\r\n20091109,0.0087,0.36,0.59,-0.37\r\n20091110,0.0087,0.15,-0.11,0.15\r\n20091111,0.0087,-0.1,0.62,-0.29\r\n20091112,0.0087,0.01,0.44,0.08\r\n20091113,0.0087,0.46,0.49,-0.16\r\n20091116,0.0087,2.79,-0.61,0.28\r\n20091117,0.0087,0.25,0.53,0.16\r\n20091118,0.0087,0.5,0.28,0.13\r\n20091119,0.0087,0.54,1.3,-0.25\r\n20091120,0.0087,-0.23,1.21,-0.03\r\n20091123,0.0087,1.02,0.88,-0.52\r\n20091124,0.0087,-3.55,-1.76,0.84\r\n20091125,0.0087,2.24,1.54,-0.33\r\n20091126,0.0087,-3.7,-0.86,-0.55\r\n20091127,0.0087,-2.52,0.36,1.17\r\n20091130,0.0087,3.43,1.15,0.04\r\n20091201,0.0082,1.38,1.41,-0.37\r\n20091202,0.0082,1.06,0.14,-0.7\r\n20091203,0.0082,-0.11,0.83,-0.54\r\n20091204,0.0082,1.11,-3.12,1.76\r\n20091207,0.0082,0.62,1.05,-0.62\r\n20091208,0.0082,-0.93,0.84,0\r\n20091209,0.0082,-1.66,0.19,0.34\r\n20091210,0.0082,0.44,0.43,0.01\r\n20091211,0.0082,-0.18,0.15,-0.13\r\n20091214,0.0082,1.46,-1.49,0.84\r\n20091215,0.0082,-0.78,1.43,-0.06\r\n20091216,0.0082,-0.7,0.3,0.08\r\n20091217,0.0082,-2.4,-0.46,0.37\r\n20091218,0.0082,-2.38,-0.49,1.51\r\n20091221,0.0082,0.39,0.92,-0.38\r\n20091222,0.0082,-2.35,0.01,0.88\r\n20091223,0.0082,0.93,0.85,0.01\r\n20091224,0.0082,2.79,0.33,0.09\r\n20091225,0.0082,-0.26,0.75,-0.46\r\n20091228,0.0082,1.5,0.13,-0.14\r\n20091229,0.0082,0.67,0.13,0.64\r\n20091230,0.0082,1.4,-1.74,0.65\r\n20091231,0.0082,0.43,0.56,0.02\r\n20100104,0.0095,-0.93,1.31,-0.51\r\n20100105,0.0095,1,-0.03,-0.17\r\n20100106,0.0095,-0.73,0.29,-0.43\r\n20100107,0.0095,-1.95,-0.46,-0.23\r\n20100108,0.0095,0.15,1.07,-0.22\r\n20100111,0.0095,0.34,0.32,-0.24\r\n20100112,0.0095,1.92,0.33,0.06\r\n20100113,0.0095,-2.94,1.73,-0.63\r\n20100114,0.0095,1.45,1.13,-0.43\r\n20100115,0.0095,0.35,0.53,-0.05\r\n20100118,0.0095,0.59,0.68,-0.63\r\n20100119,0.0095,0.17,-0.17,0.12\r\n20100120,0.0095,-3.08,-0.43,-0.03\r\n20100121,0.0095,0.18,0.32,0.22\r\n20100122,0.0095,-1.28,-0.88,0.82\r\n20100125,0.0095,-1.11,-0.5,0.03\r\n20100126,0.0095,-2.55,-0.85,1.15\r\n20100127,0.0095,-1.06,0.4,0.37\r\n20100128,0.0095,0.35,0.74,-0.39\r\n20100129,0.0095,-0.02,0.95,-0.23\r\n20100201,0.0114,-1.63,0.36,0.04\r\n20100202,0.0114,-0.29,-0.34,-0.32\r\n20100203,0.0114,2.35,-0.49,0.26\r\n20100204,0.0114,-0.26,0.95,-0.78\r\n20100205,0.0114,-1.97,-0.33,-0.01\r\n20100208,0.0114,-0.15,0.08,-0.34\r\n20100209,0.0114,0.43,-0.22,0.3\r\n20100210,0.0114,1.19,-0.01,0.21\r\n20100211,0.0114,0.14,-0.46,0.19\r\n20100212,0.0114,1.13,0.11,-0.23\r\n20100222,0.0114,-0.49,0.72,-0.4\r\n20100223,0.0114,-0.63,1.53,-0.54\r\n20100224,0.0114,1.47,0.74,-0.07\r\n20100225,0.0114,1.33,0.18,-0.23\r\n20100226,0.0114,-0.25,0.41,-0.46\r\n20100301,0.0082,1.17,0.13,0.05\r\n20100302,0.0082,-0.42,0.5,-0.24\r\n20100303,0.0082,0.77,0.37,-0.35\r\n20100304,0.0082,-2.49,-0.84,0.41\r\n20100305,0.0082,0.17,0.13,-0.26\r\n20100308,0.0082,0.77,0.39,-0.13\r\n20100309,0.0082,0.51,-0.19,0.36\r\n20100310,0.0082,-0.77,-0.4,0.14\r\n20100311,0.0082,-0.08,-0.21,0.51\r\n20100312,0.0082,-1.28,-0.1,0.32\r\n20100315,0.0082,-1.27,0.25,-0.45\r\n20100316,0.0082,0.52,0.37,-0.27\r\n20100317,0.0082,2.06,0.45,-0.23\r\n20100318,0.0082,-0.14,0.35,-0.2\r\n20100319,0.0082,0.71,-0.07,-0.39\r\n20100322,0.0082,0.23,0.57,0.08\r\n20100323,0.0082,-0.66,0.2,0.32\r\n20100324,0.0082,0.15,0.35,0.18\r\n20100325,0.0082,-1.26,0.06,0.38\r\n20100326,0.0082,1.28,0.26,0.33\r\n20100329,0.0082,1.96,-1.07,0.73\r\n20100330,0.0082,0.22,0.3,0.12\r\n20100331,0.0082,-0.53,0.85,-0.29\r\n20100401,0.0087,1.27,0.33,0.03\r\n20100402,0.0087,0.43,0.06,-0.11\r\n20100406,0.0087,-0.07,0.43,-0.26\r\n20100407,0.0087,-0.29,0.42,-0.35\r\n20100408,0.0087,-0.95,0.88,-0.28\r\n20100409,0.0087,0.93,0.44,-0.16\r\n20100412,0.0087,-0.42,0.87,-0.97\r\n20100413,0.0087,0.78,-2.14,0.91\r\n20100414,0.0087,0.28,1,-0.29\r\n20100415,0.0087,-0.19,-0.81,0\r\n20100416,0.0087,-0.97,0.77,0\r\n20100419,0.0087,-4.83,0.81,0.23\r\n20100420,0.0087,-0.01,1.3,-0.35\r\n20100421,0.0087,1.98,0.85,-0.33\r\n20100422,0.0087,-0.9,1.12,-0.35\r\n20100423,0.0087,-0.58,-0.19,0.43\r\n20100426,0.0087,-0.44,-0.02,-0.76\r\n20100427,0.0087,-2.12,-0.71,0.18\r\n20100428,0.0087,-0.32,0.01,0.74\r\n20100429,0.0087,-1.42,-1.7,0.48\r\n20100430,0.0087,-0.18,-2.62,1.3\r\n20100504,0.0095,-1.05,0.41,-0.5\r\n20100505,0.0095,1,1.89,-0.62\r\n20100506,0.0095,-4.05,0.21,-1.26\r\n20100507,0.0095,-1.97,0.2,-0.44\r\n20100510,0.0095,0.13,-1.05,1.33\r\n20100511,0.0095,-1.94,-0.89,0.33\r\n20100512,0.0095,0.11,-1.57,1.94\r\n20100513,0.0095,2.22,0.17,0.07\r\n20100514,0.0095,-0.41,0.38,-0.88\r\n20100517,0.0095,-5.23,-1.53,0.12\r\n20100518,0.0095,1.52,0.22,1.08\r\n20100519,0.0095,-0.16,-0.14,-0.31\r\n20100520,0.0095,-1.19,-0.68,0.71\r\n20100521,0.0095,1.38,0.43,-0.15\r\n20100524,0.0095,3.57,0.86,-0.46\r\n20100525,0.0095,-1.82,0.96,-0.34\r\n20100526,0.0095,0.08,0.21,-0.68\r\n20100527,0.0095,1.5,0.12,-0.48\r\n20100528,0.0095,0.03,0.38,0.13\r\n20100531,0.0095,-2.37,-0.54,0.4\r\n20100601,0.0096,-1.01,-0.88,0.36\r\n20100602,0.0096,0.38,1.23,-0.57\r\n20100603,0.0096,-0.57,-0.4,-0.21\r\n20100604,0.0096,0.35,0.54,-0.24\r\n20100607,0.0096,-1.4,0.64,-0.76\r\n20100608,0.0096,0.19,0.77,-0.5\r\n20100609,0.0096,2.76,0.3,1.03\r\n20100610,0.0096,-0.64,0.5,-0.63\r\n20100611,0.0096,0.17,-0.35,0.49\r\n20100617,0.0096,-0.57,0.26,0.61\r\n20100618,0.0096,-2.18,-1.1,1.64\r\n20100621,0.0096,2.9,0.5,0.62\r\n20100622,0.0096,0.19,0.4,-0.29\r\n20100623,0.0096,-0.69,0.35,0.02\r\n20100624,0.0096,0.02,0.14,0.06\r\n20100625,0.0096,-0.81,-0.68,0.15\r\n20100628,0.0096,-0.86,-0.36,0.76\r\n20100629,0.0096,-4.49,-1.37,0.35\r\n20100630,0.0096,-1.17,-1.29,0.31\r\n20100701,0.0086,-1.12,0.16,0.56\r\n20100702,0.0086,0.2,0.12,1.03\r\n20100705,0.0086,-0.68,0.21,0.5\r\n20100706,0.0086,2.13,0.73,-0.18\r\n20100707,0.0086,0.7,0.5,-0.32\r\n20100708,0.0086,-0.24,-0.1,0.23\r\n20100709,0.0086,2.52,0.33,0.12\r\n20100712,0.0086,0.84,0.58,0.1\r\n20100713,0.0086,-1.62,-0.34,-0.57\r\n20100714,0.0086,0.72,0.43,-0.33\r\n20100715,0.0086,-1.9,-0.46,0.22\r\n20100716,0.0086,0.21,0.69,0.24\r\n20100719,0.0086,2.2,0.31,0.38\r\n20100720,0.0086,2.34,0.56,-0.27\r\n20100721,0.0086,0.21,-0.22,-0.51\r\n20100722,0.0086,1.18,0.57,0.23\r\n20100723,0.0086,0.36,-0.12,0.44\r\n20100726,0.0086,0.75,0.24,-0.52\r\n20100727,0.0086,-0.44,0.1,-0.2\r\n20100728,0.0086,2.29,0.05,0.2\r\n20100729,0.0086,0.53,0.25,-0.23\r\n20100730,0.0086,-0.31,0.22,-0.14\r\n20100802,0.0086,1.58,0.56,0.31\r\n20100803,0.0086,-1.74,0.15,0.04\r\n20100804,0.0086,0.5,0.53,-0.36\r\n20100805,0.0086,-0.59,0.86,-0.34\r\n20100806,0.0086,1.63,0.27,-0.16\r\n20100809,0.0086,0.72,0.34,-0.49\r\n20100810,0.0086,-3.09,-0.71,0.03\r\n20100811,0.0086,0.6,0.56,-0.27\r\n20100812,0.0086,-1.29,0.1,0\r\n20100813,0.0086,1.39,0.49,-0.15\r\n20100816,0.0086,2.27,-0.09,0.05\r\n20100817,0.0086,0.54,0.19,-0.29\r\n20100818,0.0086,-0.16,0.04,-0.28\r\n20100819,0.0086,0.73,-0.38,0.11\r\n20100820,0.0086,-1.89,-0.37,-0.48\r\n20100823,0.0086,-0.05,0.62,-0.34\r\n20100824,0.0086,0.58,0.56,-0.3\r\n20100825,0.0086,-2.17,0.12,-0.02\r\n20100826,0.0086,0.32,0.21,-0.15\r\n20100827,0.0086,0.42,0.14,0\r\n20100830,0.0086,1.93,0.33,-0.22\r\n20100831,0.0086,-0.26,0.79,-0.48\r\n20100901,0.0096,-0.8,-0.33,0.7\r\n20100902,0.0096,1.3,0.85,-0.1\r\n20100903,0.0096,0.03,0.58,-0.61\r\n20100906,0.0096,1.3,-1.17,0.83\r\n20100907,0.0096,0.23,0.46,-0.4\r\n20100908,0.0096,0.12,0.06,-1.02\r\n20100909,0.0096,-1.48,-0.36,-0.66\r\n20100910,0.0096,0.34,0.47,-0.28\r\n20100913,0.0096,1.17,0.46,0.02\r\n20100914,0.0096,0.03,0.22,-0.28\r\n20100915,0.0096,-1.57,-0.06,0.21\r\n20100916,0.0096,-1.86,-0.71,0.35\r\n20100917,0.0096,-0.03,-0.14,0.1\r\n20100920,0.0096,-0.62,-0.51,0.5\r\n20100921,0.0096,0.13,-0.06,0.55\r\n20100927,0.0096,1.67,0.34,-1.13\r\n20100928,0.0096,-0.64,-0.51,-1.49\r\n20100929,0.0096,-0.24,-0.89,0.3\r\n20100930,0.0096,1.66,-0.18,0.5\r\n20101008,0.0124,3.16,-0.94,0.19\r\n20101011,0.0124,2.21,-0.64,1.58\r\n20101012,0.0124,1.22,-0.65,0.39\r\n20101013,0.0124,0.68,0.48,0.86\r\n20101014,0.0124,-0.06,-2.84,0.82\r\n20101015,0.0124,2.52,-1.95,1.11\r\n20101018,0.0124,-0.76,0.44,1.15\r\n20101019,0.0124,1.98,0.35,-0.96\r\n20101020,0.0124,0.25,-0.45,-0.28\r\n20101021,0.0124,-0.21,0.59,-0.61\r\n20101022,0.0124,0.05,0.54,-0.62\r\n20101025,0.0124,2.81,-0.78,-0.37\r\n20101026,0.0124,-0.13,0.9,-0.32\r\n20101027,0.0124,-1.51,0.73,-0.1\r\n20101028,0.0124,-0.18,0.36,-0.26\r\n20101029,0.0124,-0.16,0.62,-1.11\r\n20101101,0.0093,2.74,-0.34,-0.45\r\n20101102,0.0093,-0.42,0.03,-0.72\r\n20101103,0.0093,-1.02,0.55,1.06\r\n20101104,0.0093,1.95,-0.19,-0.98\r\n20101105,0.0093,1.3,0.26,0.16\r\n20101108,0.0093,1.11,1.37,0.27\r\n20101109,0.0093,-0.48,0.66,-0.37\r\n20101110,0.0093,-0.31,0.16,-0.88\r\n20101111,0.0093,0.68,-1.26,1.38\r\n20101112,0.0093,-5.55,-0.68,0.49\r\n20101115,0.0093,1.19,1.54,0.23\r\n20101116,0.0093,-4.04,0.67,-0.27\r\n20101117,0.0093,-2.39,-0.72,1.18\r\n20101118,0.0093,1.24,-0.01,0.02\r\n20101119,0.0093,1.3,1.17,-0.57\r\n20101122,0.0093,0.23,0.88,-0.83\r\n20101123,0.0093,-1.85,0.72,0.84\r\n20101124,0.0093,1.56,0.71,-0.83\r\n20101125,0.0093,1.14,-0.08,-0.13\r\n20101126,0.0093,-0.84,0.53,-0.07\r\n20101129,0.0093,-0.1,0.6,0.27\r\n20101130,0.0093,-1.83,-0.96,0.47\r\n20101201,0.0093,-0.18,0.5,0.03\r\n20101202,0.0093,0.49,0.35,0.27\r\n20101203,0.0093,-0.14,-0.39,0.59\r\n20101206,0.0093,0.12,-1,1.38\r\n20101207,0.0093,1.1,-0.17,-0.34\r\n20101208,0.0093,-0.77,0.44,-0.14\r\n20101209,0.0093,-1.61,-0.34,0.25\r\n20101210,0.0093,1.27,-0.12,-0.1\r\n20101213,0.0093,3.19,-0.39,0.17\r\n20101214,0.0093,0.36,0.77,-0.64\r\n20101215,0.0093,-0.5,0.26,-0.01\r\n20101216,0.0093,-0.39,0.44,0.07\r\n20101217,0.0093,-0.15,0.39,0.01\r\n20101220,0.0093,-1.5,0.04,-0.46\r\n20101221,0.0093,1.79,-0.15,1.02\r\n20101222,0.0093,-0.92,0.02,-0.14\r\n20101223,0.0093,-1.15,-0.59,0.61\r\n20101224,0.0093,-1.1,-0.23,0.88\r\n20101227,0.0093,-2.17,-0.53,0.97\r\n20101228,0.0093,-1.84,-0.5,0.44\r\n20101229,0.0093,0.86,0.42,-0.5\r\n20101230,0.0093,0.21,-0.06,-0.02\r\n20101231,0.0093,2.1,-0.29,-0.31\r\n20110104,0.0115,1.61,0.05,0.1\r\n20110105,0.0115,-0.39,0.76,-0.05\r\n20110106,0.0115,-0.6,0.24,-0.43\r\n20110107,0.0115,0.11,-0.06,0.92\r\n20110110,0.0115,-1.99,-0.41,0.63\r\n20110111,0.0115,0.23,-0.36,0.77\r\n20110112,0.0115,0.52,0.45,0.45\r\n20110113,0.0115,0.2,-0.35,0.14\r\n20110114,0.0115,-1.47,0.03,0.77\r\n20110117,0.0115,-3.41,-0.79,0.35\r\n20110118,0.0115,0.08,0.15,-0.15\r\n20110119,0.0115,2.02,0.01,-0.46\r\n20110120,0.0115,-3.14,-0.12,-0.11\r\n20110121,0.0115,1.17,0.05,0.51\r\n20110124,0.0115,-1.18,-0.2,0.98\r\n20110125,0.0115,-0.85,-0.56,0.92\r\n20110126,0.0115,1.31,0.08,-0.41\r\n20110127,0.0115,1.62,-0.4,-0.18\r\n20110128,0.0115,0.34,0.44,-0.63\r\n20110131,0.0115,1.27,0.21,0.21\r\n20110201,0.0148,0.22,0.02,-0.37\r\n20110209,0.0148,-0.97,0.73,-0.62\r\n20110210,0.0148,1.9,0.73,-0.16\r\n20110211,0.0148,0.51,0.5,0.09\r\n20110214,0.0148,2.54,-0.75,0.51\r\n20110215,0.0148,-0.05,-0.09,-0.51\r\n20110216,0.0148,1.03,0.68,-0.08\r\n20110217,0.0148,0.12,0.1,-0.68\r\n20110218,0.0148,-0.99,0.46,0.15\r\n20110221,0.0148,1.32,0.71,-0.29\r\n20110222,0.0148,-2.62,-0.03,-0.19\r\n20110223,0.0148,0.35,0.84,-0.43\r\n20110224,0.0148,0.51,0.11,-0.35\r\n20110225,0.0148,0.01,0.01,-0.38\r\n20110228,0.0148,0.98,0.44,0.18\r\n20110301,0.0109,0.41,0.06,-0.08\r\n20110302,0.0109,-0.28,0.26,0.36\r\n20110303,0.0109,-0.71,-0.48,0.74\r\n20110304,0.0109,1.34,0.04,0.1\r\n20110307,0.0109,1.8,-0.47,0.39\r\n20110308,0.0109,0.15,0.39,-0.42\r\n20110309,0.0109,0.09,0.14,-0.17\r\n20110310,0.0109,-1.39,0.87,-0.11\r\n20110311,0.0109,-0.72,0.6,-0.12\r\n20110314,0.0109,0.3,0.58,0.06\r\n20110315,0.0109,-1.48,0.42,0.06\r\n20110316,0.0109,1.17,-0.38,-0.68\r\n20110317,0.0109,-1.31,-0.15,0.5\r\n20110318,0.0109,0.37,0.31,-0.09\r\n20110321,0.0109,-0.12,-0.04,0.47\r\n20110322,0.0109,0.26,-0.13,0.25\r\n20110323,0.0109,1.06,0.05,-0.09\r\n20110324,0.0109,-0.1,0.47,-0.12\r\n20110325,0.0109,0.96,-0.15,0.31\r\n20110328,0.0109,-0.07,-0.05,-0.2\r\n20110329,0.0109,-1.12,-0.06,0.24\r\n20110330,0.0109,-0.24,-1.02,0.16\r\n20110331,0.0109,-0.9,-0.48,0.04\r\n20110401,0.0137,1.27,0.26,0.34\r\n20110406,0.0137,0.8,-0.52,0.63\r\n20110407,0.0137,0.3,0.31,-0.19\r\n20110408,0.0137,0.77,-0.02,-0.37\r\n20110411,0.0137,-0.39,-0.67,-0.35\r\n20110412,0.0137,-0.14,0.12,-0.19\r\n20110413,0.0137,1.04,0.2,0.48\r\n20110414,0.0137,-0.27,0.13,-0.17\r\n20110415,0.0137,0.15,0.12,0.26\r\n20110418,0.0137,0.13,0.45,0.07\r\n20110419,0.0137,-1.7,0.55,-0.47\r\n20110420,0.0137,0.28,0.17,-0.09\r\n20110421,0.0137,0.62,0.01,0.27\r\n20110422,0.0137,-0.56,-0.19,-0.16\r\n20110425,0.0137,-1.61,-0.38,0.71\r\n20110426,0.0137,-0.93,-0.57,0.5\r\n20110427,0.0137,-0.57,-0.4,0.43\r\n20110428,0.0137,-1.61,-1.18,1.17\r\n20110429,0.0137,0.97,-0.04,-0.75\r\n20110503,0.013,0.73,0.83,-0.34\r\n20110504,0.013,-2.29,0.11,-0.29\r\n20110505,0.013,0.2,0.49,0.25\r\n20110506,0.013,-0.13,0.42,-0.53\r\n20110509,0.013,0.34,0.32,-0.3\r\n20110510,0.013,0.57,0,-0.42\r\n20110511,0.013,-0.17,0.56,-0.03\r\n20110512,0.013,-1.39,0.19,0.08\r\n20110513,0.013,0.82,0.09,0.25\r\n20110516,0.013,-0.66,0.71,-0.09\r\n20110517,0.013,0.1,-0.6,0.31\r\n20110518,0.013,0.66,-0.04,0.13\r\n20110519,0.013,-0.46,0.06,0.23\r\n20110520,0.013,-0.11,-0.15,0.17\r\n20110523,0.013,-3.11,-0.41,0.23\r\n20110524,0.013,-0.2,0.02,-0.3\r\n20110525,0.013,-1.06,0.03,0\r\n20110526,0.013,-0.41,-0.39,0.2\r\n20110527,0.013,-1.11,-1.98,0.75\r\n20110530,0.013,-0.28,-1.16,0.95\r\n20110531,0.013,1.46,0.5,-0.14\r\n20110601,0.0126,0.42,0.67,-0.43\r\n20110602,0.0126,-1.45,-0.31,-0.49\r\n20110603,0.0126,1.16,0.36,-0.35\r\n20110607,0.0126,0.67,0.12,-0.27\r\n20110608,0.0126,0.24,0.1,-0.57\r\n20110609,0.0126,-1.74,-0.51,0.05\r\n20110610,0.0126,0.21,0.02,-0.05\r\n20110613,0.0126,-0.26,0.05,-0.18\r\n20110614,0.0126,1.32,0.27,0.15\r\n20110615,0.0126,-0.76,-0.07,0.17\r\n20110616,0.0126,-1.68,-0.43,0.43\r\n20110617,0.0126,-0.85,0,0.92\r\n20110620,0.0126,-0.82,-0.19,0.78\r\n20110621,0.0126,1.08,0.6,0.42\r\n20110622,0.0126,0.03,-0.05,0.06\r\n20110623,0.0126,1.66,0.22,0.29\r\n20110624,0.0126,2.23,-0.01,-0.19\r\n20110627,0.0126,0.6,0.6,-0.71\r\n20110628,0.0126,0.04,0.05,-0.18\r\n20110629,0.0126,-1.08,-0.13,0.22\r\n20110630,0.0126,1.35,-0.09,0.25\r\n20110701,0.0137,0.04,0.38,-0.05\r\n20110704,0.0137,2.05,0.1,-0.14\r\n20110705,0.0137,0.21,0.49,-0.39\r\n20110706,0.0137,-0.09,0.35,-0.44\r\n20110707,0.0137,-0.45,0.38,-0.01\r\n20110708,0.0137,0.06,-0.13,0.03\r\n20110711,0.0137,0.3,0.59,-0.09\r\n20110712,0.0137,-1.64,0.15,0.29\r\n20110713,0.0137,1.52,0.36,0.03\r\n20110714,0.0137,0.51,0.41,-0.15\r\n20110715,0.0137,0.44,-0.15,-0.57\r\n20110718,0.0137,-0.12,0.04,-0.38\r\n20110719,0.0137,-0.83,-0.39,-0.03\r\n20110720,0.0137,-0.05,0.35,0.1\r\n20110721,0.0137,-1.04,-0.13,0.23\r\n20110722,0.0137,0.23,0.32,0.11\r\n20110725,0.0137,-3.17,-0.91,0.79\r\n20110726,0.0137,0.51,0.04,0.06\r\n20110727,0.0137,0.95,0.48,-0.49\r\n20110728,0.0137,-0.45,-0.09,-0.01\r\n20110729,0.0137,-0.48,-0.22,0.38\r\n20110801,0.0127,0.11,0.3,-0.29\r\n20110802,0.0127,-0.82,0.22,-0.14\r\n20110803,0.0127,0.07,0.26,-0.07\r\n20110804,0.0127,0.21,0.24,0.02\r\n20110805,0.0127,-2.07,-0.32,0.36\r\n20110808,0.0127,-3.9,-1.3,0.97\r\n20110809,0.0127,-0.06,-0.32,0.33\r\n20110810,0.0127,1.03,0.55,-0.28\r\n20110811,0.0127,1.38,0.43,0.29\r\n20110812,0.0127,0.58,0.54,-0.63\r\n20110815,0.0127,1.3,0.36,-0.06\r\n20110816,0.0127,-0.75,-0.03,-0.02\r\n20110817,0.0127,-0.3,0.17,0.04\r\n20110818,0.0127,-1.67,-0.1,-0.03\r\n20110819,0.0127,-0.94,0.08,0.08\r\n20110822,0.0127,-0.87,0.58,-0.09\r\n20110823,0.0127,1.53,0.31,-0.07\r\n20110824,0.0127,-0.36,0.2,-0.25\r\n20110825,0.0127,2.69,-0.7,0.44\r\n20110826,0.0127,-0.06,0.38,-0.37\r\n20110829,0.0127,-1.26,0.43,-0.37\r\n20110830,0.0127,-0.52,-0.21,0.4\r\n20110831,0.0127,-0.09,-0.11,0.51\r\n20110901,0.0134,-0.49,-0.03,0.02\r\n20110902,0.0134,-1.1,-0.08,0.15\r\n20110905,0.0134,-2.13,-0.26,0.43\r\n20110906,0.0134,-0.54,-0.01,0.1\r\n20110907,0.0134,1.98,0.6,-0.09\r\n20110908,0.0134,-0.74,-0.31,0.03\r\n20110909,0.0134,-0.22,-0.03,0.17\r\n20110913,0.0134,-1.21,-0.07,0.59\r\n20110914,0.0134,0.61,0.44,-0.5\r\n20110915,0.0134,-0.03,0.09,-0.2\r\n20110916,0.0134,0.12,-0.09,-0.03\r\n20110919,0.0134,-1.83,-0.21,0.15\r\n20110920,0.0134,0.39,0.31,-0.05\r\n20110921,0.0134,2.73,-0.03,-0.28\r\n20110922,0.0134,-2.84,-0.01,0.06\r\n20110923,0.0134,-0.58,-0.09,0.45\r\n20110926,0.0134,-1.65,0.27,0.11\r\n20110927,0.0134,0.76,-0.3,0.43\r\n20110928,0.0134,-1.06,-0.52,0.54\r\n20110929,0.0134,-1.42,-1.75,1.63\r\n20110930,0.0134,-0.22,-0.01,0.3\r\n20111010,0.0182,-0.66,0.32,-0.13\r\n20111011,0.0182,-0.08,0.22,0.3\r\n20111012,0.0182,3.14,0.31,-0.17\r\n20111013,0.0182,0.86,0.44,-0.35\r\n20111014,0.0182,-0.35,-0.01,0.28\r\n20111017,0.0182,0.37,0.03,0.04\r\n20111018,0.0182,-2.5,-0.54,0.55\r\n20111019,0.0182,-0.38,-0.02,0.18\r\n20111020,0.0182,-2.22,-0.75,0.29\r\n20111021,0.0182,-0.83,-0.66,0.57\r\n20111024,0.0182,2.2,-0.58,0.4\r\n20111025,0.0182,1.89,0.52,-0.54\r\n20111026,0.0182,0.82,-0.15,-0.05\r\n20111027,0.0182,0.24,0.06,0.26\r\n20111028,0.0182,1.74,0.36,-0.23\r\n20111031,0.0182,-0.11,0.65,-0.46\r\n20111101,0.0128,0.01,-0.01,-0.08\r\n20111102,0.0128,1.46,0.08,-0.14\r\n20111103,0.0128,0.24,0.36,-0.16\r\n20111104,0.0128,0.75,0.13,-0.01\r\n20111107,0.0128,-0.72,0.44,-0.08\r\n20111108,0.0128,-0.42,-0.58,0.47\r\n20111109,0.0128,1,0.6,-0.33\r\n20111110,0.0128,-1.65,0.33,0.23\r\n20111111,0.0128,-0.03,0.02,0.06\r\n20111114,0.0128,2.02,0.47,-0.03\r\n20111115,0.0128,0.07,0.61,-0.46\r\n20111116,0.0128,-2.52,0.08,0.12\r\n20111117,0.0128,-0.14,0.38,-0.03\r\n20111118,0.0128,-2.14,-0.62,0.56\r\n20111121,0.0128,-0.04,0.16,0.07\r\n20111122,0.0128,-0.09,-0.18,-0.27\r\n20111123,0.0128,-0.66,0.35,0.17\r\n20111124,0.0128,0.08,-0.19,0.24\r\n20111125,0.0128,-0.7,0,0.07\r\n20111128,0.0128,0.12,-0.06,0.21\r\n20111129,0.0128,1.29,0.3,-0.11\r\n20111130,0.0128,-3.36,-0.66,0.91\r\n20111201,0.0133,2.27,-0.2,-0.02\r\n20111202,0.0133,-1.3,-0.58,0.89\r\n20111205,0.0133,-1.62,-1.51,1.23\r\n20111206,0.0133,-0.2,0.25,0.01\r\n20111207,0.0133,0.3,-0.02,-0.14\r\n20111208,0.0133,-0.06,0.11,-0.11\r\n20111209,0.0133,-0.69,-0.2,0.4\r\n20111212,0.0133,-1.09,-0.14,0.28\r\n20111213,0.0133,-2.11,-1.55,1.22\r\n20111214,0.0133,-1.04,-0.92,0.8\r\n20111215,0.0133,-2.16,-0.42,0.65\r\n20111216,0.0133,2.21,0.73,-0.06\r\n20111219,0.0133,-0.2,0.39,-0.16\r\n20111220,0.0133,-0.17,-0.12,0.43\r\n20111221,0.0133,-1.4,-0.93,1.12\r\n20111222,0.0133,-0.34,-1.3,0.81\r\n20111223,0.0133,0.79,0.16,0.08\r\n20111226,0.0133,-0.76,0.3,0.13\r\n20111227,0.0133,-1.41,-1.49,0.65\r\n20111228,0.0133,0.04,-0.96,0.1\r\n20111229,0.0133,0.16,0,0.44\r\n20111230,0.0133,1.35,0.22,-0.3\r\n20120104,0.0195,-1.74,-0.52,0.75\r\n20120105,0.0195,-1.57,-2.34,1.94\r\n20120106,0.0195,0.59,-0.27,0.14\r\n20120109,0.0195,3.05,1.08,-0.63\r\n20120110,0.0195,2.97,0.75,-1.04\r\n20120111,0.0195,-0.38,0.56,-0.42\r\n20120112,0.0195,-0.17,0.04,-0.08\r\n20120113,0.0195,-1.87,-1.91,0.87\r\n20120116,0.0195,-2.1,-0.99,0.88\r\n20120117,0.0195,4.38,0.6,-1.16\r\n20120118,0.0195,-1.74,-0.61,0.19\r\n20120119,0.0195,1.3,-0.51,0.31\r\n20120120,0.0195,1.14,0.08,0.25\r\n20120130,0.0195,-1.35,0.79,-0.52\r\n20120131,0.0195,0.22,-0.04,0.3\r\n20120201,0.013,-1.01,0.53,0.19\r\n20120202,0.013,1.94,-0.03,-0.12\r\n20120203,0.013,0.91,0.95,-0.39\r\n20120206,0.013,0.2,0.87,-0.1\r\n20120207,0.013,-1.71,-0.07,-0.03\r\n20120208,0.013,2.52,-0.1,-0.5\r\n20120209,0.013,0.18,0.64,-0.35\r\n20120210,0.013,0.17,0.43,-0.33\r\n20120213,0.013,0.2,1.09,-0.52\r\n20120214,0.013,-0.26,0.44,0.05\r\n20120215,0.013,1.03,0.54,-0.76\r\n20120216,0.013,-0.44,0.42,-0.15\r\n20120217,0.013,-0.07,-0.36,-0.12\r\n20120220,0.013,0.26,0.07,0.24\r\n20120221,0.013,0.84,0.22,-0.26\r\n20120222,0.013,1.22,0.67,-0.27\r\n20120223,0.013,0.3,0.05,0.24\r\n20120224,0.013,1.33,-0.09,-0.12\r\n20120227,0.013,0.3,-0.3,-0.23\r\n20120228,0.013,-0.02,-0.76,0.29\r\n20120229,0.013,-1.06,-0.23,0.32\r\n20120301,0.0133,0.01,0.74,-0.25\r\n20120302,0.0133,1.62,0.18,-0.44\r\n20120305,0.0133,-0.51,0.89,-0.4\r\n20120306,0.0133,-1.33,0.47,-0.49\r\n20120307,0.0133,-0.61,0.25,0.08\r\n20120308,0.0133,1.14,0.2,-0.36\r\n20120309,0.0133,1.02,0.35,0.09\r\n20120312,0.0133,-0.09,0.49,-0.23\r\n20120313,0.0133,0.91,0.09,-0.04\r\n20120314,0.0133,-3.01,-1.41,1.49\r\n20120315,0.0133,-0.74,-0.35,0.92\r\n20120316,0.0133,1.56,0.9,-0.54\r\n20120319,0.0133,0.42,1.1,-0.6\r\n20120320,0.0133,-1.61,-0.78,-0.01\r\n20120321,0.0133,0.03,0.2,-0.41\r\n20120322,0.0133,-0.17,0.04,0.05\r\n20120323,0.0133,-1.3,-0.85,0.16\r\n20120326,0.0133,0.02,-0.12,0.27\r\n20120327,0.0133,-0.28,0.19,0.19\r\n20120328,0.0133,-2.98,-1.68,1.05\r\n20120329,0.0133,-1.47,-0.66,0.54\r\n20120330,0.0133,0.27,-0.75,0.13\r\n20120405,0.0166,2.09,0.65,-0.98\r\n20120406,0.0166,0.26,0.63,-0.48\r\n20120409,0.0166,-0.95,-0.15,0.16\r\n20120410,0.0166,0.87,0.15,-0.4\r\n20120411,0.0166,0.13,0.55,-0.18\r\n20120412,0.0166,1.84,0.33,0.07\r\n20120413,0.0166,0.38,0.31,-0.37\r\n20120416,0.0166,-0.15,0.46,-0.19\r\n20120417,0.0166,-1.06,-0.34,0.28\r\n20120418,0.0166,1.99,0.18,-0.3\r\n20120419,0.0166,-0.21,0.16,0.17\r\n20120420,0.0166,1.05,-0.3,-0.17\r\n20120423,0.0166,-0.99,-0.71,0.15\r\n20120424,0.0166,-0.19,-0.62,0.45\r\n20120425,0.0166,0.77,0.83,-0.41\r\n20120426,0.0166,-0.1,-0.66,-0.13\r\n20120427,0.0166,-0.33,-0.4,0.69\r\n20120502,0.0133,1.84,-0.8,0.59\r\n20120503,0.0133,0.23,0.15,0.29\r\n20120504,0.0133,0.64,0.07,-0.41\r\n20120507,0.0133,0.25,0.55,-0.02\r\n20120508,0.0133,-0.18,0.35,-0.36\r\n20120509,0.0133,-1.71,-0.03,-0.24\r\n20120510,0.0133,0.14,0.03,-0.24\r\n20120511,0.0133,-0.63,0.04,0.14\r\n20120514,0.0133,-0.59,0.35,-0.05\r\n20120515,0.0133,-0.19,0.01,0.25\r\n20120516,0.0133,-1.3,0.38,0.03\r\n20120517,0.0133,1.36,0.01,0.1\r\n20120518,0.0133,-1.48,-0.14,0.13\r\n20120521,0.0133,0.21,-0.41,0.11\r\n20120522,0.0133,1.19,0.08,-0.03\r\n20120523,0.0133,-0.32,-0.12,-0.06\r\n20120524,0.0133,-0.66,-0.05,-0.01\r\n20120525,0.0133,-0.81,-0.48,-0.37\r\n20120528,0.0133,1.35,-0.01,0.18\r\n20120529,0.0133,1.26,0.02,-0.53\r\n20120530,0.0133,-0.15,0.38,-0.05\r\n20120531,0.0133,-0.29,0.41,-0.15\r\n20120601,0.0134,0.01,-0.07,0.36\r\n20120604,0.0134,-2.78,-0.69,0.58\r\n20120605,0.0134,0.01,-0.02,0.07\r\n20120606,0.0134,-0.13,-0.16,0.15\r\n20120607,0.0134,-0.51,-0.02,0.12\r\n20120608,0.0134,-0.42,0.05,0.02\r\n20120611,0.0134,1.42,0.17,-0.2\r\n20120612,0.0134,-0.61,0.11,0.03\r\n20120613,0.0134,1.54,0.21,0.12\r\n20120614,0.0134,-0.61,0.13,0.35\r\n20120615,0.0134,0.32,-0.01,0.44\r\n20120618,0.0134,0.62,0.25,-0.04\r\n20120619,0.0134,-0.72,0.21,0.04\r\n20120620,0.0134,-0.17,0.18,0.24\r\n20120621,0.0134,-1.42,0.06,0.31\r\n20120625,0.0134,-1.94,-0.09,1.08\r\n20120626,0.0134,-0.03,-0.06,-0.01\r\n20120627,0.0134,-0.2,0.04,0.36\r\n20120628,0.0134,-0.97,-0.36,0.78\r\n20120629,0.0134,1.36,-0.21,0.23\r\n20120702,0.0116,0.34,0.62,-0.39\r\n20120703,0.0116,0.2,0.32,0.24\r\n20120704,0.0116,-0.19,-0.03,0.14\r\n20120705,0.0116,-1.3,-0.85,0.83\r\n20120706,0.0116,1.39,-0.01,-0.36\r\n20120709,0.0116,-2.3,-0.63,0.08\r\n20120710,0.0116,-0.46,-0.52,0.44\r\n20120711,0.0116,0.62,0.11,0.01\r\n20120712,0.0116,0.81,0.39,-0.33\r\n20120713,0.0116,-0.1,-0.43,0.36\r\n20120716,0.0116,-2.15,-1.64,1.46\r\n20120717,0.0116,0.56,0.22,-0.03\r\n20120718,0.0116,0.29,0.7,-0.1\r\n20120719,0.0116,0.73,0.55,-0.16\r\n20120720,0.0116,-0.78,0.49,-0.01\r\n20120723,0.0116,-1.21,0.18,-0.11\r\n20120724,0.0116,0.39,0.4,-0.26\r\n20120725,0.0116,-0.54,-0.21,0.16\r\n20120726,0.0116,-0.57,-0.29,0.17\r\n20120727,0.0116,-0.05,-0.26,0.28\r\n20120730,0.0116,-0.98,-1.03,1.45\r\n20120731,0.0116,-0.52,-1.17,1.38\r\n20120801,0.0109,1.11,0.38,-0.3\r\n20120802,0.0109,-0.66,-0.01,0.05\r\n20120803,0.0109,1.23,0.89,-0.28\r\n20120806,0.0109,1.34,0.81,-0.83\r\n20120807,0.0109,0.28,0.71,-0.88\r\n20120808,0.0109,0.03,-0.19,0.15\r\n20120809,0.0109,0.86,0.45,-0.08\r\n20120810,0.0109,-0.3,0.27,-0.26\r\n20120813,0.0109,-1.65,-0.24,0.43\r\n20120814,0.0109,0.38,0.93,-0.01\r\n20120815,0.0109,-1.01,-0.2,-0.09\r\n20120816,0.0109,-0.47,-0.24,0.18\r\n20120817,0.0109,-0.05,0.37,-0.01\r\n20120820,0.0109,-0.24,0.66,-0.09\r\n20120821,0.0109,0.71,0.89,-0.42\r\n20120822,0.0109,-0.61,0.3,0.11\r\n20120823,0.0109,0.25,0.66,-0.4\r\n20120824,0.0109,-1.25,-0.8,0.41\r\n20120827,0.0109,-1.86,0.12,0.67\r\n20120828,0.0109,0.68,0.36,0.1\r\n20120829,0.0109,-0.91,0.69,0.1\r\n20120830,0.0109,-0.51,-1.12,0.65\r\n20120831,0.0109,-0.14,0.62,-0.09\r\n20120903,0.0122,0.98,0.61,-0.08\r\n20120904,0.0122,-0.86,0.18,0.18\r\n20120905,0.0122,-0.11,0.79,-0.33\r\n20120906,0.0122,0.76,0.35,-0.12\r\n20120907,0.0122,3.78,-0.66,-0.3\r\n20120910,0.0122,0.48,0.57,-0.5\r\n20120911,0.0122,-0.59,0.21,-0.39\r\n20120912,0.0122,0.33,0.25,-0.46\r\n20120913,0.0122,-0.93,-0.42,0.11\r\n20120914,0.0122,0.51,-0.51,-0.03\r\n20120917,0.0122,-2.3,-0.61,0.36\r\n20120918,0.0122,-0.74,0.54,0.32\r\n20120919,0.0122,0.46,0.28,-0.38\r\n20120920,0.0122,-2.3,-0.91,0.72\r\n20120921,0.0122,-0.11,-0.8,0.06\r\n20120924,0.0122,0.44,0.01,-0.12\r\n20120925,0.0122,-0.28,-0.35,0.5\r\n20120926,0.0122,-1.5,-1.06,0.56\r\n20120927,0.0122,2.63,-0.5,0.3\r\n20120928,0.0122,1.59,0.02,-0.31\r\n20121008,0.014,-0.61,0.61,-0.35\r\n20121009,0.014,2.14,0.55,-0.15\r\n20121010,0.014,0.38,0.57,-0.93\r\n20121011,0.014,-0.99,-0.56,0.6\r\n20121012,0.014,0,0.12,0.21\r\n20121015,0.014,-0.4,-0.37,0.49\r\n20121016,0.014,0.07,0.13,-0.1\r\n20121017,0.014,0.25,0.19,0.1\r\n20121018,0.014,1.38,0.31,0.01\r\n20121019,0.014,-0.18,-0.02,0.08\r\n20121022,0.014,0.27,0.29,0.04\r\n20121023,0.014,-1.06,0.23,-0.36\r\n20121024,0.014,-0.09,-0.03,-0.46\r\n20121025,0.014,-0.76,-0.38,0.18\r\n20121026,0.014,-1.79,-0.13,0.58\r\n20121029,0.014,-0.31,0.21,0.24\r\n20121030,0.014,0.17,-0.1,0.4\r\n20121031,0.014,0.36,0,0\r\n20121101,0.0111,1.77,-0.18,-0.4\r\n20121102,0.0111,0.49,0.11,-0.07\r\n20121105,0.0111,-0.26,0.06,-0.03\r\n20121106,0.0111,-0.46,-0.41,-0.05\r\n20121107,0.0111,-0.08,-0.04,0.03\r\n20121108,0.0111,-1.82,-0.64,1.03\r\n20121109,0.0111,-0.24,0.11,0.16\r\n20121112,0.0111,0.45,0.45,-0.06\r\n20121113,0.0111,-1.65,-0.41,0.32\r\n20121114,0.0111,0.33,-0.4,-0.34\r\n20121115,0.0111,-1.3,-0.7,0.33\r\n20121116,0.0111,-0.77,0.38,0.44\r\n20121119,0.0111,0.04,0.53,-0.17\r\n20121120,0.0111,-0.37,0.18,0.09\r\n20121121,0.0111,1.05,-0.07,-0.22\r\n20121122,0.0111,-0.88,-0.25,0.11\r\n20121123,0.0111,0.51,-0.29,-0.2\r\n20121126,0.0111,-0.77,-0.32,0.31\r\n20121127,0.0111,-1.67,-2.28,1.43\r\n20121128,0.0111,-1.14,-1.03,1.13\r\n20121129,0.0111,-0.63,-0.35,0.48\r\n20121130,0.0111,0.95,0.33,-0.49\r\n20121203,0.012,-1.39,-1.16,0.6\r\n20121204,0.012,0.91,0.19,-0.41\r\n20121205,0.012,3.13,0.42,-0.63\r\n20121206,0.012,-0.18,0.2,0.09\r\n20121207,0.012,1.71,0.78,-0.35\r\n20121210,0.012,1.15,0.73,-0.37\r\n20121211,0.012,-0.49,-0.39,-0.1\r\n20121212,0.012,0.26,-0.18,0.28\r\n20121213,0.012,-1.08,-0.12,0.45\r\n20121214,0.012,4.3,-0.37,-0.22\r\n20121217,0.012,0.42,0.18,-0.66\r\n20121218,0.012,0,-0.28,0.09\r\n20121219,0.012,0.16,0.34,0.17\r\n20121220,0.012,0.35,-0.23,-0.13\r\n20121221,0.012,-0.49,0.48,0.28\r\n20121224,0.012,0.35,0.28,0.22\r\n20121225,0.012,2.51,-0.05,0.35\r\n20121226,0.012,0.52,1.16,-0.15\r\n20121227,0.012,-0.61,-0.53,-0.15\r\n20121228,0.012,1.21,-0.11,0.19\r\n20121231,0.012,1.45,-0.83,-0.37\r\n20130104,0.0126,0.09,-0.41,0.17\r\n20130107,0.0126,0.43,1,0.02\r\n20130108,0.0126,-0.05,1.57,-0.16\r\n20130109,0.0126,0.09,0.02,-0.19\r\n20130110,0.0126,0.39,0.32,-0.5\r\n20130111,0.0126,-1.83,-0.11,0.33\r\n20130114,0.0126,3.26,0.22,-0.12\r\n20130115,0.0126,0.84,0.94,-0.62\r\n20130116,0.0126,-0.64,0.51,-0.5\r\n20130117,0.0126,-1.04,-0.12,0.57\r\n20130118,0.0126,1.45,0.2,0.3\r\n20130121,0.0126,0.45,0.65,0.13\r\n20130122,0.0126,-0.84,-0.93,0.24\r\n20130123,0.0126,0.19,-0.04,-0.34\r\n20130124,0.0126,-1.12,-1.08,0.11\r\n20130125,0.0126,-0.46,0.37,0.6\r\n20130128,0.0126,2.46,0.02,0.18\r\n20130129,0.0126,0.61,0.18,0.09\r\n20130130,0.0126,0.79,-0.62,0.16\r\n20130131,0.0126,-0.11,-0.46,0.22\r\n20130201,0.0151,1.3,-0.8,0.15\r\n20130204,0.0151,0.03,-0.72,0.14\r\n20130205,0.0151,0.48,0.44,0.16\r\n20130206,0.0151,0.15,0.26,-0.11\r\n20130207,0.0151,-0.33,0.32,-0.33\r\n20130208,0.0151,0.78,0.54,0.15\r\n20130218,0.0151,-0.37,1.55,-0.2\r\n20130219,0.0151,-1.73,0.35,-0.03\r\n20130220,0.0151,0.94,0.76,-0.1\r\n20130221,0.0151,-2.78,0.98,0.31\r\n20130222,0.0151,-0.45,0.03,-0.34\r\n20130225,0.0151,0.6,0.68,0.25\r\n20130226,0.0151,-1.42,0.07,0.31\r\n20130227,0.0151,0.67,-0.74,-0.15\r\n20130228,0.0151,2.23,-0.49,-0.17\r\n20130301,0.012,0.09,1.22,-0.36\r\n20130304,0.012,-3.71,0.55,-0.05\r\n20130305,0.012,2.34,-0.14,0.3\r\n20130306,0.012,1.04,0.93,0.04\r\n20130307,0.012,-1.05,0.14,-0.14\r\n20130308,0.012,-0.52,-0.2,0.03\r\n20130311,0.012,-0.28,0.11,0.19\r\n20130312,0.012,-1.44,-1.33,0.33\r\n20130313,0.012,-0.99,0.12,-0.09\r\n20130314,0.012,0.32,0.03,0.02\r\n20130315,0.012,0.35,-0.26,0.33\r\n20130318,0.012,-1.56,0.25,0.07\r\n20130319,0.012,0.6,-0.23,0.09\r\n20130320,0.012,2.7,-0.22,0.06\r\n20130321,0.012,0.44,0.63,-0.4\r\n20130322,0.012,0.16,0.08,0.07\r\n20130325,0.012,-0.11,0.13,-0.23\r\n20130326,0.012,-1.11,0.53,0.18\r\n20130327,0.012,0.15,-0.05,-0.45\r\n20130328,0.012,-2.77,-0.32,0.15\r\n20130329,0.012,-0.07,0.4,0.26\r\n20130401,0.0135,0.09,0.71,0\r\n20130402,0.0135,-0.64,-0.5,-0.07\r\n20130403,0.0135,-0.33,-0.6,0.22\r\n20130408,0.0135,-0.26,0.39,-0.07\r\n20130409,0.0135,0.67,0.31,-0.04\r\n20130410,0.0135,-0.12,-0.21,-0.09\r\n20130411,0.0135,-0.31,0.07,0.2\r\n20130412,0.0135,-0.62,-0.27,0.27\r\n20130415,0.0135,-1.17,-0.12,0.58\r\n20130416,0.0135,0.74,0.16,0.12\r\n20130417,0.0135,0.28,0.3,-0.16\r\n20130418,0.0135,0.2,0.06,0.14\r\n20130419,0.0135,2.18,-0.31,0.03\r\n20130422,0.0135,0.09,0.37,-0.49\r\n20130423,0.0135,-2.66,-0.19,0.12\r\n20130424,0.0135,1.75,0.23,-0.23\r\n20130425,0.0135,-1.14,-0.34,0.47\r\n20130426,0.0135,-1.15,-0.64,0.33\r\n20130502,0.0114,0.09,0.57,0.26\r\n20130503,0.0114,1.64,0.27,-0.36\r\n20130506,0.0114,1.38,0.26,-0.7\r\n20130507,0.0114,0.16,0.04,-0.21\r\n20130508,0.0114,0.65,0.28,-0.09\r\n20130509,0.0114,-0.37,0.31,-0.57\r\n20130510,0.0114,0.51,0.47,0.13\r\n20130513,0.0114,-0.1,0.43,-0.54\r\n20130514,0.0114,-1.23,0.07,-0.33\r\n20130515,0.0114,0.71,0.45,-0.23\r\n20130516,0.0114,1.21,-0.36,0.24\r\n20130517,0.0114,1.49,-0.09,0.02\r\n20130520,0.0114,0.94,0.79,-0.34\r\n20130521,0.0114,0.55,0.62,-0.72\r\n20130522,0.0114,-0.35,-0.49,0.3\r\n20130523,0.0114,-1.03,0.23,-0.21\r\n20130524,0.0114,0.82,0.5,-0.6\r\n20130527,0.0114,0.36,0.53,-0.11\r\n20130528,0.0114,0.86,-0.93,0.91\r\n20130529,0.0114,0.23,0.34,-0.4\r\n20130530,0.0114,-0.19,0.33,-0.02\r\n20130531,0.0114,-0.77,0.37,0.05\r\n20130603,0.0143,-0.26,-0.09,0.5\r\n20130604,0.0143,-1.45,-0.34,0.41\r\n20130605,0.0143,0.14,0.28,-0.17\r\n20130606,0.0143,-1.21,-0.41,0.45\r\n20130607,0.0143,-1.44,-0.05,0.11\r\n20130613,0.0143,-2.82,-0.2,0.26\r\n20130614,0.0143,1.11,0.43,-0.82\r\n20130617,0.0143,-0.13,0.33,-0.43\r\n20130618,0.0143,0.31,0.06,-0.04\r\n20130619,0.0143,-0.62,-0.02,-0.09\r\n20130620,0.0143,-3.07,0.14,0.77\r\n20130621,0.0143,-0.44,-0.19,0.1\r\n20130624,0.0143,-5.68,-0.21,0.84\r\n20130625,0.0143,-0.18,0,0.4\r\n20130626,0.0143,0.77,0.48,-1.33\r\n20130627,0.0143,-0.52,-0.95,1.34\r\n20130628,0.0143,1.38,-1.17,1.76\r\n20130701,0.0109,1.3,0.16,-1.11\r\n20130702,0.0109,0.99,0.21,-0.96\r\n20130703,0.0109,-0.34,-0.15,-0.91\r\n20130704,0.0109,0.61,0.11,-0.01\r\n20130705,0.0109,-0.16,-0.01,0.7\r\n20130708,0.0109,-2.7,-1.15,1.09\r\n20130709,0.0109,0.36,0.42,-0.09\r\n20130710,0.0109,2.42,0.46,-0.4\r\n20130711,0.0109,3.3,-0.54,0.82\r\n20130712,0.0109,-1.34,0.49,-0.82\r\n20130715,0.0109,1.42,0.36,-0.97\r\n20130716,0.0109,0.52,0.58,-0.57\r\n20130717,0.0109,-1.2,0.04,0.03\r\n20130718,0.0109,-0.91,0.54,-0.77\r\n20130719,0.0109,-1.75,-0.57,0.51\r\n20130722,0.0109,1.01,0.74,-0.86\r\n20130723,0.0109,2.25,0.07,-0.27\r\n20130724,0.0109,-0.15,0.42,-0.88\r\n20130725,0.0109,-1.02,-0.52,1.28\r\n20130726,0.0109,-0.47,0.62,-0.16\r\n20130729,0.0109,-1.88,0.3,0.29\r\n20130730,0.0109,0.37,0.01,1.16\r\n20130731,0.0109,0.25,0.14,-0.05\r\n20130801,0.0114,2.24,0.22,-0.92\r\n20130802,0.0114,0.1,0.39,-0.3\r\n20130805,0.0114,1.27,0.23,-0.06\r\n20130806,0.0114,0.67,0.19,-0.14\r\n20130807,0.0114,-0.85,-0.02,0.42\r\n20130808,0.0114,0.04,0.28,-0.18\r\n20130809,0.0114,0.26,0.19,0.29\r\n20130812,0.0114,2.07,-0.3,0.83\r\n20130813,0.0114,0.31,0.4,-0.32\r\n20130814,0.0114,-0.31,-0.05,-0.02\r\n20130815,0.0114,-1.03,0.09,-0.14\r\n20130816,0.0114,-1.09,-0.39,1.29\r\n20130819,0.0114,1.16,0.59,-0.73\r\n20130820,0.0114,-0.5,0.58,-0.44\r\n20130821,0.0114,0.15,0.49,-0.35\r\n20130822,0.0114,-0.09,0.1,-0.29\r\n20130823,0.0114,-0.39,-0.24,-0.34\r\n20130826,0.0114,1.9,0.31,0.28\r\n20130827,0.0114,0.37,0.7,-0.14\r\n20130828,0.0114,-0.51,-0.19,0.13\r\n20130829,0.0114,-0.25,0.6,0.46\r\n20130830,0.0114,-0.55,-0.15,1.49\r\n20130902,0.0128,0.4,-0.04,-0.71\r\n20130903,0.0128,1.34,0.21,-0.08\r\n20130904,0.0128,0.11,0.02,-0.33\r\n20130905,0.0128,-0.15,0.06,-0.63\r\n20130906,0.0128,0.68,-0.16,-0.33\r\n20130909,0.0128,2.63,-1.49,1.57\r\n20130910,0.0128,0.93,-0.11,-0.09\r\n20130911,0.0128,-0.26,-0.62,0.32\r\n20130912,0.0128,0.54,-0.15,0.68\r\n20130913,0.0128,-0.45,0.36,-0.37\r\n20130916,0.0128,-0.08,0.48,-0.16\r\n20130917,0.0128,-1.87,0.04,-0.19\r\n20130918,0.0128,0.36,0.46,-0.33\r\n20130923,0.0128,1.64,0.07,-0.71\r\n20130924,0.0128,-0.42,0.73,-0.3\r\n20130925,0.0128,-0.51,-0.09,-0.14\r\n20130926,0.0128,-1.75,-0.22,-0.11\r\n20130927,0.0128,0.31,-0.5,-0.15\r\n20130930,0.0128,0.81,0.29,-0.64\r\n20131008,0.014,1.26,0.48,-0.33\r\n20131009,0.014,0.68,0.46,-0.59\r\n20131010,0.014,-0.99,0.51,0.22\r\n20131011,0.014,1.46,0.38,0.79\r\n20131014,0.014,0.56,0.54,-0.39\r\n20131015,0.014,-0.25,0.26,0.11\r\n20131016,0.014,-2.11,-0.4,1.41\r\n20131017,0.014,-0.17,0.49,-0.1\r\n20131018,0.014,0.34,0,-0.56\r\n20131021,0.014,1.92,0.32,-0.45\r\n20131022,0.014,-1.01,0.76,0.6\r\n20131023,0.014,-1.55,-0.81,1.07\r\n20131024,0.014,-0.85,0.27,0.66\r\n20131025,0.014,-1.59,0.09,0.83\r\n20131028,0.014,-0.07,-0.3,-0.15\r\n20131029,0.014,-0.63,-2.51,1.49\r\n20131030,0.014,1.51,0.51,0.25\r\n20131031,0.014,-1.09,0.38,0.69\r\n20131101,0.0116,0.14,-0.39,0.6\r\n20131104,0.0116,0.11,0.59,-0.33\r\n20131105,0.0116,0.63,0.42,-1.03\r\n20131106,0.0116,-0.95,0.38,0.03\r\n20131107,0.0116,-0.74,-0.36,0.4\r\n20131108,0.0116,-1.2,-0.22,0.25\r\n20131111,0.0116,0.3,0.6,0.38\r\n20131112,0.0116,0.85,0.1,-0.06\r\n20131113,0.0116,-1.85,-0.16,-0.34\r\n20131114,0.0116,0.86,0.12,-0.79\r\n20131115,0.0116,1.75,0.18,-0.3\r\n20131118,0.0116,2.71,-0.24,-0.12\r\n20131119,0.0116,-0.14,0.62,-0.19\r\n20131120,0.0116,0.61,0.49,-0.4\r\n20131121,0.0116,-0.18,-0.27,-0.27\r\n20131122,0.0116,-0.44,0.15,0.09\r\n20131125,0.0116,-0.39,0.58,0.07\r\n20131126,0.0116,-0.07,0.5,-0.03\r\n20131127,0.0116,1.03,-0.25,-0.68\r\n20131128,0.0116,0.88,0.09,0.54\r\n20131129,0.0116,0.27,0.57,-0.49\r\n20131202,0.0114,-1.95,-2.7,3.58\r\n20131203,0.0114,1.02,0.84,0.03\r\n20131204,0.0114,1.22,0.22,0.09\r\n20131205,0.0114,-0.4,0.48,0.61\r\n20131206,0.0114,-0.39,0.27,0.05\r\n20131209,0.0114,0.34,0.57,-0.37\r\n20131210,0.0114,-0.07,0.26,0.1\r\n20131211,0.0114,-1.35,0.05,-0.44\r\n20131212,0.0114,0.15,0.14,-0.57\r\n20131213,0.0114,-0.12,0.58,0.07\r\n20131216,0.0114,-1.54,-0.03,0.04\r\n20131217,0.0114,-0.54,-0.2,0.05\r\n20131218,0.0114,-0.13,0.18,0.23\r\n20131219,0.0114,-0.86,0.22,0.19\r\n20131220,0.0114,-1.88,0.42,-0.81\r\n20131223,0.0114,0.2,-0.29,0.76\r\n20131224,0.0114,0.35,0.42,-0.12\r\n20131225,0.0114,0.92,0.54,-0.08\r\n20131226,0.0114,-1.67,-0.14,0.13\r\n20131227,0.0114,1.47,0.32,0.07\r\n20131230,0.0114,-0.07,0.56,-0.38\r\n20131231,0.0114,0.74,-0.72,0.28\r\n20140102,0.012,0.06,0.62,-0.54\r\n20140103,0.012,-0.96,0.04,-0.85\r\n20140106,0.012,-2.07,-0.69,0.67\r\n20140107,0.012,0.24,0.25,-0.58\r\n20140108,0.012,0.22,-0.22,-0.81\r\n20140109,0.012,-1,-0.68,-0.02\r\n20140110,0.012,-1.04,-0.55,0.45\r\n20140113,0.012,-0.23,0.07,0.21\r\n20140114,0.012,1.21,0.87,-0.25\r\n20140115,0.012,0.11,0.32,-0.73\r\n20140116,0.012,-0.09,0.29,0.4\r\n20140117,0.012,-0.96,0.01,0.03\r\n20140120,0.012,-0.79,0.23,-0.04\r\n20140121,0.012,1.07,0.49,-0.19\r\n20140122,0.012,2.26,-0.03,-0.11\r\n20140123,0.012,-0.07,0.49,-1.01\r\n20140124,0.012,0.85,0.58,-0.32\r\n20140127,0.012,-0.91,0.51,-0.69\r\n20140128,0.012,0.04,-0.12,0.27\r\n20140129,0.012,0.64,0.45,-0.41\r\n20140130,0.012,-0.82,0.32,-0.34\r\n20140207,0.0142,0.93,0.73,-1.31\r\n20140210,0.0142,2.26,0.46,-0.4\r\n20140211,0.0142,0.49,0.25,0.57\r\n20140212,0.0142,0.5,1.09,-0.29\r\n20140213,0.0142,-0.97,-0.3,0.98\r\n20140214,0.0142,1.08,0.96,-0.22\r\n20140217,0.0142,1.25,0.68,-1.15\r\n20140218,0.0142,-0.76,0.84,-0.65\r\n20140219,0.0142,0.81,-0.55,0.57\r\n20140220,0.0142,-0.69,-0.53,0.78\r\n20140221,0.0142,-0.94,0.3,-0.9\r\n20140224,0.0142,-1.28,1.16,-1.41\r\n20140225,0.0142,-2.72,-1.17,1.39\r\n20140226,0.0142,0.39,0.53,-0.15\r\n20140227,0.0142,-0.45,-0.27,1.16\r\n20140228,0.0142,0.73,0.15,-0.16\r\n20140303,0.012,1.13,1.13,-0.68\r\n20140304,0.012,-0.25,0.33,-0.3\r\n20140305,0.012,-0.72,0.62,-0.29\r\n20140306,0.012,0.2,0.43,0.66\r\n20140307,0.012,-0.1,-0.35,-0.16\r\n20140310,0.012,-3.14,-0.57,0.77\r\n20140311,0.012,0.03,0.16,0.78\r\n20140312,0.012,-0.19,0.12,0.6\r\n20140313,0.012,1.27,-0.07,-0.36\r\n20140314,0.012,-0.67,0.22,-0.4\r\n20140317,0.012,1.34,0.64,-0.62\r\n20140318,0.012,0.11,0.23,-0.36\r\n20140319,0.012,-0.37,0.64,0.12\r\n20140320,0.012,-1.86,-0.44,0.96\r\n20140321,0.012,2.48,-0.26,0.83\r\n20140324,0.012,0.65,0.21,0.6\r\n20140325,0.012,-0.07,-0.09,-0.19\r\n20140326,0.012,0.04,0.23,-0.37\r\n20140327,0.012,-1.2,-0.15,1.11\r\n20140328,0.012,-0.88,-0.87,1.8\r\n20140331,0.012,-0.4,-0.25,0.49\r\n20140401,0.0116,1.03,0.73,-0.31\r\n20140402,0.0116,0.31,-0.38,0.37\r\n20140403,0.0116,-0.53,0.36,-0.36\r\n20140404,0.0116,0.92,0.2,-0.33\r\n20140408,0.0116,1.64,-0.78,0.58\r\n20140409,0.0116,0.49,0.31,-0.49\r\n20140410,0.0116,1.09,-0.94,-0.06\r\n20140411,0.0116,-0.28,-0.07,0.48\r\n20140414,0.0116,0.16,0.69,-0.04\r\n20140415,0.0116,-1.28,0.4,-0.48\r\n20140416,0.0116,0.07,0.28,0.42\r\n20140417,0.0116,-0.17,0.13,-0.06\r\n20140418,0.0116,0.03,0.05,-0.41\r\n20140421,0.0116,-1.52,-0.24,0.01\r\n20140422,0.0116,-0.01,-0.61,0.58\r\n20140423,0.0116,-0.23,0.08,0.2\r\n20140424,0.0116,-0.57,-0.37,0.37\r\n20140425,0.0116,-1.31,-0.86,0.67\r\n20140428,0.0116,-1.95,-1.2,1.31\r\n20140429,0.0116,0.96,0.31,0.01\r\n20140430,0.0116,0.16,0.55,-0.56\r\n20140505,0.0126,0.28,0.71,-0.61\r\n20140506,0.0126,0.23,0.26,-0.69\r\n20140507,0.0126,-1.2,-0.23,0.92\r\n20140508,0.0126,0.11,0.17,-0.05\r\n20140509,0.0126,-0.42,-0.59,0.43\r\n20140512,0.0126,2.21,0.08,-0.96\r\n20140513,0.0126,-0.12,0.21,-0.03\r\n20140514,0.0126,-0.08,0.05,0.03\r\n20140515,0.0126,-1.39,-0.38,0.64\r\n20140516,0.0126,-0.05,0.05,0.11\r\n20140519,0.0126,-0.93,0.48,-0.66\r\n20140520,0.0126,0.2,0.28,-0.18\r\n20140521,0.0126,0.92,-0.02,-0.27\r\n20140522,0.0126,-0.07,0.1,-0.03\r\n20140523,0.0126,0.86,0.1,-0.45\r\n20140526,0.0126,0.61,0.31,-0.61\r\n20140527,0.0126,-0.38,0.02,0.05\r\n20140528,0.0126,1.01,-0.04,-0.76\r\n20140529,0.0126,-0.74,-0.32,0.49\r\n20140530,0.0126,0.07,0.02,-0.64\r\n20140603,0.0122,-0.02,-0.02,0.2\r\n20140604,0.0122,-0.78,-0.13,-0.17\r\n20140605,0.0122,1.13,-0.06,-0.25\r\n20140606,0.0122,-0.49,0.17,-0.08\r\n20140609,0.0122,-0.21,-0.26,0.42\r\n20140610,0.0122,1.25,0.11,-0.08\r\n20140611,0.0122,0.22,0.28,-0.24\r\n20140612,0.0122,-0.22,0.15,-0.32\r\n20140613,0.0122,1.07,0.05,-0.24\r\n20140616,0.0122,0.71,-0.03,-0.03\r\n20140617,0.0122,-0.97,0.12,0.04\r\n20140618,0.0122,-0.58,-0.38,-0.39\r\n20140619,0.0122,-1.93,-0.57,1.04\r\n20140620,0.0122,0.71,0.45,-0.3\r\n20140623,0.0122,0.22,0.61,-0.99\r\n20140624,0.0122,0.59,0.22,-0.14\r\n20140625,0.0122,-0.41,0.14,0.26\r\n20140626,0.0122,0.89,0.66,-0.42\r\n20140627,0.0122,0.29,0.55,-0.17\r\n20140630,0.0122,0.71,0.27,-0.3\r\n20140701,0.0109,0.18,0.3,-0.52\r\n20140702,0.0109,0.48,0.03,-0.12\r\n20140703,0.0109,0.59,0.09,-0.13\r\n20140704,0.0109,-0.16,0.12,0.23\r\n20140707,0.0109,-0.05,0.32,0.15\r\n20140708,0.0109,0.38,0.37,-0.18\r\n20140709,0.0109,-1.41,-0.12,0.06\r\n20140710,0.0109,-0.09,0.68,0.44\r\n20140711,0.0109,0.61,0.37,-0.47\r\n20140714,0.0109,1.12,0.58,0.23\r\n20140715,0.0109,0,-0.09,0.42\r\n20140716,0.0109,-0.43,-0.47,0.78\r\n20140717,0.0109,-0.53,0.24,0.14\r\n20140718,0.0109,0.27,-0.01,0.51\r\n20140721,0.0109,-0.15,-0.27,-0.09\r\n20140722,0.0109,1.15,-0.03,-0.61\r\n20140723,0.0109,-0.23,-0.46,0.57\r\n20140724,0.0109,0.89,-0.82,1.49\r\n20140725,0.0109,1.05,-0.21,-0.2\r\n20140728,0.0109,2.36,-0.56,0.08\r\n20140729,0.0109,0.47,0.48,-0.62\r\n20140730,0.0109,0.05,0.31,-0.6\r\n20140731,0.0109,0.93,-0.01,0.14\r\n20140801,0.012,-0.88,-0.33,0.58\r\n20140804,0.012,1.66,-0.05,-0.07\r\n20140805,0.012,0.08,0.48,-0.58\r\n20140806,0.012,0,-0.07,-0.9\r\n20140807,0.012,-1.28,0.17,-0.54\r\n20140808,0.012,0.41,0.42,-0.28\r\n20140811,0.012,1.43,0.1,-0.06\r\n20140812,0.012,-0.03,0.59,-0.56\r\n20140813,0.012,-0.02,0.19,0.15\r\n20140814,0.012,-0.75,0.44,0.01\r\n20140815,0.012,0.99,0.23,0.04\r\n20140818,0.012,0.83,0.37,-0.64\r\n20140819,0.012,0.27,-0.05,-0.36\r\n20140820,0.012,-0.28,0.26,0.41\r\n20140821,0.012,-0.29,0.11,-0.44\r\n20140822,0.012,0.52,0.28,0.02\r\n20140825,0.012,-0.68,-0.09,0.1\r\n20140826,0.012,-1.25,-0.64,0.61\r\n20140827,0.012,0.22,0.18,-0.46\r\n20140828,0.012,-0.74,-0.06,0.18\r\n20140829,0.012,1.04,0.05,-0.16\r\n20140901,0.0116,1.12,0.66,-0.67\r\n20140902,0.0116,1.36,0.22,-0.32\r\n20140903,0.0116,0.92,-0.11,-0.02\r\n20140904,0.0116,0.79,0.32,0.04\r\n20140905,0.0116,0.7,-0.19,0.25\r\n20140909,0.0116,0.09,0.59,-0.47\r\n20140910,0.0116,-0.19,0.68,-0.52\r\n20140911,0.0116,-0.31,0.07,-0.02\r\n20140912,0.0116,0.86,0.38,-0.25\r\n20140915,0.0116,0.38,0.74,-0.38\r\n20140916,0.0116,-2.34,-0.89,1.12\r\n20140917,0.0116,0.51,0.41,-0.23\r\n20140918,0.0116,0.42,0.34,-0.02\r\n20140919,0.0116,0.72,0.12,-0.25\r\n20140922,0.0116,-1.69,0.43,-0.16\r\n20140923,0.0116,0.96,0.32,-0.2\r\n20140924,0.0116,1.51,-0.21,-0.13\r\n20140925,0.0116,-0.02,0.02,-0.36\r\n20140926,0.0116,0.11,0.36,0.03\r\n20140929,0.0116,0.58,0.42,-0.17\r\n20140930,0.0116,0.33,0.25,-0.39\r\n20141008,0.014,0.98,0.42,-0.03\r\n20141009,0.014,0.26,-0.01,-0.25\r\n20141010,0.014,-0.61,0.12,-0.24\r\n20141013,0.014,-0.33,0.37,-0.34\r\n20141014,0.014,-0.34,0.11,-0.03\r\n20141015,0.014,0.52,-0.15,0.19\r\n20141016,0.014,-0.98,-0.25,0.84\r\n20141017,0.014,-0.68,-0.54,0.52\r\n20141020,0.014,0.91,0.67,-0.34\r\n20141021,0.014,-0.83,0,0.38\r\n20141022,0.014,-0.76,-0.52,0.46\r\n20141023,0.014,-1.21,-0.51,0.48\r\n20141024,0.014,-0.04,0.37,0.08\r\n20141027,0.014,-0.24,1.06,-0.71\r\n20141028,0.014,2.17,0.43,-0.44\r\n20141029,0.014,1.4,0.13,0.21\r\n20141030,0.014,0.48,-0.19,0.69\r\n20141031,0.014,0.89,-0.79,1.12\r\n20141103,0.0118,0.47,0.73,-0.11\r\n20141104,0.0118,-0.21,-0.35,0.21\r\n20141105,0.0118,-0.41,0.34,-0.06\r\n20141106,0.0118,0.43,0.35,-0.22\r\n20141107,0.0118,-0.51,-0.45,0.57\r\n20141110,0.0118,1.91,-0.76,1.03\r\n20141111,0.0118,-1.08,-1.95,1.92\r\n20141112,0.0118,1.12,0.42,-0.43\r\n20141113,0.0118,-0.77,-0.41,0.64\r\n20141114,0.0118,-0.15,-0.49,-0.51\r\n20141117,0.0118,0.12,1.06,-0.44\r\n20141118,0.0118,-0.59,0.83,-0.72\r\n20141119,0.0118,0.03,0.69,-0.48\r\n20141120,0.0118,-0.07,0.19,0.42\r\n20141121,0.0118,1.36,-0.1,0.45\r\n20141124,0.0118,1.75,-0.52,0.27\r\n20141125,0.0118,1.43,-0.02,-0.25\r\n20141126,0.0118,1.17,-0.17,0.77\r\n20141127,0.0118,0.95,-0.15,0.16\r\n20141128,0.0118,1.59,-0.79,1.2\r\n20141201,0.01,-0.26,-0.88,0.55\r\n20141202,0.01,2.71,-1.02,1.5\r\n20141203,0.01,0.9,-0.8,-1\r\n20141204,0.01,3.84,-2.06,1.14\r\n20141205,0.01,0.41,-3.23,1.3\r\n20141208,0.01,2.42,-2.07,0.75\r\n20141209,0.01,-5.3,0.63,0.43\r\n20141210,0.01,3.28,0.12,0.17\r\n20141211,0.01,-0.19,1.39,-1.25\r\n20141212,0.01,0.59,0.36,-0.31\r\n20141215,0.01,0.76,0.42,-0.65\r\n20141216,0.01,1.79,-1.88,0.97\r\n20141217,0.01,0.84,-1.39,1.8\r\n20141218,0.01,-0.26,-0.19,-0.85\r\n20141219,0.01,0.93,-2.01,1.13\r\n20141222,0.01,-0.38,-3.11,3.02\r\n20141223,0.01,-2.74,1.53,0.51\r\n20141224,0.01,-1.45,2.81,-1.77\r\n20141225,0.01,2.92,-1.1,1.14\r\n20141226,0.01,2.41,-1.47,1.48\r\n20141229,0.01,-0.08,-0.95,1.76\r\n20141230,0.01,-0.4,-0.73,1.35\r\n20141231,0.01,1.94,-0.94,0.66\r\n20150105,0.0115,3.24,-1.55,1.55\r\n20150106,0.0115,0.44,1.25,-0.4\r\n20150107,0.0115,0.35,-0.59,-0.26\r\n20150108,0.0115,-2.07,1.56,-0.96\r\n20150109,0.0115,-0.41,0.01,0.78\r\n20150112,0.0115,-1.37,0,-0.05\r\n20150113,0.0115,0.5,1.18,-0.7\r\n20150114,0.0115,-0.47,0.18,0.55\r\n20150115,0.0115,2.83,-1.73,1.41\r\n20150116,0.0115,1.17,0,-0.47\r\n20150119,0.0115,-6.87,3.52,-1.38\r\n20150120,0.0115,2.14,1.69,-0.88\r\n20150121,0.0115,4.22,-1.81,1.13\r\n20150122,0.0115,0.7,0.43,-1.16\r\n20150123,0.0115,-0.11,-0.62,0.79\r\n20150126,0.0115,1.18,0.76,-0.94\r\n20150127,0.0115,-0.62,1.05,-0.91\r\n20150128,0.0115,-1.28,0.9,-0.2\r\n20150129,0.0115,-1.2,0.38,-0.61\r\n20150130,0.0115,-1.39,0.32,0.42\r\n20150202,0.0138,-2.09,1.29,-1.13\r\n20150203,0.0138,2.42,-0.67,0.41\r\n20150204,0.0138,-0.83,0.34,-0.81\r\n20150205,0.0138,-0.96,0.44,-0.39\r\n20150206,0.0138,-2.09,0.16,0.52\r\n20150209,0.0138,0.35,-0.74,0.98\r\n20150210,0.0138,1.69,-0.62,0.04\r\n20150211,0.0138,0.74,0.12,-0.68\r\n20150212,0.0138,0.51,0.39,-0.14\r\n20150213,0.0138,1.15,0.3,-0.54\r\n20150216,0.0138,1.01,0.47,-1.21\r\n20150217,0.0138,0.72,-0.01,0.01\r\n20150225,0.0138,-0.67,0.71,0.04\r\n20150226,0.0138,1.87,-1.04,0.58\r\n20150227,0.0138,0.52,0.36,-0.58\r\n20150302,0.0096,1.21,0.75,-0.89\r\n20150303,0.0096,-2.01,1.06,-1.07\r\n20150304,0.0096,0.91,0.83,-0.6\r\n20150305,0.0096,-0.69,1.18,-0.85\r\n20150306,0.0096,-0.72,0.04,0.74\r\n20150309,0.0096,1.77,0.07,0.38\r\n20150310,0.0096,-0.04,1.05,-0.93\r\n20150311,0.0096,0.01,0.37,0.32\r\n20150312,0.0096,1.21,-1.58,1.11\r\n20150313,0.0096,0.9,0.42,-0.49\r\n20150316,0.0096,2.54,0.49,-0.92\r\n20150317,0.0096,1.43,0.01,-0.28\r\n20150318,0.0096,2.16,0.11,0.25\r\n20150319,0.0096,0.15,0.55,-0.41\r\n20150320,0.0096,1.06,0.03,-0.08\r\n20150323,0.0096,2.21,0.48,-0.45\r\n20150324,0.0096,0.34,-0.08,-0.86\r\n20150325,0.0096,-0.39,1.56,-0.92\r\n20150326,0.0096,-0.05,-0.89,1.46\r\n20150327,0.0096,0.51,0.61,-0.21\r\n20150330,0.0096,2.07,-0.99,1.61\r\n20150331,0.0096,-0.54,0.68,-0.72\r\n20150401,0.0097,2.02,0.78,0.14\r\n20150402,0.0097,0.8,1.93,-0.78\r\n20150403,0.0097,1.14,1.17,0.1\r\n20150407,0.0097,2.34,0.07,0.48\r\n20150408,0.0097,0.08,-1.06,1.32\r\n20150409,0.0097,-1.13,-0.25,0.01\r\n20150410,0.0097,2.17,0.63,0.05\r\n20150413,0.0097,1.99,0.84,0.82\r\n20150414,0.0097,-0.04,0.14,0.13\r\n20150415,0.0097,-2.24,-1.47,1.3\r\n20150416,0.0097,2.25,-1.04,0.5\r\n20150417,0.0097,1.59,-1.51,-0.48\r\n20150420,0.0097,-1.84,-0.79,-0.79\r\n20150421,0.0097,2.63,1,-0.44\r\n20150422,0.0097,2.65,-0.04,-0.17\r\n20150423,0.0097,0.48,0.36,-1.58\r\n20150424,0.0097,-0.65,0.31,-1.16\r\n20150427,0.0097,2.43,-2.07,0.22\r\n20150428,0.0097,-1.53,-1.85,0.84\r\n20150429,0.0097,0.41,1.85,-0.96\r\n20150430,0.0097,-0.43,0.6,-0.34\r\n20150504,0.0098,0.87,0.39,0.34\r\n20150505,0.0098,-3.88,0.65,-0.54\r\n20150506,0.0098,-1.35,-0.25,-0.44\r\n20150507,0.0098,-2.32,0.25,0.35\r\n20150508,0.0098,2.94,1.54,-1.67\r\n20150511,0.0098,3.5,0.95,-1.42\r\n20150512,0.0098,1.63,0.23,-1.1\r\n20150513,0.0098,-0.26,1.28,-0.96\r\n20150514,0.0098,-0.11,0.98,-0.29\r\n20150515,0.0098,-1.49,0.86,-0.89\r\n20150518,0.0098,0.11,2.36,-1.37\r\n20150519,0.0098,2.99,0.03,0.86\r\n20150520,0.0098,1.02,0.86,-0.45\r\n20150521,0.0098,2.47,1.38,-0.69\r\n20150522,0.0098,2.04,-0.66,0.84\r\n20150525,0.0098,2.67,0.28,0.99\r\n20150526,0.0098,2.38,1.48,-1.62\r\n20150527,0.0098,0.49,1.53,-1.15\r\n20150528,0.0098,-6.53,0.56,0.52\r\n20150529,0.0098,0.22,0.93,-0.73\r\n20150601,0.0086,5.39,0.84,0.05\r\n20150602,0.0086,2.58,2.03,-1.11\r\n20150603,0.0086,0.07,0.45,0.13\r\n20150604,0.0086,0.38,-0.52,0.48\r\n20150605,0.0086,1.41,0.54,0.03\r\n20150608,0.0086,0.8,-2.04,3.45\r\n20150609,0.0086,-0.27,-0.04,-0.97\r\n20150610,0.0086,0.53,1.66,-0.52\r\n20150611,0.0086,0.52,1.72,-0.48\r\n20150612,0.0086,1.07,0.72,-0.02\r\n20150615,0.0086,-2.15,0.64,0.08\r\n20150616,0.0086,-3.71,-0.77,0.96\r\n20150617,0.0086,1.84,0.07,-0.29\r\n20150618,0.0086,-3.76,0.76,0.31\r\n20150619,0.0086,-6.41,-1.04,0.35\r\n20150623,0.0086,2.05,-2.12,1.66\r\n20150624,0.0086,2.35,0.12,0\r\n20150625,0.0086,-3.65,-0.5,0.44\r\n20150626,0.0086,-7.78,-0.84,0.48\r\n20150629,0.0086,-4.02,-2.85,2.73\r\n20150630,0.0086,5.58,-0.79,1.67\r\n20150701,0.0073,-5.62,-1.34,1.33\r\n20150702,0.0073,-4.52,-3.38,3.64\r\n20150703,0.0073,-6.01,-1.75,1.97\r\n20150706,0.0073,1.28,-5.57,7.17\r\n20150707,0.0073,-2.32,-4.89,2.8\r\n20150708,0.0073,-5.2,1.75,-0.37\r\n20150709,0.0073,5.74,-0.82,1.64\r\n20150710,0.0073,4.9,0.07,0.42\r\n20150713,0.0073,2.95,2.93,-1.97\r\n20150714,0.0073,-0.66,3.86,-2.74\r\n20150715,0.0073,-3.27,-2.02,1.06\r\n20150716,0.0073,0.96,1.05,0.34\r\n20150717,0.0073,4.28,2.16,-1.61\r\n20150720,0.0073,1.44,1.11,-2.86\r\n20150721,0.0073,0.83,1.82,-1.22\r\n20150722,0.0073,0.49,0.93,-1.78\r\n20150723,0.0073,2.49,1.01,-0.98\r\n20150724,0.0073,-1.47,-0.38,-1.45\r\n20150727,0.0073,-8.63,0.34,-0.75\r\n20150728,0.0073,-1.89,-1.62,2.87\r\n20150729,0.0073,3.87,1.88,-1.47\r\n20150730,0.0073,-2.79,-0.29,-0.5\r\n20150731,0.0073,-0.91,-0.64,1.73\r\n20150803,0.0078,-1.52,-2.87,3.88\r\n20150804,0.0078,4.43,2.84,-2.44\r\n20150805,0.0078,-1.77,1.27,-0.63\r\n20150806,0.0078,-0.91,0.67,-0.26\r\n20150807,0.0078,2.71,1.19,-1.75\r\n20150810,0.0078,5.21,0.67,-1.11\r\n20150811,0.0078,-0.01,0.99,-1.07\r\n20150812,0.0078,-1.3,0.04,-0.37\r\n20150813,0.0078,2.05,1.33,-1.09\r\n20150814,0.0078,0.26,0.81,-0.68\r\n20150817,0.0078,0.79,1.74,-1.01\r\n20150818,0.0078,-6.7,-1.77,0.82\r\n20150819,0.0078,1.68,1.53,-0.84\r\n20150820,0.0078,-3.48,0.25,1.02\r\n20150821,0.0078,-4.97,-1.67,0.73\r\n20150824,0.0078,-9.05,-0.28,-0.2\r\n20150825,0.0078,-8.26,-0.51,0.88\r\n20150826,0.0078,-1.86,-1.68,5.65\r\n20150827,0.0078,5.52,-0.74,1.99\r\n20150828,0.0078,5.4,1.43,-2.49\r\n20150831,0.0078,-1.16,-3.67,0.47\r\n20150901,0.0072,-2.36,-4.78,3.49\r\n20150902,0.0072,-0.52,-2.48,5.12\r\n20150907,0.0072,-2.23,4.34,-2.28\r\n20150908,0.0072,3.59,2.77,-1.08\r\n20150909,0.0072,2.93,1.67,-1.94\r\n20150910,0.0072,-1.59,-0.18,-0.04\r\n20150911,0.0072,0.21,1.54,-1.07\r\n20150914,0.0072,-4.24,-3.59,2.4\r\n20150915,0.0072,-4.41,-0.63,2.59\r\n20150916,0.0072,6.16,1.78,-2.16\r\n20150917,0.0072,-2.05,0.53,-0.31\r\n20150918,0.0072,0.82,0.76,-0.76\r\n20150921,0.0072,2.69,1.3,-1.92\r\n20150922,0.0072,0.97,0.15,-0.47\r\n20150923,0.0072,-1.98,0.76,-0.44\r\n20150924,0.0072,1.09,0.37,-1.04\r\n20150925,0.0072,-2.3,-0.96,1.6\r\n20150928,0.0072,1.03,1.41,-1.07\r\n20150929,0.0072,-2.07,0.32,0.8\r\n20150930,0.0072,0.53,-0.25,0.18\r\n20151008,0.0084,3.82,1.02,-1.17\r\n20151009,0.0084,1.53,0.57,-0.18\r\n20151012,0.0084,4.05,1.05,-0.92\r\n20151013,0.0084,0.49,1.07,-1.08\r\n20151014,0.0084,-1.16,-0.6,0.04\r\n20151015,0.0084,2.85,0.8,-0.89\r\n20151016,0.0084,1.67,-0.11,-0.9\r\n20151019,0.0084,-0.14,0.14,0.14\r\n20151020,0.0084,1.56,0.4,-0.76\r\n20151021,0.0084,-4.38,-2.3,2.66\r\n20151022,0.0084,2.38,1.57,-2.34\r\n20151023,0.0084,1.94,1.86,-0.64\r\n20151026,0.0084,0.62,0.36,0.37\r\n20151027,0.0084,0.3,0.34,-1.02\r\n20151028,0.0084,-2.05,0.06,1\r\n20151029,0.0084,0.49,0.77,-0.19\r\n20151030,0.0084,-0.15,-0.16,0.37\r\n20151102,0.0059,-2.16,-0.18,1.8\r\n20151103,0.0059,-0.33,0.55,0.26\r\n20151104,0.0059,5.22,0.76,-0.55\r\n20151105,0.0059,1.43,-1.27,0.37\r\n20151106,0.0059,2.36,0.5,-0.23\r\n20151109,0.0059,1.75,0.07,-0.18\r\n20151110,0.0059,0.09,0.73,-0.98\r\n20151111,0.0059,0.83,1.26,-0.61\r\n20151112,0.0059,-0.46,0.69,-0.56\r\n20151113,0.0059,-1.91,-0.7,0.77\r\n20151116,0.0059,1.32,1.43,-0.57\r\n20151117,0.0059,-0.38,-0.61,0.27\r\n20151118,0.0059,-1.32,-0.37,0.48\r\n20151119,0.0059,1.99,1.25,-0.77\r\n20151120,0.0059,0.57,1.12,-0.83\r\n20151123,0.0059,-0.72,0.17,0.62\r\n20151124,0.0059,0.46,1.01,-0.26\r\n20151125,0.0059,1.25,1.02,-0.78\r\n20151126,0.0059,-0.6,0.1,0.02\r\n20151127,0.0059,-6.18,-0.31,0.94\r\n20151130,0.0059,0.43,1.03,0.57\r\n20151201,0.0055,0.14,-0.02,1.25\r\n20151202,0.0055,1.57,-2.78,2.01\r\n20151203,0.0055,1.83,1.17,-0.75\r\n20151204,0.0055,-1.51,1.23,-0.59\r\n20151207,0.0055,0.63,0.85,-0.5\r\n20151208,0.0055,-2.1,-0.22,0.43\r\n20151209,0.0055,-0.04,-0.52,0.01\r\n20151210,0.0055,-0.31,0.12,0.3\r\n20151211,0.0055,-0.68,-0.16,0.26\r\n20151214,0.0055,2.51,-0.53,0.22\r\n20151215,0.0055,0.18,1.32,-0.75\r\n20151216,0.0055,0.29,0.61,-0.29\r\n20151217,0.0055,2.17,0.97,-0.28\r\n20151218,0.0055,-0.17,-0.13,0.54\r\n20151221,0.0055,1.58,-0.41,1.17\r\n20151222,0.0055,0.46,0.51,-0.69\r\n20151223,0.0055,-0.8,-0.39,1.2\r\n20151224,0.0055,-0.62,0.47,-0.24\r\n20151225,0.0055,0.42,0.48,0.18\r\n20151228,0.0055,-2.47,0.16,0.35\r\n20151229,0.0055,0.92,0.53,0.19\r\n20151230,0.0055,0.56,0.95,-0.37\r\n20151231,0.0055,-1.27,-0.48,0.44\r\n20160104,0.0064,-7.85,-1.28,1.07\r\n20160105,0.0064,-0.86,-1.08,1.74\r\n20160106,0.0064,2.43,0.58,-1.49\r\n20160107,0.0064,-7.89,-1.29,1.31\r\n20160108,0.0064,1.77,-0.68,0.41\r\n20160111,0.0064,-6.04,-1.7,1.34\r\n20160112,0.0064,0.47,-0.92,0.58\r\n20160113,0.0064,-2.85,-1.34,1.3\r\n20160114,0.0064,2.8,1.02,-1.13\r\n20160115,0.0064,-3.64,-0.78,0.63\r\n20160118,0.0064,0.98,1.2,-0.84\r\n20160119,0.0064,3.53,0.92,-0.42\r\n20160120,0.0064,-1.17,0.68,0.06\r\n20160121,0.0064,-3.66,-0.45,1.29\r\n20160122,0.0064,1.42,0.59,-0.67\r\n20160125,0.0064,0.83,0.6,-0.54\r\n20160126,0.0064,-7.01,-0.95,1.32\r\n20160127,0.0064,-0.57,-0.7,0.03\r\n20160128,0.0064,-3.48,-1.42,1.74\r\n20160129,0.0064,3.53,0.54,-0.35\r\n20160201,0.0074,-1.58,0.16,1.05\r\n20160202,0.0074,2.87,1.1,-1.02\r\n20160203,0.0074,-0.07,0.85,-0.48\r\n20160204,0.0074,1.73,0.82,-0.67\r\n20160205,0.0074,-0.91,-0.04,0.26\r\n20160215,0.0074,-0.41,0.08,-0.6\r\n20160216,0.0074,3.79,0.91,-1.09\r\n20160217,0.0074,1.21,0.46,-0.78\r\n20160218,0.0074,-0.34,0.1,-0.35\r\n20160219,0.0074,0.1,0.58,-0.31\r\n20160222,0.0074,2.36,-0.17,-0.18\r\n20160223,0.0074,-0.83,0.38,-0.23\r\n20160224,0.0074,0.53,-0.09,0.08\r\n20160225,0.0074,-7.03,-1.46,1.34\r\n20160226,0.0074,0.71,-0.95,0.89\r\n20160229,0.0074,-3.82,-2.6,2.59\r\n20160301,0.0055,2.13,0.35,-0.21\r\n20160302,0.0055,4.84,0.67,-1.02\r\n20160303,0.0055,0.17,0.19,0.1\r\n20160304,0.0055,-0.65,-3.16,1.96\r\n20160307,0.0055,1.26,1.47,-1.33\r\n20160308,0.0055,0.26,0.84,-0.22\r\n20160309,0.0055,-1.71,-0.54,1.79\r\n20160310,0.0055,-1.9,0.28,0.17\r\n20160311,0.0055,0.18,-0.17,0.77\r\n20160314,0.0055,2.55,1.61,-1.12\r\n20160315,0.0055,-0.19,-0.47,0.85\r\n20160316,0.0055,-0.27,-0.94,0.77\r\n20160317,0.0055,2.17,1.65,-1.2\r\n20160318,0.0055,2.54,1.57,-1.45\r\n20160321,0.0055,2.47,0.34,-0.8\r\n20160322,0.0055,-0.62,0.37,0.14\r\n20160323,0.0055,0.68,0.7,-0.13\r\n20160324,0.0055,-1.64,0.26,0.58\r\n20160325,0.0055,0.6,0.45,-0.08\r\n20160328,0.0055,-0.74,0.21,0.21\r\n20160329,0.0055,-1.52,-0.66,0.49\r\n20160330,0.0055,3.23,0.89,-0.45\r\n20160331,0.0055,0.09,0.2,-0.15\r\n20160401,0.0062,-0.11,-0.42,0.01\r\n20160405,0.0062,1.98,1.15,-0.21\r\n20160406,0.0062,0.14,0.73,-0.19\r\n20160407,0.0062,-1.64,0.01,0.18\r\n20160408,0.0062,-0.89,0.08,0.3\r\n20160411,0.0062,1.86,0.63,-0.25\r\n20160412,0.0062,-0.58,-0.31,-0.22\r\n20160413,0.0062,1.5,0.02,-0.39\r\n20160414,0.0062,0.62,0.55,0.18\r\n20160415,0.0062,-0.23,0.2,-0.12\r\n20160418,0.0062,-1.52,-0.18,0.04\r\n20160419,0.0062,0.28,0.21,0.02\r\n20160420,0.0062,-3.19,-2.12,0.84\r\n20160421,0.0062,-0.89,-0.26,-0.36\r\n20160422,0.0062,0.58,0.63,0.73\r\n20160425,0.0062,-0.49,-0.03,-0.01\r\n20160426,0.0062,0.82,0.62,0\r\n20160427,0.0062,-0.4,0.24,0.16\r\n20160428,0.0062,-0.21,-0.08,0.14\r\n20160429,0.0062,-0.2,0.22,0.16\r\n20160503,0.0061,2.45,0.97,-0.32\r\n20160504,0.0061,-0.08,0.22,0.06\r\n20160505,0.0061,0.41,0.46,-0.42\r\n20160506,0.0061,-3.24,-0.97,0.89\r\n20160509,0.0061,-3.09,-1.27,1.53\r\n20160510,0.0061,0.04,-0.19,0.39\r\n20160511,0.0061,0.01,-0.61,1.24\r\n20160512,0.0061,0.07,-0.27,0.43\r\n20160513,0.0061,-0.35,0.11,-0.1\r\n20160516,0.0061,1.2,0.81,-0.48\r\n20160517,0.0061,-0.17,-0.18,-0.86\r\n20160518,0.0061,-1.77,-1.72,1.02\r\n20160519,0.0061,0.19,0.29,-0.7\r\n20160520,0.0061,0.81,0.55,-0.32\r\n20160523,0.0061,0.9,0.92,-0.33\r\n20160524,0.0061,-0.86,-0.24,-0.08\r\n20160525,0.0061,-0.27,0.18,0\r\n20160526,0.0061,0.35,0.18,-0.28\r\n20160527,0.0061,-0.1,0.13,-0.02\r\n20160530,0.0061,-0.11,-0.38,0.27\r\n20160531,0.0061,3.75,0.5,-0.46\r\n20160601,0.0062,0.13,0.55,-0.6\r\n20160602,0.0062,0.56,0.78,-0.56\r\n20160603,0.0062,0.54,-0.12,0.51\r\n20160606,0.0062,-0.04,0.27,-0.14\r\n20160607,0.0062,0.06,0.45,0.13\r\n20160608,0.0062,-0.28,-0.17,0.01\r\n20160613,0.0062,-3.85,-1.33,1.32\r\n20160614,0.0062,0.35,0.26,0.36\r\n20160615,0.0062,2.19,1.46,-0.83\r\n20160616,0.0062,-0.43,0.55,-0.36\r\n20160617,0.0062,0.61,0.28,0\r\n20160620,0.0062,0.27,0.47,0.13\r\n20160621,0.0062,-0.61,-0.41,0.63\r\n20160622,0.0062,1.29,0.48,-0.67\r\n20160623,0.0062,-0.38,-0.03,-0.13\r\n20160624,0.0062,-1.03,0.11,-0.16\r\n20160627,0.0062,1.81,0.63,-0.79\r\n20160628,0.0062,0.76,0.34,-0.04\r\n20160629,0.0062,0.34,-0.2,0\r\n20160630,0.0062,0.03,0.11,0.27\r\n20160701,0.0061,-0.05,-0.09,0.43\r\n20160704,0.0061,1.94,0.09,-0.23\r\n20160705,0.0061,0.39,0.11,-0.18\r\n20160706,0.0061,0.44,0.28,0.32\r\n20160707,0.0061,0.13,-0.01,0.12\r\n20160708,0.0061,-0.49,0.24,-0.24\r\n20160711,0.0061,0.02,-0.68,0.2\r\n20160712,0.0061,1.71,-0.51,0.78\r\n20160713,0.0061,0.63,0,0.04\r\n20160714,0.0061,-0.11,0.29,-0.08\r\n20160715,0.0061,-0.11,-0.3,0.66\r\n20160718,0.0061,-0.39,-0.11,0.3\r\n20160719,0.0061,-0.04,0.51,-0.8\r\n20160720,0.0061,-0.22,0.22,0.09\r\n20160721,0.0061,0.29,-0.18,0.22\r\n20160722,0.0061,-0.87,-0.05,-0.39\r\n20160725,0.0061,0.09,0.32,0.45\r\n20160726,0.0061,1.2,0.18,0.3\r\n20160727,0.0061,-2.84,-1.68,2.21\r\n20160728,0.0061,0.01,0.13,1.12\r\n20160729,0.0061,-0.5,-0.15,0.54\r\n20160801,0.0055,-1.08,-0.67,0.05\r\n20160802,0.0055,0.65,0.35,-0.38\r\n20160803,0.0055,0.23,0.27,0.02\r\n20160804,0.0055,0.38,0.23,-0.47\r\n20160805,0.0055,-0.24,-0.33,0.28\r\n20160808,0.0055,1.04,0.13,-0.24\r\n20160809,0.0055,0.82,0.35,0.02\r\n20160810,0.0055,-0.33,0,-0.27\r\n20160811,0.0055,-0.82,-0.74,0.76\r\n20160812,0.0055,1.49,-0.37,0.56\r\n20160815,0.0055,2.6,-0.34,0.31\r\n20160816,0.0055,-0.11,0.67,-0.71\r\n20160817,0.0055,0.06,0.31,-0.41\r\n20160818,0.0055,-0.19,0.2,-0.16\r\n20160819,0.0055,0.16,0.18,-0.13\r\n20160822,0.0055,-0.98,-0.24,0.49\r\n20160823,0.0055,0.12,0.16,0.05\r\n20160824,0.0055,-0.01,0.26,-0.36\r\n20160825,0.0055,-0.62,-0.01,0.14\r\n20160826,0.0055,0.11,0.34,-0.23\r\n20160829,0.0055,0.06,0.16,-0.25\r\n20160830,0.0055,0.11,-0.15,-0.12\r\n20160831,0.0055,0.26,-0.22,0.2\r\n20160901,0.0062,-0.78,-0.01,0.08\r\n20160902,0.0062,-0.12,-0.3,0.46\r\n20160905,0.0062,0.27,0.2,0.02\r\n20160906,0.0062,0.95,0.67,-0.25\r\n20160907,0.0062,-0.06,0.19,0.31\r\n20160908,0.0062,0.15,0.37,0.07\r\n20160909,0.0062,-0.65,-0.05,0.04\r\n20160912,0.0062,-2.28,-0.65,1.32\r\n20160913,0.0062,0.21,0.6,-0.28\r\n20160914,0.0062,-0.64,-0.1,-0.35\r\n20160919,0.0062,0.86,0.53,-0.06\r\n20160920,0.0062,-0.12,0.2,0.14\r\n20160921,0.0062,0.22,-0.02,-0.02\r\n20160922,0.0062,0.62,0.01,0.24\r\n20160923,0.0062,-0.36,-0.06,0.1\r\n20160926,0.0062,-1.82,-0.29,0.41\r\n20160927,0.0062,0.64,0.16,-0.09\r\n20160928,0.0062,-0.26,-0.02,-0.08\r\n20160929,0.0062,0.36,-0.08,0.14\r\n20160930,0.0062,0.33,0.11,-0.12\r\n20161010,0.0079,1.6,0.54,-0.82\r\n20161011,0.0079,0.45,0.14,-0.33\r\n20161012,0.0079,-0.13,0.18,-0.01\r\n20161013,0.0079,0.07,0.16,0.07\r\n20161014,0.0079,0,0.1,0.07\r\n20161017,0.0079,-0.84,-0.09,-0.03\r\n20161018,0.0079,1.44,0.26,-0.06\r\n20161019,0.0079,-0.13,0.01,-0.34\r\n20161020,0.0079,0.15,0.26,0\r\n20161021,0.0079,-0.06,-0.33,0.48\r\n20161024,0.0079,1.03,-0.17,-0.08\r\n20161025,0.0079,0.13,0.24,-0.21\r\n20161026,0.0079,-0.52,0.19,0.25\r\n20161027,0.0079,-0.19,0.23,-0.09\r\n20161028,0.0079,-0.43,-0.53,0.25\r\n20161031,0.0079,-0.11,0.18,0.09\r\n20161101,0.0056,0.79,0.47,0.08\r\n20161102,0.0056,-0.81,0.13,0.06\r\n20161103,0.0056,0.69,-0.01,0.19\r\n20161104,0.0056,-0.31,0.13,-0.21\r\n20161107,0.0056,0.11,-0.08,0.19\r\n20161108,0.0056,0.48,0.02,-0.14\r\n20161109,0.0056,-0.72,-0.18,0.21\r\n20161110,0.0056,1.22,0.35,0.02\r\n20161111,0.0056,0.75,-0.17,-0.09\r\n20161114,0.0056,0.44,-0.07,-0.25\r\n20161115,0.0056,0.11,0.43,-0.26\r\n20161116,0.0056,0.05,-0.18,-0.24\r\n20161117,0.0056,-0.09,-0.32,0.21\r\n20161118,0.0056,-0.47,0.35,0.06\r\n20161121,0.0056,0.61,0,0.11\r\n20161122,0.0056,0.89,-0.01,-0.29\r\n20161123,0.0056,-0.29,-0.35,0.39\r\n20161124,0.0056,-0.08,-0.36,0.45\r\n20161125,0.0056,0.61,-0.27,0.37\r\n20161128,0.0056,0.34,-0.5,0.14\r\n20161129,0.0056,0.01,-1.06,0.5\r\n20161130,0.0056,-0.72,0.4,-0.26\r\n20161201,0.0058,0.64,-0.32,-0.32\r\n20161202,0.0058,-1.14,-0.84,0.3\r\n20161205,0.0058,-1.08,0.61,0.14\r\n20161206,0.0058,-0.11,0.35,-0.07\r\n20161207,0.0058,0.74,0.45,-0.38\r\n20161208,0.0058,-0.41,-0.25,0.24\r\n20161209,0.0058,0.28,-0.52,0.41\r\n20161212,0.0058,-3.29,-1.94,1.41\r\n20161213,0.0058,0.12,0.12,-0.71\r\n20161214,0.0058,-0.67,0.03,-0.14\r\n20161215,0.0058,-0.37,1.36,-0.96\r\n20161216,0.0058,0.4,0.63,-0.03\r\n20161219,0.0058,-0.27,0.32,-0.01\r\n20161220,0.0058,-0.4,0.31,-0.05\r\n20161221,0.0058,1.02,-0.37,-0.43\r\n20161222,0.0058,-0.03,-0.07,-0.23\r\n20161223,0.0058,-0.89,-0.39,0.27\r\n20161226,0.0058,0.37,0.21,0.13\r\n20161227,0.0058,-0.2,0.47,0.05\r\n20161228,0.0058,-0.4,0.07,-0.12\r\n20161229,0.0058,-0.23,-0.01,0.33\r\n20161230,0.0058,0.23,-0.12,0.45\r\n20170103,0.0071,0.95,0.01,-0.15\r\n20170104,0.0071,0.93,0.36,-0.11\r\n20170105,0.0071,0.09,-0.15,-0.42\r\n20170106,0.0071,-0.45,-0.21,-0.12\r\n20170109,0.0071,0.49,0.07,-0.06\r\n20170110,0.0071,-0.3,-0.18,0.16\r\n20170111,0.0071,-0.81,-0.28,0.2\r\n20170112,0.0071,-0.65,-0.28,0.42\r\n20170113,0.0071,-0.59,-1.3,0.78\r\n20170116,0.0071,-1.34,-2.85,1.92\r\n20170117,0.0071,0.46,0.72,-0.41\r\n20170118,0.0071,0.04,-0.17,0.65\r\n20170119,0.0071,-0.41,0.2,0.25\r\n20170120,0.0071,0.95,0.72,-0.27\r\n20170123,0.0071,0.59,0.38,-0.38\r\n20170124,0.0071,0.06,-0.41,0.07\r\n20170125,0.0071,0.29,0.08,0.12\r\n20170126,0.0071,0.45,0.26,0.13\r\n20170203,0.0064,-0.56,0.19,-0.18\r\n20170206,0.0064,0.58,0.63,-0.05\r\n20170207,0.0064,-0.09,0.29,-0.12\r\n20170208,0.0064,0.51,0.41,-0.06\r\n20170209,0.0064,0.56,0.51,-0.12\r\n20170210,0.0064,0.24,-0.31,0.8\r\n20170213,0.0064,0.61,0.1,0.08\r\n20170214,0.0064,-0.01,-0.03,0.14\r\n20170215,0.0064,-0.55,-0.49,0.23\r\n20170216,0.0064,0.5,0.18,-0.09\r\n20170217,0.0064,-0.76,-0.42,-0.04\r\n20170220,0.0064,1.21,0.09,0.64\r\n20170221,0.0064,0.61,0.3,-0.25\r\n20170222,0.0064,0.37,0.05,0.21\r\n20170223,0.0064,-0.24,-0.04,-0.06\r\n20170224,0.0064,0.14,0.08,-0.3\r\n20170227,0.0064,-0.73,-0.16,-0.16\r\n20170228,0.0064,0.44,-0.14,-0.35\r\n20170301,0.0055,0.26,-0.06,-0.46\r\n20170302,0.0055,-0.68,-0.03,0.18\r\n20170303,0.0055,-0.15,0.61,0.4\r\n20170306,0.0055,0.68,0.55,-0.2\r\n20170307,0.0055,0.19,0.12,0.17\r\n20170308,0.0055,-0.21,-0.21,-0.19\r\n20170309,0.0055,-0.81,0.12,0.19\r\n20170310,0.0055,-0.01,0.1,0.34\r\n20170313,0.0055,0.78,0.11,0.4\r\n20170314,0.0055,-0.07,-0.23,0.01\r\n20170315,0.0055,0.02,0.06,0.39\r\n20170316,0.0055,0.74,0.29,-0.35\r\n20170317,0.0055,-0.89,0.04,-0.18\r\n20170320,0.0055,0.3,0.17,0.16\r\n20170321,0.0055,0.32,-0.09,0.47\r\n20170322,0.0055,-0.49,-0.1,0.24\r\n20170323,0.0055,0.09,0.01,0.46\r\n20170324,0.0055,0.6,-0.2,0.27\r\n20170327,0.0055,-0.2,-0.09,-0.37\r\n20170328,0.0055,-0.32,-0.06,0.08\r\n20170329,0.0055,-0.25,-0.5,0.1\r\n20170330,0.0055,-1.2,-0.78,1.07\r\n20170331,0.0055,0.49,-0.36,0.23\r\n20170405,0.0068,1.72,0.47,-0.15\r\n20170406,0.0068,0.31,-0.1,-0.07\r\n20170407,0.0068,0.18,-0.28,-0.06\r\n20170410,0.0068,-0.45,-0.69,0.3\r\n20170411,0.0068,0.6,0.31,-0.27\r\n20170412,0.0068,-0.49,-0.46,0.45\r\n20170413,0.0068,0.2,0.14,0.1\r\n20170414,0.0068,-1.12,-0.6,-0.13\r\n20170417,0.0068,-0.76,-0.8,1.15\r\n20170418,0.0068,-0.73,-0.43,0.32\r\n20170419,0.0068,-0.86,-0.45,0.66\r\n20170420,0.0068,0.09,-0.5,0.71\r\n20170421,0.0068,-0.08,-0.41,0.33\r\n20170424,0.0068,-1.74,-1.47,1.41\r\n20170425,0.0068,0.27,0.2,0.09\r\n20170426,0.0068,0.19,0.29,0.49\r\n20170427,0.0068,0.27,0.42,0.07\r\n20170428,0.0068,0.16,0.47,-0.52\r\n20170502,0.0064,-0.2,0.28,-0.13\r\n20170503,0.0064,-0.38,0.31,0.17\r\n20170504,0.0064,-0.25,-0.15,-0.07\r\n20170505,0.0064,-0.91,-0.33,0.48\r\n20170508,0.0064,-1.19,-0.64,1.2\r\n20170509,0.0064,0.22,0.38,-0.15\r\n20170510,0.0064,-1.04,-0.99,0.68\r\n20170511,0.0064,0.17,-0.88,0.86\r\n20170512,0.0064,0.52,-0.65,0.42\r\n20170515,0.0064,0.33,0.05,-0.15\r\n20170516,0.0064,1.24,0.89,-0.57\r\n20170517,0.0064,-0.2,0.58,-0.6\r\n20170518,0.0064,-0.47,-0.25,0.36\r\n20170519,0.0064,0.03,-0.15,-0.18\r\n20170522,0.0064,-0.53,-1.45,0.77\r\n20170523,0.0064,-0.79,-2.14,2.28\r\n20170524,0.0064,0.22,0.32,-0.32\r\n20170525,0.0064,1.27,-0.29,0.4\r\n20170526,0.0064,-0.05,0.16,-0.09\r\n20170531,0.0064,0.14,-0.35,0.44\r\n20170601,0.0056,-0.77,-2.02,1.94\r\n20170602,0.0056,0.29,1.1,-0.67\r\n20170605,0.0056,-0.1,0.94,-0.79\r\n20170606,0.0056,0.47,0.02,0.03\r\n20170607,0.0056,1.58,0.82,-0.59\r\n20170608,0.0056,0.34,-0.33,0.27\r\n20170609,0.0056,0.28,-0.2,0.17\r\n20170612,0.0056,-0.63,-0.85,0.82\r\n20170613,0.0056,0.68,0.88,-0.41\r\n20170614,0.0056,-0.63,0.72,-0.55\r\n20170615,0.0056,0.25,0.92,-0.87\r\n20170616,0.0056,-0.18,-0.02,-0.22\r\n20170619,0.0056,0.69,0.06,0.11\r\n20170620,0.0056,-0.03,0.06,-0.08\r\n20170621,0.0056,0.6,-0.52,0.62\r\n20170622,0.0056,-0.52,-1.25,1.1\r\n20170623,0.0056,0.47,-0.41,0.62\r\n20170626,0.0056,1.13,0.17,0.18\r\n20170627,0.0056,0.11,0.16,0.27\r\n20170628,0.0056,-0.63,-0.02,0.11\r\n20170629,0.0056,0.53,-0.05,0.08\r\n20170630,0.0056,0.19,-0.11,0.36\r\n20170703,0.0061,0.15,0.81,-0.52\r\n20170704,0.0061,-0.51,0.26,-0.46\r\n20170705,0.0061,0.85,0.05,0.14\r\n20170706,0.0061,0.12,0.1,0.01\r\n20170707,0.0061,0.13,0.27,-0.01\r\n20170710,0.0061,-0.17,-0.24,0.37\r\n20170711,0.0061,-0.2,-0.93,0.95\r\n20170712,0.0061,-0.15,0.05,0.09\r\n20170713,0.0061,0.6,-0.48,0.44\r\n20170714,0.0061,0.15,-0.5,0.77\r\n20170717,0.0061,-2.2,-2.88,3.06\r\n20170718,0.0061,0.45,0.66,0.13\r\n20170719,0.0061,1.56,-0.12,0.31\r\n20170720,0.0061,0.52,-0.08,-0.13\r\n20170721,0.0061,-0.15,0.19,0.05\r\n20170724,0.0061,0.43,0.4,0.08\r\n20170725,0.0061,-0.29,0.15,0.02\r\n20170726,0.0061,-0.04,0.32,0.07\r\n20170727,0.0061,0.4,0.68,-1.42\r\n20170728,0.0061,0.17,-0.15,0.33\r\n20170731,0.0061,0.62,0.32,0\r\n20170801,0.0055,0.47,-0.42,0.25\r\n20170802,0.0055,-0.38,-0.46,0.3\r\n20170803,0.0055,-0.34,0.69,-0.67\r\n20170804,0.0055,-0.45,-0.06,-0.25\r\n20170807,0.0055,0.64,0.14,-0.26\r\n20170808,0.0055,0.19,0.05,-0.37\r\n20170809,0.0055,0.03,0.19,-0.22\r\n20170810,0.0055,-0.51,-0.48,-0.04\r\n20170811,0.0055,-1.74,0.11,-0.11\r\n20170814,0.0055,1.36,0.24,-1.15\r\n20170815,0.0055,0.38,0.02,0.1\r\n20170816,0.0055,0.05,0.43,-0.44\r\n20170817,0.0055,0.66,0.27,-0.33\r\n20170818,0.0055,-0.11,-0.19,0.31\r\n20170821,0.0055,0.55,0.12,-0.43\r\n20170822,0.0055,-0.14,-0.47,0.5\r\n20170823,0.0055,-0.2,-0.19,0.6\r\n20170824,0.0055,-0.53,-0.16,0.05\r\n20170825,0.0055,1.56,-0.44,0.48\r\n20170828,0.0055,1.07,-0.14,-0.31\r\n20170829,0.0055,-0.08,0.08,0.44\r\n20170830,0.0055,0.15,0.01,-0.33\r\n20170831,0.0055,-0.12,0.3,-0.36\r\n20170901,0.0059,0.29,0.23,-0.47\r\n20170904,0.0059,0.51,0.15,-0.31\r\n20170905,0.0059,0.22,-0.05,0.3\r\n20170906,0.0059,0.15,0.29,-0.46\r\n20170907,0.0059,-0.59,0.41,0.15\r\n20170908,0.0059,0.04,0.33,-0.04\r\n20170911,0.0059,0.48,0.39,-0.4\r\n20170912,0.0059,0.02,-0.5,0.04\r\n20170913,0.0059,0.23,0.16,-0.05\r\n20170914,0.0059,-0.28,0.19,-0.03\r\n20170915,0.0059,-0.37,0.05,0.17\r\n20170918,0.0059,0.5,0.19,-0.29\r\n20170919,0.0059,-0.28,-0.14,0.17\r\n20170920,0.0059,0.49,0.19,-0.61\r\n20170921,0.0059,-0.5,-0.49,0.75\r\n20170922,0.0059,-0.25,-0.11,0.23\r\n20170925,0.0059,-0.69,-0.69,0.64\r\n20170926,0.0059,0.15,-0.06,-0.15\r\n20170927,0.0059,0.28,0.29,-0.32\r\n20170928,0.0059,-0.18,-0.38,-0.01\r\n20170929,0.0059,0.44,0.5,-0.11\r\n20171009,0.0075,1.01,0.25,-0.2\r\n20171010,0.0075,0.35,0.71,-0.08\r\n20171011,0.0075,0.07,-0.25,0.89\r\n20171012,0.0075,-0.09,-0.16,0.11\r\n20171013,0.0075,0.31,0.04,-0.15\r\n20171016,0.0075,-0.7,-1.49,1.08\r\n20171017,0.0075,-0.11,0.18,0.18\r\n20171018,0.0075,0.14,-1.05,0.84\r\n20171019,0.0075,-0.52,-0.55,0.64\r\n20171020,0.0075,0.41,0.72,-0.41\r\n20171023,0.0075,0.24,0.35,-0.03\r\n20171024,0.0075,0.2,-0.77,0.66\r\n20171025,0.0075,0.41,0.2,-0.17\r\n20171026,0.0075,0.29,-0.03,-0.55\r\n20171027,0.0075,0.09,-0.75,0.56\r\n20171030,0.0075,-0.98,-1.64,1\r\n20171031,0.0075,0.24,0.4,-0.55\r\n20171101,0.0056,0.02,0.06,-0.04\r\n20171102,0.0056,-0.49,-0.84,0.35\r\n20171103,0.0056,-0.5,-0.37,-0.16\r\n20171106,0.0056,0.78,0.01,-0.39\r\n20171107,0.0056,0.72,-0.3,-0.2\r\n20171108,0.0056,0,0.39,-0.03\r\n20171109,0.0056,0.51,-0.12,-0.19\r\n20171110,0.0056,0.31,-0.29,0.14\r\n20171113,0.0056,0.4,-0.43,0.26\r\n20171114,0.0056,-0.71,0.11,0.24\r\n20171115,0.0056,-0.89,0.22,0.39\r\n20171116,0.0056,0.18,-0.57,0.2\r\n20171117,0.0056,-1.11,-2.23,1.98\r\n20171120,0.0056,0.61,-0.55,-0.01\r\n20171121,0.0056,0.78,-0.93,0.28\r\n20171122,0.0056,0.28,-0.11,0.32\r\n20171123,0.0056,-2.64,0.57,0.03\r\n20171124,0.0056,0.04,0.36,0.22\r\n20171127,0.0056,-1.18,0.48,0.47\r\n20171128,0.0056,0.68,0.67,-0.53\r\n20171129,0.0056,0.14,0.17,0.46\r\n20171130,0.0056,-0.82,0.44,0.08\r\n20171201,0.0061,0.29,0.42,-0.62\r\n20171204,0.0061,-0.18,-1.19,0.67\r\n20171205,0.0061,-0.71,-1.69,1.92\r\n20171206,0.0061,-0.03,0.85,-0.79\r\n20171207,0.0061,-0.73,0.5,-0.66\r\n20171208,0.0061,0.76,0.29,-0.24\r\n20171211,0.0061,1.26,-0.34,-0.26\r\n20171212,0.0061,-1.13,-0.03,-0.19\r\n20171213,0.0061,0.7,0,0.09\r\n20171214,0.0061,-0.29,0.35,-0.29\r\n20171215,0.0061,-0.84,0.57,-0.51\r\n20171218,0.0061,-0.14,-0.19,0.88\r\n20171219,0.0061,0.93,-0.28,0.27\r\n20171220,0.0061,-0.39,-0.66,0.32\r\n20171221,0.0061,0.56,-0.59,0.32\r\n20171222,0.0061,-0.13,-0.12,0.12\r\n20171225,0.0061,-0.65,-0.53,0.89\r\n20171226,0.0061,0.51,0.17,0.07\r\n20171227,0.0061,-0.94,0.54,-0.14\r\n20171228,0.0061,0.57,-0.1,0.52\r\n20171229,0.0061,0.52,0.24,-0.06\r\n20180102,0.0058,1.25,-0.16,0.62\r\n20180103,0.0058,0.71,0.17,-0.35\r\n20180104,0.0058,0.49,-0.44,0.14\r\n20180105,0.0058,0.16,0.02,0.31\r\n20180108,0.0058,0.45,-0.31,0.56\r\n20180109,0.0058,0.23,-0.4,0\r\n20180110,0.0058,0.12,-0.91,0.15\r\n20180111,0.0058,0.14,0.31,-0.62\r\n20180112,0.0058,0,-0.44,0.58\r\n20180115,0.0058,-0.92,-1.56,1.39\r\n20180116,0.0058,0.69,-0.17,0.02\r\n20180117,0.0058,-0.02,0.07,0.23\r\n20180118,0.0058,0.65,-0.19,1.04\r\n20180119,0.0058,0.24,-0.27,0.26\r\n20180122,0.0058,0.87,-0.05,0.03\r\n20180123,0.0058,1,-0.9,0.41\r\n20180124,0.0058,0.38,0.04,-0.47\r\n20180125,0.0058,-0.35,0.25,-0.04\r\n20180126,0.0058,0.2,-0.33,0.31\r\n20180129,0.0058,-1.27,0.24,0.43\r\n20180130,0.0058,-0.88,0.82,-0.25\r\n20180131,0.0058,-0.5,-1.56,1.77\r\n20180201,0.0077,-1.61,-2.35,2.41\r\n20180202,0.0077,0.44,-1.39,0.42\r\n20180205,0.0077,0.27,-0.49,1.09\r\n20180206,0.0077,-3.73,-1.2,1.48\r\n20180207,0.0077,-1.62,2.01,-1.1\r\n20180208,0.0077,-0.6,1.51,-1.42\r\n20180209,0.0077,-3.91,0.91,-0.17\r\n20180212,0.0077,1.48,0.83,-0.77\r\n20180213,0.0077,0.9,-0.74,0.32\r\n20180214,0.0077,0.53,-0.24,0\r\n20180222,0.0077,2.15,-0.57,-0.05\r\n20180223,0.0077,0.43,-0.07,0.21\r\n20180226,0.0077,1.59,0.49,-0.66\r\n20180227,0.0077,-0.95,1.04,-1.28\r\n20180228,0.0077,-0.46,0.72,-0.39\r\n20180301,0.0058,0.78,0.66,-0.69\r\n20180302,0.0058,-0.65,0.23,-0.3\r\n20180305,0.0058,0.18,0.1,-0.82\r\n20180306,0.0058,1.12,-0.09,0.07\r\n20180307,0.0058,-0.69,0.07,0.23\r\n20180308,0.0058,0.68,0.15,0.17\r\n20180309,0.0058,0.94,0.51,-0.89\r\n20180312,0.0058,0.81,0.66,-0.75\r\n20180313,0.0058,-0.67,0.35,-0.02\r\n20180314,0.0058,-0.69,-0.36,0.78\r\n20180315,0.0058,0.12,-0.76,-0.04\r\n20180316,0.0058,-0.75,0.78,-0.31\r\n20180319,0.0058,0.23,0.03,-0.12\r\n20180320,0.0058,0.27,-0.19,0.05\r\n20180321,0.0058,-0.52,0.06,0.04\r\n20180322,0.0058,-0.63,0.84,-0.5\r\n20180323,0.0058,-3.72,-2.03,0.97\r\n20180326,0.0058,0.04,0.8,-2.25\r\n20180327,0.0058,1.42,1.01,-1.33\r\n20180328,0.0058,-1.32,0.88,-0.34\r\n20180329,0.0058,1.19,-0.03,0.67\r\n20180330,0.0058,0.59,0.68,-1.3\r\n20180402,0.0068,-0.1,0.47,-0.73\r\n20180403,0.0068,-0.88,-0.11,0.36\r\n20180404,0.0068,-0.38,-0.06,0.33\r\n20180409,0.0068,0.09,0.22,-0.14\r\n20180410,0.0068,1.35,-1.09,1.1\r\n20180411,0.0068,0.54,0.24,0.24\r\n20180412,0.0068,-0.82,0.19,-0.3\r\n20180413,0.0068,-0.56,0.37,-0.05\r\n20180416,0.0068,-1.16,0.27,-1.37\r\n20180417,0.0068,-1.77,-0.4,0.46\r\n20180418,0.0068,0.94,0.43,-0.45\r\n20180419,0.0068,0.82,-0.51,0.28\r\n20180420,0.0068,-1.61,-0.66,0.28\r\n20180423,0.0068,-0.37,-0.42,0.35\r\n20180424,0.0068,2.16,0.03,0.04\r\n20180425,0.0068,-0.16,0.5,-0.82\r\n20180426,0.0068,-1.67,-0.32,0.26\r\n20180427,0.0068,0.34,-0.43,0.14\r\n20180502,0.0058,0.03,-0.5,0.91\r\n20180503,0.0058,0.8,0.21,-0.27\r\n20180504,0.0058,-0.26,-0.05,0.07\r\n20180507,0.0058,1.7,0.23,0.04\r\n20180508,0.0058,0.79,-0.06,0.06\r\n20180509,0.0058,-0.12,0.28,0.19\r\n20180510,0.0058,0.51,0.16,0.36\r\n20180511,0.0058,-0.61,-0.44,0.35\r\n20180514,0.0058,0.25,-0.94,0.84\r\n20180515,0.0058,0.65,0.37,-0.31\r\n20180516,0.0058,-0.62,0.11,0.01\r\n20180517,0.0058,-0.55,0.43,0.4\r\n20180518,0.0058,1,-0.38,0.35\r\n20180521,0.0058,0.75,0.38,-0.74\r\n20180522,0.0058,0.01,0.5,-0.33\r\n20180523,0.0058,-1.37,0.19,-0.58\r\n20180524,0.0058,-0.52,0.32,0.08\r\n20180525,0.0058,-0.55,-0.29,0.41\r\n20180528,0.0058,-0.05,-0.72,0.95\r\n20180529,0.0058,-0.75,0.06,0.54\r\n20180530,0.0058,-2.5,-1.43,0.44\r\n20180531,0.0058,1.8,0.07,0.78\r\n20180601,0.0062,-0.91,0.1,0.68\r\n20180604,0.0062,0.42,-0.53,0.68\r\n20180605,0.0062,1.19,0.3,-0.75\r\n20180606,0.0062,0.02,0.11,-0.36\r\n20180607,0.0062,-0.34,-0.41,0.64\r\n20180608,0.0062,-1.23,0.13,-0.2\r\n20180611,0.0062,-0.5,-0.2,0.58\r\n20180612,0.0062,1.06,-0.34,0.43\r\n20180613,0.0062,-1.14,-0.75,0.61\r\n20180614,0.0062,-0.3,-0.1,0.24\r\n20180615,0.0062,-0.92,-1.15,1.46\r\n20180619,0.0062,-4.44,-2.88,1.84\r\n20180620,0.0062,0.64,0.63,0.03\r\n20180621,0.0062,-1.47,-0.89,1.66\r\n20180622,0.0062,0.81,0.58,-0.42\r\n20180625,0.0062,-1,0.64,-0.44\r\n20180626,0.0062,-0.29,1.14,-1.56\r\n20180627,0.0062,-1.31,0.78,-0.47\r\n20180628,0.0062,-0.97,-0.12,0.2\r\n20180629,0.0062,2.67,0.03,-0.92\r\n20180702,0.0058,-2.32,0.72,-1.23\r\n20180703,0.0058,0.5,1.16,-1\r\n20180704,0.0058,-1.26,-0.63,0.52\r\n20180705,0.0058,-1.4,-1.29,1.3\r\n20180706,0.0058,0.49,-0.04,0.04\r\n20180709,0.0058,2.72,-0.46,0.47\r\n20180710,0.0058,0.58,0.43,-0.21\r\n20180711,0.0058,-1.87,-0.76,0.51\r\n20180712,0.0058,2.42,0.12,-0.3\r\n20180713,0.0058,0.36,-0.14,0.24\r\n20180716,0.0058,-0.42,0.32,-0.55\r\n20180717,0.0058,-0.44,0.8,-0.28\r\n20180718,0.0058,-0.49,0.17,0.31\r\n20180719,0.0058,-0.53,-0.43,0.91\r\n20180720,0.0058,1.8,-0.57,0.58\r\n20180723,0.0058,0.98,0,0.26\r\n20180724,0.0058,1.57,0.32,0.42\r\n20180725,0.0058,0,0.13,-0.25\r\n20180726,0.0058,-0.88,0.98,0.1\r\n20180727,0.0058,-0.38,-0.19,0.29\r\n20180730,0.0058,-0.57,-0.76,1.28\r\n20180731,0.0058,0.19,-0.1,0.23\r\n20180801,0.0055,-1.85,0.1,-0.34\r\n20180802,0.0055,-2.16,-0.65,0.28\r\n20180803,0.0055,-1.28,0.49,0.79\r\n20180806,0.0055,-1.58,-0.01,1\r\n20180807,0.0055,2.89,-0.17,0.14\r\n20180808,0.0055,-1.48,0.17,0.41\r\n20180809,0.0055,2.23,-0.45,-0.87\r\n20180810,0.0055,0.27,0.01,-0.94\r\n20180813,0.0055,-0.11,0,-1.06\r\n20180814,0.0055,-0.31,0.34,0.03\r\n20180815,0.0055,-2.13,0.37,0.01\r\n20180816,0.0055,-0.77,-0.21,0.24\r\n20180817,0.0055,-1.46,-0.3,0.51\r\n20180820,0.0055,1,-0.36,0.01\r\n20180821,0.0055,1.41,-0.5,-0.13\r\n20180822,0.0055,-0.84,-0.19,0.39\r\n20180823,0.0055,0.42,0.03,-0.57\r\n20180824,0.0055,0.08,-0.35,0.47\r\n20180827,0.0055,2.23,-0.51,-0.41\r\n20180828,0.0055,-0.06,0.12,-0.19\r\n20180829,0.0055,-0.44,-0.13,-0.11\r\n20180830,0.0055,-1.24,-0.34,0.43\r\n20180831,0.0055,-0.62,-0.29,0.59\r\n20180903,0.0065,-0.17,0.02,-0.2\r\n20180904,0.0065,1.1,0.1,-0.2\r\n20180905,0.0065,-1.74,0.35,-0.26\r\n20180906,0.0065,-0.69,0.35,-0.35\r\n20180907,0.0065,0.3,0.09,0.51\r\n20180910,0.0065,-1.44,-0.25,0.81\r\n20180911,0.0065,-0.08,0.23,-0.61\r\n20180912,0.0065,-0.38,0.78,0.29\r\n20180913,0.0065,1.03,-0.03,0.24\r\n20180914,0.0065,-0.3,-0.66,0.87\r\n20180917,0.0065,-1.24,-0.41,0.16\r\n20180918,0.0065,1.78,-0.1,0.27\r\n20180919,0.0065,1.31,-0.3,-0.25\r\n20180920,0.0065,-0.1,-0.09,0.29\r\n20180921,0.0065,2.42,-1.32,0.83\r\n20180925,0.0065,-0.61,0.15,-0.46\r\n20180926,0.0065,0.9,-0.74,0.14\r\n20180927,0.0065,-0.69,-1.09,0.77\r\n20180928,0.0065,0.98,-0.06,0.22\r\n20181008,0.0071,-3.89,0.74,0.06\r\n20181009,0.0071,0.03,0.38,-0.06\r\n20181010,0.0071,0.03,0.27,0.19\r\n20181011,0.0071,-5.64,-2.1,1.02\r\n20181012,0.0071,0.8,-2.17,1.04\r\n20181015,0.0071,-1.49,0.23,-0.13\r\n20181016,0.0071,-1.23,-0.41,0.42\r\n20181017,0.0071,0.67,0.15,-0.29\r\n20181018,0.0071,-2.89,-0.41,0.67\r\n20181019,0.0071,2.67,-0.9,0.17\r\n20181022,0.0071,4.42,0.03,-1.11\r\n20181023,0.0071,-2.3,0.95,-0.45\r\n20181024,0.0071,0.11,0.17,0.37\r\n20181025,0.0071,-0.1,-0.59,0.32\r\n20181026,0.0071,-0.19,1.18,0.58\r\n20181029,0.0071,-2.18,1.1,-0.01\r\n20181030,0.0071,0.94,0.12,0.02\r\n20181031,0.0071,1.38,0,-0.19\r\n20181101,0.0056,0.43,-0.13,-0.43\r\n20181102,0.0056,3.03,-0.52,-0.03\r\n20181105,0.0056,-0.36,1.18,-0.76\r\n20181106,0.0056,-0.26,0.42,-0.17\r\n20181107,0.0056,-0.55,0.36,-0.15\r\n20181108,0.0056,-0.32,0.36,0.09\r\n20181109,0.0056,-1.13,0.98,-0.51\r\n20181112,0.0056,1.62,1.35,-0.63\r\n20181113,0.0056,1.14,0.93,-0.76\r\n20181114,0.0056,-0.75,0.81,-0.36\r\n20181115,0.0056,1.39,0.56,-0.38\r\n20181116,0.0056,0.57,0.71,-0.27\r\n20181119,0.0056,0.78,-0.42,0.2\r\n20181120,0.0056,-2.35,-0.35,-0.19\r\n20181121,0.0056,0.3,0.72,-0.22\r\n20181122,0.0056,-0.18,0.35,-0.09\r\n20181123,0.0056,-2.8,-1.76,1.22\r\n20181126,0.0056,-0.18,-0.04,-0.12\r\n20181127,0.0056,0.09,0.63,-0.73\r\n20181128,0.0056,1.19,-0.18,0.08\r\n20181129,0.0056,-1.53,-0.94,0.76\r\n20181130,0.0056,0.88,-0.42,0.24\r\n20181203,0.0064,2.81,0.45,-0.22\r\n20181204,0.0064,0.37,0.61,-0.27\r\n20181205,0.0064,-0.56,-0.43,0.32\r\n20181206,0.0064,-1.91,0.29,0.44\r\n20181207,0.0064,0.04,0.11,0.23\r\n20181210,0.0064,-0.97,-0.4,0.02\r\n20181211,0.0064,0.52,0.28,-0.46\r\n20181212,0.0064,0.27,-0.31,0.35\r\n20181213,0.0064,1.24,-0.35,0.39\r\n20181214,0.0064,-1.8,-0.64,0.81\r\n20181217,0.0064,-0.06,0.23,0.09\r\n20181218,0.0064,-0.82,0.12,-0.19\r\n20181219,0.0064,-1.15,0.06,0.14\r\n20181220,0.0064,-0.31,0.74,-0.56\r\n20181221,0.0064,-0.89,0.25,0.06\r\n20181224,0.0064,0.58,0.21,-0.11\r\n20181225,0.0064,-0.85,-0.35,0.27\r\n20181226,0.0064,-0.34,0.22,0.23\r\n20181227,0.0064,-0.74,-0.96,0.68\r\n20181228,0.0064,0.46,-0.44,-0.22\r\n20190102,0.0058,-1.13,0.97,-0.62\r\n20190103,0.0058,-0.3,-0.29,-0.15\r\n20190104,0.0058,2.26,0.15,-0.6\r\n20190107,0.0058,1.04,1.1,-0.89\r\n20190108,0.0058,-0.21,0.27,0.09\r\n20190109,0.0058,0.65,-0.32,0.16\r\n20190110,0.0058,-0.37,0.41,0.35\r\n20190111,0.0058,0.69,0.33,-0.26\r\n20190114,0.0058,-0.71,0.36,0.05\r\n20190115,0.0058,1.43,-0.53,0.07\r\n20190116,0.0058,-0.03,-0.15,0.34\r\n20190117,0.0058,-0.59,-0.32,0.21\r\n20190118,0.0058,1.32,-0.85,0.97\r\n20190121,0.0058,0.58,-0.09,-0.03\r\n20190122,0.0058,-1.24,0.31,0.13\r\n20190123,0.0058,0.07,0.24,0.3\r\n20190124,0.0058,0.44,-0.13,-0.02\r\n20190125,0.0058,0.28,-1,1.3\r\n20190128,0.0058,-0.19,-0.81,0.5\r\n20190129,0.0058,-0.36,-1.65,1.42\r\n20190130,0.0058,-0.86,-0.69,0.83\r\n20190131,0.0058,0.14,-1.83,2.15\r\n20190201,0.0077,1.76,0.72,-0.98\r\n20190211,0.0077,1.87,0.71,-0.97\r\n20190212,0.0077,0.81,0.67,-0.53\r\n20190213,0.0077,1.84,-0.4,-0.54\r\n20190214,0.0077,0.18,0.28,-0.46\r\n20190215,0.0077,-1.22,1.22,-0.72\r\n20190218,0.0077,3.05,0.13,-0.55\r\n20190219,0.0077,-0.01,0.55,-0.82\r\n20190220,0.0077,0.26,0.08,0.17\r\n20190221,0.0077,-0.32,-0.02,-0.25\r\n20190222,0.0077,2.01,-0.12,-1.31\r\n20190225,0.0077,5.53,-0.74,-0.63\r\n20190226,0.0077,-0.69,0.57,-0.62\r\n20190227,0.0077,-0.04,0.02,-0.31\r\n20190228,0.0077,-0.15,0.6,0.36\r\n20190301,0.0061,1.57,-1.21,0.03\r\n20190304,0.0061,1.44,0.14,-0.97\r\n20190305,0.0061,1.3,1.37,-1.93\r\n20190306,0.0061,1.35,0.94,-0.92\r\n20190307,0.0061,-0.02,1.78,-1.67\r\n20190308,0.0061,-4.05,-0.56,0.76\r\n20190311,0.0061,2.75,1.82,-0.42\r\n20190312,0.0061,1.26,0.79,-0.75\r\n20190313,0.0061,-1.5,-0.19,1.21\r\n20190314,0.0061,-1.48,-1.47,1.4\r\n20190315,0.0061,1.16,0.63,0.31\r\n20190318,0.0061,2.58,-0.4,0.27\r\n20190319,0.0061,-0.09,0.54,-0.44\r\n20190320,0.0061,-0.18,0.31,-0.05\r\n20190321,0.0061,0.48,0.68,-0.88\r\n20190322,0.0061,0.14,0.45,-0.08\r\n20190325,0.0061,-1.78,0.87,-0.69\r\n20190326,0.0061,-1.74,-0.86,1.22\r\n20190327,0.0061,0.9,-0.02,1.01\r\n20190328,0.0061,-0.84,-0.6,0.19\r\n20190329,0.0061,3.3,-0.94,-0.46\r\n20190401,0.0059,2.94,0.77,-0.42\r\n20190402,0.0059,0.1,0.73,-0.55\r\n20190403,0.0059,1.07,-0.05,0.06\r\n20190404,0.0059,0.78,-0.26,0.69\r\n20190408,0.0059,-0.2,0.21,0.86\r\n20190409,0.0059,0.17,0.46,-0.38\r\n20190410,0.0059,-0.03,0.03,0.26\r\n20190411,0.0059,-1.87,0.13,-0.48\r\n20190412,0.0059,-0.1,0.05,0.05\r\n20190415,0.0059,-0.55,-0.18,0.31\r\n20190416,0.0059,2.33,-0.34,-0.22\r\n20190417,0.0059,0.44,0.67,-0.4\r\n20190418,0.0059,-0.46,-0.12,0.13\r\n20190419,0.0059,0.76,0.11,-0.45\r\n20190422,0.0059,-1.69,0.43,-0.38\r\n20190423,0.0059,-0.78,-1.35,0.58\r\n20190424,0.0059,0.5,0.5,0.04\r\n20190425,0.0059,-2.76,-1.63,0.8\r\n20190426,0.0059,-1.22,-0.44,-0.2\r\n20190429,0.0059,-1.15,-2.49,2.44\r\n20190430,0.0059,0.56,0.61,0.28\r\n20190506,0.0064,-6.25,-0.82,1.87\r\n20190507,0.0064,0.99,0.68,0.39\r\n20190508,0.0064,-0.99,0.82,-0.6\r\n20190509,0.0064,-1.47,0.66,-0.52\r\n20190510,0.0064,3.37,-0.27,-0.88\r\n20190513,0.0064,-1.21,0.97,0.34\r\n20190514,0.0064,-0.66,-0.12,-0.14\r\n20190515,0.0064,2.07,0.31,0.05\r\n20190516,0.0064,0.5,0.27,0.02\r\n20190517,0.0064,-2.72,-0.75,0.75\r\n20190520,0.0064,-0.54,0.28,-0.27\r\n20190521,0.0064,1.45,0.21,-0.54\r\n20190522,0.0064,-0.49,-0.25,-0.44\r\n20190523,0.0064,-1.71,-0.18,0.72\r\n20190524,0.0064,-0.19,-0.49,1.29\r\n20190527,0.0064,1.76,0.63,-1.52\r\n20190528,0.0064,0.62,-0.69,-0.29\r\n20190529,0.0064,-0.02,0.48,-0.12\r\n20190530,0.0064,-0.45,-0.17,0.33\r\n20190531,0.0064,-0.21,-0.01,-0.32\r\n20190603,0.0065,-0.43,-1.07,0.87\r\n20190604,0.0065,-1.09,-0.46,0.83\r\n20190605,0.0065,0,0.18,0.13\r\n20190606,0.0065,-1.42,-1,1.36\r\n20190610,0.0065,1.02,0.15,-0.13\r\n20190611,0.0065,3.04,0.5,-0.82\r\n20190612,0.0065,-0.62,0.22,-0.42\r\n20190613,0.0065,0.11,0.65,-0.44\r\n20190614,0.0065,-1.21,-0.96,0.65\r\n20190617,0.0065,0.04,0.13,0.67\r\n20190618,0.0065,0.1,-0.27,0.23\r\n20190619,0.0065,1.27,0.18,-0.1\r\n20190620,0.0065,2.28,-0.75,0.62\r\n20190621,0.0065,0.73,0.9,-0.92\r\n20190624,0.0065,0.13,-0.24,0.15\r\n20190625,0.0065,-0.92,-0.08,0.28\r\n20190626,0.0065,-0.11,-0.08,0.13\r\n20190627,0.0065,0.9,-0.3,0.21\r\n20190628,0.0065,-0.6,-0.8,0.56\r\n20190701,0.0055,2.64,0.01,-0.62\r\n20190702,0.0055,0.03,-0.18,-0.05\r\n20190703,0.0055,-0.89,0.07,0.47\r\n20190704,0.0055,-0.43,0.36,0.43\r\n20190705,0.0055,0.38,-0.03,0.08\r\n20190708,0.0055,-2.62,-0.94,0.66\r\n20190709,0.0055,-0.02,0.53,-0.2\r\n20190710,0.0055,-0.41,-0.3,0.03\r\n20190711,0.0055,0.03,0.05,0.41\r\n20190712,0.0055,0.57,-0.14,0.68\r\n20190715,0.0055,0.6,-0.23,-0.97\r\n20190716,0.0055,-0.15,0.45,-0.31\r\n20190717,0.0055,-0.04,-0.1,-0.52\r\n20190718,0.0055,-1.21,-0.53,0.58\r\n20190719,0.0055,0.82,-0.4,0.13\r\n20190722,0.0055,-1.3,-1.57,1.25\r\n20190723,0.0055,0.59,0.57,-0.77\r\n20190724,0.0055,0.89,0.18,-0.49\r\n20190725,0.0055,0.56,-0.53,-0.05\r\n20190726,0.0055,0.19,-0.19,0.1\r\n20190729,0.0055,-0.04,-0.19,-0.39\r\n20190730,0.0055,0.42,0.17,0.37\r\n20190731,0.0055,-0.67,-0.16,-0.17\r\n20190801,0.0058,-0.74,0.33,-0.3\r\n20190802,0.0058,-1.43,-0.37,-0.28\r\n20190805,0.0058,-1.62,0.43,-0.53\r\n20190806,0.0058,-1.59,-1.5,0.56\r\n20190807,0.0058,-0.36,0.02,0.01\r\n20190808,0.0058,1,-0.54,-0.3\r\n20190809,0.0058,-0.89,-0.02,0.56\r\n20190812,0.0058,1.65,0,-0.21\r\n20190813,0.0058,-0.66,0.13,-0.1\r\n20190814,0.0058,0.53,-0.06,-0.24\r\n20190815,0.0058,0.38,-0.15,-0.48\r\n20190816,0.0058,0.38,-0.05,-0.19\r\n20190819,0.0058,2.44,0.54,-0.56\r\n20190820,0.0058,-0.02,0.32,-0.22\r\n20190821,0.0058,-0.04,0.29,-0.18\r\n20190822,0.0058,0.22,0.27,-0.4\r\n20190823,0.0058,0.33,-0.54,0.54\r\n20190826,0.0058,-1.03,0.24,-0.42\r\n20190827,0.0058,1.53,0.31,-0.51\r\n20190828,0.0058,-0.29,0.21,-0.32\r\n20190829,0.0058,-0.13,0.06,-0.59\r\n20190830,0.0058,-0.33,-1.38,0.43\r\n20190902,0.0062,1.62,0.79,-0.41\r\n20190903,0.0062,0.35,0.26,-0.42\r\n20190904,0.0062,0.85,-0.02,0.3\r\n20190905,0.0062,0.93,-0.14,-0.46\r\n20190906,0.0062,0.41,0.13,0.2\r\n20190909,0.0062,1.21,0.93,-0.99\r\n20190910,0.0062,-0.16,0.55,0.01\r\n20190911,0.0062,-0.62,0.18,0.6\r\n20190912,0.0062,0.68,0.11,0.08\r\n20190916,0.0062,0.02,0.37,-0.43\r\n20190917,0.0062,-1.8,-0.68,0.35\r\n20190918,0.0062,0.25,-0.14,0.15\r\n20190919,0.0062,0.62,0.34,-0.43\r\n20190920,0.0062,0.23,-0.29,-0.13\r\n20190923,0.0062,-0.98,0.07,-0.34\r\n20190924,0.0062,0.29,-0.25,-0.23\r\n20190925,0.0062,-1.2,-0.58,0.78\r\n20190926,0.0062,-1.44,-1.24,2.27\r\n20190927,0.0062,0.39,0.51,-0.36\r\n20190930,0.0062,-0.97,-0.13,0.49\r\n20191008,0.0071,0.29,0.06,0.86\r\n20191009,0.0071,0.44,0.56,-0.05\r\n20191010,0.0071,1,-0.04,-0.87\r\n20191011,0.0071,0.68,-0.37,0.55\r\n20191014,0.0071,1.23,0.32,-0.27\r\n20191015,0.0071,-0.76,-0.1,0.9\r\n20191016,0.0071,-0.4,-0.24,0.42\r\n20191017,0.0071,0,-0.21,-0.15\r\n20191018,0.0071,-1.28,0,0.06\r\n20191021,0.0071,0.04,-0.61,0.56\r\n20191022,0.0071,0.66,0.43,-0.6\r\n20191023,0.0071,-0.59,0.05,0.33\r\n20191024,0.0071,-0.08,0.12,0.53\r\n20191025,0.0071,0.69,-0.43,-0.59\r\n20191028,0.0071,1.11,0.73,-1.67\r\n20191029,0.0071,-0.82,-0.5,0.6\r\n20191030,0.0071,-0.6,-0.88,0.26\r\n20191031,0.0071,-0.46,-1.03,0.39\r\n20191101,0.0059,1.12,-0.16,0.34\r\n20191104,0.0059,0.61,-0.42,0.22\r\n20191105,0.0059,0.56,-0.49,0.38\r\n20191106,0.0059,-0.59,-0.09,0.59\r\n20191107,0.0059,0.24,0.44,-0.23\r\n20191108,0.0059,-0.39,0.08,-0.15\r\n20191111,0.0059,-1.98,-0.66,0.26\r\n20191112,0.0059,0.14,0.22,-0.08\r\n20191113,0.0059,-0.19,-0.51,-0.33\r\n20191114,0.0059,0.3,0.17,-0.6\r\n20191115,0.0059,-0.78,-0.6,0\r\n20191118,0.0059,0.64,0.2,0.36\r\n20191119,0.0059,1.2,0.37,-0.67\r\n20191120,0.0059,-0.77,0.01,-0.37\r\n20191121,0.0059,-0.28,0.21,0.06\r\n20191122,0.0059,-1,0.45,0.79\r\n20191125,0.0059,0.31,-0.34,1.92\r\n20191126,0.0059,0.19,-0.47,-0.22\r\n20191127,0.0059,-0.18,0.05,-0.3\r\n20191128,0.0059,-0.38,0.07,-0.02\r\n20191129,0.0059,-0.52,0.63,-0.13\r\n20191202,0.0058,0.16,-0.11,0.15\r\n20191203,0.0058,0.37,0.22,-0.49\r\n20191204,0.0058,-0.09,-0.01,-0.56\r\n20191205,0.0058,0.89,0.06,-0.11\r\n20191206,0.0058,0.59,0.03,-0.3\r\n20191209,0.0058,0.04,0.44,0.34\r\n20191210,0.0058,0.19,0.14,-0.56\r\n20191211,0.0058,-0.02,-0.17,0.18\r\n20191212,0.0058,-0.24,-0.03,-0.1\r\n20191213,0.0058,1.69,-0.57,0.71\r\n20191216,0.0058,0.92,0.8,-0.57\r\n20191217,0.0058,1.27,-0.02,-0.23\r\n20191218,0.0058,-0.13,0.39,-0.1\r\n20191219,0.0058,0.06,0.91,-0.03\r\n20191220,0.0058,-0.53,-0.48,0.84\r\n20191223,0.0058,-1.59,-0.58,0.71\r\n20191224,0.0058,0.93,0.52,-0.63\r\n20191225,0.0058,0.14,0.13,-0.44\r\n20191226,0.0058,0.79,0.18,0.4\r\n20191227,0.0058,-0.31,-0.21,0.43\r\n20191230,0.0058,1.15,-0.67,0.57\r\n20191231,0.0058,0.41,0.09,-0.07\r\n20200102,0.0079,1.43,0.54,-0.38\r\n20200103,0.0079,0.04,0.58,-0.46\r\n20200106,0.0079,0.14,0.63,-0.75\r\n20200107,0.0079,0.93,0.64,0\r\n20200108,0.0079,-1.23,-0.29,-0.2\r\n20200109,0.0079,1.25,0.33,-0.13\r\n20200110,0.0079,-0.11,-0.13,0.22\r\n20200113,0.0079,1.04,-0.1,-0.2\r\n20200114,0.0079,-0.3,0.54,0.38\r\n20200115,0.0079,-0.38,-0.12,-0.43\r\n20200116,0.0079,-0.35,-0.23,-0.26\r\n20200117,0.0079,-0.08,-0.46,0.11\r\n20200120,0.0079,0.94,-0.28,-0.43\r\n20200121,0.0079,-1.37,0.17,-0.27\r\n20200122,0.0079,0.5,-0.89,-0.43\r\n20200123,0.0079,-3.05,-0.49,0.13\r\n20200203,0.0059,-8.09,-0.71,0.37\r\n20200204,0.0059,1.84,-3.17,0.57\r\n20200205,0.0059,1.75,1.08,-1.12\r\n20200206,0.0059,2.22,0.43,-0.94\r\n20200207,0.0059,0.41,0.68,-1.15\r\n20200210,0.0059,0.77,0.92,-0.26\r\n20200211,0.0059,0.3,-1.09,0.56\r\n20200212,0.0059,1.18,0.24,-1.24\r\n20200213,0.0059,-0.7,-0.46,-0.33\r\n20200214,0.0059,0.36,-0.39,0.45\r\n20200217,0.0059,2.66,0.81,-1\r\n20200218,0.0059,0.4,1.48,-1.79\r\n20200219,0.0059,-0.43,0.07,0.73\r\n20200220,0.0059,1.92,-0.19,-0.23\r\n20200221,0.0059,0.66,0.35,-1.25\r\n20200224,0.0059,0.42,1.04,-1.78\r\n20200225,0.0059,-0.09,-0.13,-1.41\r\n20200226,0.0059,-1.77,0.68,2.73\r\n20200227,0.0059,0.12,0.33,1.01\r\n20200228,0.0059,-4.24,-1.08,1.99\r\n20200302,0.0058,3.4,0.99,0.25\r\n20200303,0.0058,0.86,0.39,-0.58\r\n20200304,0.0058,0.43,1.22,1.12\r\n20200305,0.0058,1.87,0.19,0.84\r\n20200306,0.0058,-1.07,1.01,-0.55\r\n20200309,0.0058,-3.43,0.46,1.13\r\n20200310,0.0058,2.13,-0.51,-1.34\r\n20200311,0.0058,-1.25,0.61,0.48\r\n20200312,0.0058,-1.87,-0.21,0.01\r\n20200313,0.0058,-1.17,-0.48,-0.86\r\n20200316,0.0058,-4.08,0.46,1.37\r\n20200317,0.0058,-0.37,-0.09,-0.21\r\n20200318,0.0058,-1.74,0.15,0.05\r\n20200319,0.0058,-0.5,0.99,-1.68\r\n20200320,0.0058,1.45,0.1,0.91\r\n20200323,0.0058,-3.66,0.13,1.46\r\n20200324,0.0058,2.28,-0.49,0.31\r\n20200325,0.0058,2.51,-0.37,-0.24\r\n20200326,0.0058,-0.7,-0.32,0.33\r\n20200327,0.0058,-0.01,-0.17,1.26\r\n20200330,0.0058,-1.39,-0.52,1.36\r\n20200331,0.0058,0.3,-0.35,-0.14\r\n20200401,0.0059,-0.47,-0.19,0.03\r\n20200402,0.0059,1.96,0,-0.82\r\n20200403,0.0059,-0.55,0.08,0.1\r\n20200407,0.0059,2.52,0.8,-0.96\r\n20200408,0.0059,-0.26,0.82,-0.13\r\n20200409,0.0059,0.56,0.33,-0.51\r\n20200410,0.0059,-1.39,-0.98,1.38\r\n20200413,0.0059,-0.6,-0.19,0.68\r\n20200414,0.0059,1.89,-0.37,-0.65\r\n20200415,0.0059,-0.52,-0.1,-0.12\r\n20200416,0.0059,0.39,-0.03,-0.3\r\n20200417,0.0059,0.59,-0.75,0.13\r\n20200420,0.0059,0.71,0.67,-0.23\r\n20200421,0.0059,-0.91,0.56,-0.03\r\n20200422,0.0059,0.8,0.22,-0.38\r\n20200423,0.0059,-0.31,-0.18,0.14\r\n20200424,0.0059,-1.18,-0.85,0.8\r\n20200427,0.0059,0.24,-1.08,0.89\r\n20200428,0.0059,-0.14,-2.75,0.28\r\n20200429,0.0059,0.17,-0.23,0.99\r\n20200430,0.0059,1.57,0.41,-1.01\r\n20200506,0.0071,1.1,0.55,-0.77\r\n20200507,0.0071,-0.22,0.11,0.34\r\n20200508,0.0071,0.99,-0.04,-0.42\r\n20200511,0.0071,-0.14,0.26,0.09\r\n20200512,0.0071,0.08,-0.31,0.1\r\n20200513,0.0071,0.42,0.08,0.16\r\n20200514,0.0071,-0.97,0.12,-0.35\r\n20200515,0.0071,0.06,0.08,-0.52\r\n20200518,0.0071,-0.07,-0.4,0.75\r\n20200519,0.0071,1.04,-0.16,-0.39\r\n20200520,0.0071,-0.73,-0.05,0.1\r\n20200521,0.0071,-0.79,0.02,0.65\r\n20200522,0.0071,-2,0.41,-0.23\r\n20200525,0.0071,0.01,0.01,1.08\r\n20200526,0.0071,1.56,0.37,-0.86\r\n20200527,0.0071,-0.64,0.7,0.25\r\n20200528,0.0071,0.06,-0.31,0.55\r\n20200529,0.0071,0.57,0.17,-0.45\r\n20200601,0.0062,2.68,0.17,-0.31\r\n20200602,0.0062,0.16,0.6,0.32\r\n20200603,0.0062,0.04,0,-0.09\r\n20200604,0.0062,0.01,0.42,0.27\r\n20200605,0.0062,0.34,-0.65,-0.24\r\n20200608,0.0062,0.11,-0.32,0.23\r\n20200609,0.0062,0.66,-0.33,0.19\r\n20200610,0.0062,-0.03,-0.2,0.04\r\n20200611,0.0062,-0.66,0.23,-0.57\r\n20200612,0.0062,0.09,-0.07,0.12\r\n20200615,0.0062,-0.7,0.75,0.36\r\n20200616,0.0062,1.63,0.09,-0.12\r\n20200617,0.0062,0.21,0.5,0.21\r\n20200618,0.0062,0.25,-0.59,-0.1\r\n20200619,0.0062,1.15,-0.92,0.04\r\n20200622,0.0062,0.15,0.05,-0.36\r\n20200623,0.0062,0.45,-0.78,0.06\r\n20200624,0.0062,0.32,-0.6,0.63\r\n20200629,0.0062,-0.54,0.04,0.61\r\n20200630,0.0062,1.45,-0.08,-0.3\r\n20200701,0.0055,1.13,-0.57,0.63\r\n20200702,0.0055,1.73,-0.18,0.99\r\n20200703,0.0055,1.69,-0.69,0.07\r\n20200706,0.0055,5.06,-1.09,0.68\r\n20200707,0.0055,0.96,-0.1,-0.64\r\n20200708,0.0055,1.78,0.5,-0.3\r\n20200709,0.0055,1.96,0.68,-0.95\r\n20200710,0.0055,-1.09,0.22,-1.14\r\n20200713,0.0055,2.56,1.09,0.17\r\n20200714,0.0055,-0.92,0.61,0.47\r\n20200715,0.0055,-1.75,-1.16,0.82\r\n20200716,0.0055,-4.84,0.52,0.6\r\n20200717,0.0055,0.42,-0.37,0.46\r\n20200720,0.0055,2.88,1.02,0.46\r\n20200721,0.0055,0.49,-0.15,-0.55\r\n20200722,0.0055,0.6,-0.39,-0.89\r\n20200723,0.0055,-0.12,-0.36,-0.11\r\n20200724,0.0055,-4.42,0.31,0.12\r\n20200727,0.0055,0.25,-0.35,0.36\r\n20200728,0.0055,1,0.02,0.12\r\n20200729,0.0055,2.46,-0.07,-0.56\r\n20200730,0.0055,-0.37,0.61,0.4\r\n20200731,0.0055,0.96,0.23,-0.56\r\n20200803,0.0061,2.1,1.09,-0.92\r\n20200804,0.0061,-0.21,-0.29,0.93\r\n20200805,0.0061,0.45,0.53,-0.84\r\n20200806,0.0061,-0.16,-0.02,-0.44\r\n20200807,0.0061,-1.15,-0.14,-0.43\r\n20200810,0.0061,0.5,0.49,0.5\r\n20200811,0.0061,-1.3,-0.67,0.25\r\n20200812,0.0061,-0.94,0.27,0.33\r\n20200813,0.0061,0.02,0.78,0.02\r\n20200814,0.0061,1.26,-0.54,0.27\r\n20200817,0.0061,2.12,0.06,0.6\r\n20200818,0.0061,0.4,0.54,-0.25\r\n20200819,0.0061,-1.56,0.28,0.83\r\n20200820,0.0061,-1.32,0.27,0.18\r\n20200821,0.0061,0.76,-0.21,0.3\r\n20200824,0.0061,0.64,0.07,-0.33\r\n20200825,0.0061,-0.2,0.04,0.23\r\n20200826,0.0061,-1.53,-0.51,0.53\r\n20200827,0.0061,0.77,0.22,-0.31\r\n20200828,0.0061,1.79,-0.93,0.36\r\n20200831,0.0061,-0.34,0.86,-0.15\r\n20200901,0.0056,0.5,-0.12,-0.11\r\n20200902,0.0056,0.05,0.75,-0.65\r\n20200903,0.0056,-0.71,-0.43,-0.02\r\n20200904,0.0056,-0.78,0.99,-0.43\r\n20200907,0.0056,-2.09,1.52,0.37\r\n20200908,0.0056,0.51,0.93,0.62\r\n20200909,0.0056,-2.46,-0.1,0.83\r\n20200910,0.0056,-1.04,-3.42,1.35\r\n20200911,0.0056,1.12,0.08,-0.43\r\n20200914,0.0056,0.76,1.14,-0.58\r\n20200915,0.0056,0.63,-0.59,-0.38\r\n20200916,0.0056,-0.63,-0.2,0.6\r\n20200917,0.0056,-0.21,0.48,-0.02\r\n20200918,0.0056,1.86,-0.55,0.75\r\n20200921,0.0056,-0.63,0.6,-0.21\r\n20200922,0.0056,-1.19,-0.42,0.13\r\n20200923,0.0056,0.41,0.05,-0.69\r\n20200924,0.0056,-2,-0.86,0.23\r\n20200925,0.0056,-0.17,-0.57,0.06\r\n20200928,0.0056,-0.28,-1.31,0.84\r\n20200929,0.0056,0.58,0.1,-0.81\r\n20200930,0.0056,-0.06,-0.3,-0.37\r\n20201009,0.0079,2.23,0.98,-0.49\r\n20201012,0.0079,2.9,0.2,-0.59\r\n20201013,0.0079,0.29,-0.02,-0.21\r\n20201014,0.0079,-0.64,-0.02,0.55\r\n20201015,0.0079,-0.46,-0.59,0.77\r\n20201016,0.0079,-0.15,0.39,0.76\r\n20201019,0.0079,-0.72,0.41,-0.18\r\n20201020,0.0079,0.78,0.16,-0.51\r\n20201021,0.0079,-0.54,-0.8,0.85\r\n20201022,0.0079,-0.46,0.07,-0.04\r\n20201023,0.0079,-1.45,0.14,0.88\r\n20201026,0.0079,-0.27,0.4,-0.23\r\n20201027,0.0079,0.34,-0.51,-0.53\r\n20201028,0.0079,0.55,-0.63,-0.2\r\n20201029,0.0079,0.32,-0.8,0.5\r\n20201030,0.0079,-1.74,-1.17,-0.17\r\n20201102,0.0059,0.46,-0.52,0.59\r\n20201103,0.0059,1.4,0.53,-0.78\r\n20201104,0.0059,0.25,-0.6,0.42\r\n20201105,0.0059,1.45,0.3,-0.21\r\n20201106,0.0059,-0.5,-0.45,0.82\r\n20201109,0.0059,2.03,-0.09,-0.89\r\n20201110,0.0059,-0.73,-0.27,0.53\r\n20201111,0.0059,-1.17,0.26,2.05\r\n20201112,0.0059,0.1,-0.14,-0.33\r\n20201113,0.0059,-0.6,0.54,-0.88\r\n20201116,0.0059,0.95,0.31,0.87\r\n20201117,0.0059,-0.53,0.03,0.83\r\n20201118,0.0059,-0.06,0.7,0.67\r\n20201119,0.0059,0.61,-0.49,-0.6\r\n20201120,0.0059,0.48,-0.04,-0.15\r\n20201123,0.0059,0.87,-0.16,1.01\r\n20201124,0.0059,-0.33,0.3,-0.23\r\n20201125,0.0059,-1.45,0.21,0.24\r\n20201126,0.0059,-0.05,-0.54,0.26\r\n20201127,0.0059,0.87,-0.99,0.74\r\n20201130,0.0059,-0.38,-0.12,-0.56\r\n20201201,0.0055,1.75,-0.79,0.07\r\n20201202,0.0055,0.02,0.1,-0.18\r\n20201203,0.0055,-0.12,-0.26,-0.54\r\n20201204,0.0055,0.25,-0.24,-0.57\r\n20201207,0.0055,-0.6,-0.04,-1.02\r\n20201208,0.0055,-0.11,-0.11,-0.37\r\n20201209,0.0055,-1.43,-0.55,0.2\r\n20201210,0.0055,0.08,-0.38,0.04\r\n20201211,0.0055,-0.98,-0.69,0.02\r\n20201214,0.0055,0.79,-0.38,-0.6\r\n20201215,0.0055,0.15,-0.19,-0.26\r\n20201216,0.0055,-0.14,-0.98,0.46\r\n20201217,0.0055,1.06,-0.08,0.25\r\n20201218,0.0055,-0.31,0.09,0.01\r\n20201221,0.0055,1.23,-0.43,-1.32\r\n20201222,0.0055,-1.83,0.1,-0.19\r\n20201223,0.0055,0.77,-0.35,-0.52\r\n20201224,0.0055,-0.72,-1.11,0.75\r\n20201225,0.0055,0.91,-0.17,0.23\r\n20201228,0.0055,-0.05,-0.96,0.54\r\n20201229,0.0055,-0.57,0.07,-0.22\r\n20201230,0.0055,1.15,-1.06,-0.59\r\n20201231,0.0055,1.76,-0.69,-0.79\r\n20210104,0.0063,1.46,-0.2,-1.45\r\n20210105,0.0063,1.16,-1.43,-0.27\r\n20210106,0.0063,0.35,-1.69,0.98\r\n20210107,0.0063,0.54,-3.98,0.39\r\n20210108,0.0063,-0.27,0.11,0.35\r\n20210111,0.0063,-1.23,-1.17,0.44\r\n20210112,0.0063,2.08,-1.51,0.02\r\n20210113,0.0063,-0.52,-1.03,0.88\r\n20210114,0.0063,-1.15,2.04,0.9\r\n20210115,0.0063,0,1.39,0.54\r\n20210118,0.0063,1.11,0.33,-0.49\r\n20210119,0.0063,-1.02,1.6,0.48\r\n20210120,0.0063,0.83,-1.06,-1.26\r\n20210121,0.0063,1.31,-0.9,-0.3\r\n20210122,0.0063,-0.03,-1.56,-0.74\r\n20210125,0.0063,0.4,-2.31,-0.03\r\n20210126,0.0063,-1.72,1.16,1.18\r\n20210127,0.0063,0.15,-0.01,0.87\r\n20210128,0.0063,-2.29,1.21,0.36\r\n20210129,0.0063,-0.71,-0.97,0.67\r\n20210201,0.0083,0.85,-0.27,1.05\r\n20210202,0.0083,1.21,-1.06,-1.02\r\n20210203,0.0083,-0.63,-1.06,0.84\r\n20210204,0.0083,-0.73,-1.34,0.68\r\n20210205,0.0083,-0.45,-0.86,1.15\r\n20210208,0.0083,1.21,-1.25,-0.46\r\n20210209,0.0083,2.1,-0.36,-0.69\r\n20210210,0.0083,1.61,-1.05,-0.78\r\n20210218,0.0083,0.06,3.03,1.12\r\n20210219,0.0083,0.59,2.74,0.79\r\n20210222,0.0083,-1.82,3.46,1.58\r\n20210223,0.0083,-0.4,-0.12,0.34\r\n20210224,0.0083,-2.02,2.29,0.48\r\n20210225,0.0083,0.15,-0.94,1.4\r\n20210226,0.0083,-2.04,2.08,-0.12\r\n20210301,0.0054,1.72,0.7,-0.65\r\n20210302,0.0054,-0.98,0.52,0.24\r\n20210303,0.0054,1.65,-0.32,0.88\r\n20210304,0.0054,-2.44,2.06,1.31\r\n20210305,0.0054,0,1.01,-0.34\r\n20210308,0.0054,-2.78,2.26,2.11\r\n20210309,0.0054,-2.28,-0.07,0.9\r\n20210310,0.0054,0.1,-1.69,-1.11\r\n20210311,0.0054,2.34,-0.39,0.04\r\n20210312,0.0054,0.34,-0.1,1.18\r\n20210315,0.0054,-1.52,1.92,1.48\r\n20210316,0.0054,0.83,0.52,0.02\r\n20210317,0.0054,0.41,-0.29,-1.49\r\n20210318,0.0054,0.67,-0.96,-0.03\r\n20210319,0.0054,-1.85,1.81,-0.03\r\n20210322,0.0054,1.16,0.77,0.81\r\n20210323,0.0054,-1.02,-0.03,0.03\r\n20210324,0.0054,-1.38,0.71,0.14\r\n20210325,0.0054,-0.05,-0.57,-1\r\n20210326,0.0054,1.91,-0.99,-0.85\r\n20210329,0.0054,0.34,-0.2,0.46\r\n20210330,0.0054,0.59,-1.87,-0.14\r\n20210331,0.0054,-0.5,0.94,0.74\r\n20210401,0.006,0.94,-0.85,-0.5\r\n20210402,0.006,0.69,-0.53,-1.2\r\n20210406,0.006,0,1.3,0.14\r\n20210407,0.006,-0.27,1.07,0.01\r\n20210408,0.006,0.07,-1.09,-0.19\r\n20210409,0.006,-0.95,1.21,0.14\r\n20210412,0.006,-1.56,0.55,0.38\r\n20210413,0.006,-0.3,-0.7,-0.41\r\n20210414,0.006,0.94,-0.02,-0.76\r\n20210415,0.006,-0.5,0.09,0.22\r\n20210416,0.006,0.68,1.09,-0.16\r\n20210419,0.006,1.94,-0.91,-0.84\r\n20210420,0.006,-0.11,-0.27,0.03\r\n20210421,0.006,0.09,-0.63,-0.05\r\n20210422,0.006,0.08,0.31,-0.09\r\n20210423,0.006,0.41,-1.5,-0.01\r\n20210426,0.006,-0.82,0.74,0.09\r\n20210427,0.006,0.06,-0.8,-0.32\r\n20210428,0.006,0.58,-0.7,-0.44\r\n20210429,0.006,0.38,-0.94,0.59\r\n20210430,0.006,-0.56,-0.22,-0.79\r\n20210506,0.0069,-0.6,1.45,1.43\r\n20210507,0.0069,-1.14,1.19,1.59\r\n20210510,0.0069,0.2,0.41,-0.18\r\n20210511,0.0069,0.37,-0.12,-0.72\r\n20210512,0.0069,0.73,0.42,-0.36\r\n20210513,0.0069,-0.93,0.62,-0.62\r\n20210514,0.0069,1.82,-0.91,-0.18\r\n20210517,0.0069,1.04,-2.59,-0.72\r\n20210518,0.0069,0.24,0.3,-0.1\r\n20210519,0.0069,-0.23,-0.07,-0.74\r\n20210520,0.0069,0.03,-0.62,-0.39\r\n20210521,0.0069,-0.56,0.89,0.36\r\n20210524,0.0069,0.52,0.2,-0.58\r\n20210525,0.0069,2.26,-1.74,-0.27\r\n20210526,0.0069,0.12,0.81,0.5\r\n20210527,0.0069,0.62,0.13,-0.82\r\n20210528,0.0069,-0.24,-0.1,0.01\r\n20210531,0.0069,0.76,0.39,-1.43\r\n20210601,0.006,0.34,0.59,-0.34\r\n20210602,0.006,-0.97,0.31,1.03\r\n20210603,0.006,-0.37,0.47,0.09\r\n20210604,0.006,0.49,-0.79,-0.42\r\n20210607,0.006,0.25,0.84,-0.45\r\n20210608,0.006,-0.75,0.64,-0.06\r\n20210609,0.006,0.19,-0.01,0.28\r\n20210610,0.006,0.8,-0.38,-1.58\r\n20210611,0.006,-0.59,-0.25,-0.12\r\n20210615,0.006,-0.86,0.18,-0.58\r\n20210616,0.006,-1.64,0.94,0.94\r\n20210617,0.006,0.78,-0.34,-1.54\r\n20210618,0.006,0.38,0.34,-1.15\r\n20210621,0.006,0.36,0.93,-1.3\r\n20210622,0.006,0.63,-0.15,0.61\r\n20210623,0.006,0.52,-0.41,-0.27\r\n20210624,0.006,-0.22,-0.23,1.34\r\n20210625,0.006,1.25,-1.28,0.42\r\n20210628,0.006,0.37,0.26,-0.87\r\n20210629,0.006,-0.88,0.11,0.02\r\n20210630,0.006,0.77,-0.18,-0.64\r\n20210701,0.0057,-0.49,-1.06,1.09\r\n20210702,0.0057,-1.99,1.99,0.54\r\n20210705,0.0057,0.56,0.67,-0.97\r\n20210706,0.0057,-0.18,0.49,1.23\r\n20210707,0.0057,1.16,-0.96,-1.35\r\n20210708,0.0057,-0.6,0.15,-1.15\r\n20210709,0.0057,0.01,0.82,0.84\r\n20210712,0.0057,1.32,-0.11,-1.49\r\n20210713,0.0057,0.4,0.26,0.71\r\n20210714,0.0057,-1,0.36,0.54\r\n20210715,0.0057,0.81,-1.68,0.68\r\n20210716,0.0057,-0.82,1.21,1.01\r\n20210719,0.0057,-0.05,0.03,-0.2\r\n20210720,0.0057,0.06,0.23,-1.02\r\n20210721,0.0057,1.12,-0.07,-1.85\r\n20210722,0.0057,0.43,0.37,0.4\r\n20210723,0.0057,-0.97,0.19,0.33\r\n20210726,0.0057,-2.34,1.21,-0.41\r\n20210727,0.0057,-2.89,1.08,-0.01\r\n20210728,0.0057,-0.58,-2.43,0.01\r\n20210729,0.0057,2.24,0.39,-2.54\r\n20210730,0.0057,-0.21,1.47,-0.17\r\n20210802,0.0057,2.03,0.01,0.24\r\n20210803,0.0057,-0.58,0.58,1.26\r\n20210804,0.0057,1.29,-0.51,-1.83\r\n20210805,0.0057,-0.48,-0.34,-0.3\r\n20210806,0.0057,-0.2,0.53,0.47\r\n20210809,0.0057,0.9,0.42,1.71\r\n20210810,0.0057,0.89,0.29,0.28\r\n20210811,0.0057,0.07,0.81,0.46\r\n20210812,0.0057,-0.33,0.78,0.14\r\n20210813,0.0057,-0.35,0.71,1.1\r\n20210816,0.0057,-0.28,0.46,1.14\r\n20210817,0.0057,-2.23,-0.45,0.77\r\n20210818,0.0057,0.96,-0.08,0.54\r\n20210819,0.0057,-0.21,-0.5,-1.7\r\n20210820,0.0057,-1.16,0.94,0.39\r\n20210823,0.0057,1.82,0.67,-1.03\r\n20210824,0.0057,0.96,-0.66,-0.11\r\n20210825,0.0057,0.62,0.16,0.11\r\n20210826,0.0057,-1.33,1.16,0.91\r\n20210827,0.0057,0.38,-0.65,0.62\r\n20210830,0.0057,0.12,0.17,-0.39\r\n20210831,0.0057,0.03,0.16,1.69\r\n20210901,0.0063,0.12,-0.31,1.94\r\n20210902,0.0063,0.61,0.91,1.3\r\n20210903,0.0063,-0.52,0.91,0.33\r\n20210906,0.0063,1.55,-0.69,-0.82\r\n20210907,0.0063,1.35,-0.17,0.98\r\n20210908,0.0063,-0.07,1.37,0.68\r\n20210909,0.0063,0.31,-0.31,0.7\r\n20210910,0.0063,0.37,-1.33,-0.57\r\n20210913,0.0063,0.12,0.92,1.8\r\n20210914,0.0063,-1.02,-0.44,-2.1\r\n20210915,0.0063,-0.2,1.17,0.32\r\n20210916,0.0063,-1.65,0.53,1.27\r\n20210917,0.0063,0.34,-0.71,-0.05\r\n20210922,0.0063,0.09,0.61,1.15\r\n20210923,0.0063,0.38,0.2,-0.07\r\n20210924,0.0063,-0.7,-1.59,-1.52\r\n20210927,0.0063,-0.89,-2.4,-0.79\r\n20210928,0.0063,0.2,0.53,1.67\r\n20210929,0.0063,-2.01,-1.24,-0.14\r\n20210930,0.0063,1.34,0.69,-1.06\r\n20211008,0.0078,0.64,1.22,0.3\r\n20211011,0.0078,-0.22,0.08,0.64\r\n20211012,0.0078,-1.46,-0.31,0.17\r\n20211013,0.0078,0.83,-1.12,-1.79\r\n20211014,0.0078,0.01,0.84,-0.32\r\n20211015,0.0078,0.3,-1.54,-0.3\r\n20211018,0.0078,-0.1,0.95,0.14\r\n20211019,0.0078,0.81,-0.27,0.07\r\n20211020,0.0078,-0.12,-0.04,-0.89\r\n20211021,0.0078,0.06,-0.48,1.18\r\n20211022,0.0078,-0.23,-0.88,-0.44\r\n20211025,0.0078,0.81,-0.28,-0.91\r\n20211026,0.0078,-0.35,-0.03,-0.04\r\n20211027,0.0078,-1.03,-0.83,-0.44\r\n20211028,0.0078,-1.32,-0.83,-0.15\r\n20211029,0.0078,1.15,0.37,-1.6\r\n20211101,0.0057,0.15,1.8,0.31\r\n20211102,0.0057,-0.95,-0.41,-1.52\r\n20211103,0.0057,-0.14,1,0.56\r\n20211104,0.0057,1.03,0.65,-0.97\r\n20211105,0.0057,-0.94,0.64,-0.13\r\n20211108,0.0057,0.29,0.09,-0.41\r\n20211109,0.0057,0.45,0.7,-0.86\r\n20211110,0.0057,-0.39,1.51,0.2\r\n20211111,0.0057,1.14,-0.19,0.73\r\n20211112,0.0057,0.23,1,-0.51\r\n20211115,0.0057,-0.27,1.63,0.57\r\n20211116,0.0057,-0.39,-0.32,0.29\r\n20211117,0.0057,0.7,0.67,-0.6\r\n20211118,0.0057,-0.55,-0.27,-0.16\r\n20211119,0.0057,1.15,-0.06,0.19\r\n20211122,0.0057,0.99,-0.26,-1.14\r\n20211123,0.0057,-0.01,0.6,0.43\r\n20211124,0.0057,0,0.49,0.56\r\n20211125,0.0057,-0.32,0.39,0.1\r\n20211126,0.0057,-0.39,0.28,-0.36\r\n20211129,0.0057,0.16,-0.14,-0.79\r\n20211130,0.0057,0.01,1.02,-0.13\r\n20211201,0.0054,0.22,0.92,0.76\r\n20211202,0.0054,-0.31,-1.12,0.79\r\n20211203,0.0054,0.86,-0.53,0.01\r\n20211206,0.0054,-0.8,-0.56,1.17\r\n20211207,0.0054,-0.26,-0.69,1.48\r\n20211208,0.0054,1.47,-0.23,-1.07\r\n20211209,0.0054,0.94,-0.58,0.29\r\n20211210,0.0054,-0.1,0.42,-0.39\r\n20211213,0.0054,0.49,0.16,0.06\r\n20211214,0.0054,-0.44,1.18,-0.72\r\n20211215,0.0054,-0.52,0.94,0.48\r\n20211216,0.0054,0.66,0.27,0.52\r\n20211217,0.0054,-1.33,0.67,0.4\r\n20211220,0.0054,-1.47,0.82,0.98\r\n20211221,0.0054,0.92,0.76,-0.08\r\n20211222,0.0054,0.27,0.31,-0.41\r\n20211223,0.0054,0.46,-1.03,-0.06\r\n20211224,0.0054,-0.92,-0.59,0.75\r\n20211227,0.0054,-0.04,0.79,0.51\r\n20211228,0.0054,0.57,0.03,-0.34\r\n20211229,0.0054,-0.93,1.09,0.18\r\n20211230,0.0054,0.75,0.43,-0.5\r\n20211231,0.0054,0.54,0.18,0.17\r\n'}

In [ ]:
def read_author_snapshot(text, date_column):
    lines = text.splitlines()
    header = next(i for i,line in enumerate(lines) if line.split(',')[0].strip().lower()==date_column)
    data = pd.read_csv(StringIO('\n'.join(lines[header:])))
    data.columns = data.columns.str.strip().str.lower()
    data[date_column] = pd.to_datetime(data[date_column].astype(str),errors='coerce')
    data = data.dropna(subset=[date_column]).set_index(date_column).sort_index()
    for column in data:
        data[column] = pd.to_numeric(data[column],errors='coerce') / 100
    return data


author_ch4_monthly = read_author_snapshot(author_csv_snapshots['ch4_monthly'],'mnthdt')
author_ch4_monthly.index = author_ch4_monthly.index.to_period('M')
author_ch4_monthly = author_ch4_monthly.loc['2000-01':'2016-12']
author_ch3_daily = read_author_snapshot(author_csv_snapshots['ch3_daily'],'date')
author_rf_monthly = author_ch4_monthly['rf_mon']
assert len(author_rf_monthly)==204 and author_rf_monthly.notna().all()
assert not author_ch4_monthly.index.duplicated().any()
print('作者月度样本：',author_ch4_monthly.shape)
print('作者日度样本：',author_ch3_daily.shape)
display(author_ch4_monthly.head())
